# Tensor Shapes and Manual Backpropagation — 154 Progressive Exercises

This supplementary workbook prepares you for the manual-backpropagation portion of Andrej Karpathy's **Backprop Ninja** lecture. It begins with rank-zero tensors and one-operation graphs, then adds reductions, broadcasting, matrix multiplication, branches, embeddings, stable softmax, BatchNorm, and a complete next-character MLP.

It contains **no student solutions**. Supplied fixtures build forward graphs and private autograd references; you must write every manual derivative in the answer cells.

## How to use this notebook

1. Run the test-helper cell once.
2. Work strictly from top to bottom.
3. Whenever an exercise requests one, write the output shape tuple before deriving gradients.
4. Draw or narrate the local graph before writing a gradient expression.
5. Use only ordinary tensor operations in answer cells—never `.backward()`, `torch.autograd.grad`, or `.grad`.
6. Run the supplied test immediately below your answer.
7. Move on only after every required variable prints `PASS`.

The tests check shape, dtype, and values without displaying expected tensors. When stuck, ask for one hint about the current exercise rather than requesting the whole chain.


## The six-question backward checklist

For every forward line, ask:

1. What does each axis mean?
2. What are the input and output shapes?
3. Is the operation elementwise, a reduction, a broadcast, an index, or a matrix contraction?

For every backward line, ask:

1. What upstream gradient arrives, and what is its shape?
2. What is this operation's local derivative?
3. Did forward broadcasting create repeated paths that must be summed?
4. Did a forward reduction remove axes that backward must restore?
5. Does the variable appear along more than one path?
6. Does the final gradient have exactly the same shape as its forward variable?

A name beginning with `d` means the derivative of the final scalar objective with respect to the named forward tensor. For example, `dprobs` has the same shape as `probs`.


## Course map

- Exercises 001–014: tensor ranks, axes, reductions, reshape, matmul, lookup
- Exercises 015–032: local scalar and elementwise derivatives
- Exercises 033–048: reduction backward
- Exercises 049–066: broadcasting and unbroadcasting
- Exercises 067–084: matrix multiplication and affine layers
- Exercises 085–100: chains, fan-out, and gradient accumulation
- Exercises 101–110: embedding lookups and indexed accumulation
- Exercises 111–126: stable softmax and mean negative log-likelihood
- Exercises 127–140: BatchNorm as atomic operations
- Exercises 141–154: complete next-character MLP capstone


In [1]:
# Supplied test infrastructure: run once after starting or restarting the kernel.
import torch

# Float64 keeps tiny hand-derived examples stable under different operation orders.
DTYPE = torch.float64
_REFS = {}
_MISSING = object()


def _store_forward(key, output):
    """Store a private forward reference without displaying the answer."""
    assert isinstance(output, torch.Tensor)
    _REFS[key] = {
        "out": output.detach().clone(),
        "out_shape": tuple(output.shape),
    }


def _capture(key, output, inputs, upstream=None, retain_graph=False):
    """Use autograd only to build private test references for manual derivations."""
    assert isinstance(output, torch.Tensor)
    if upstream is None:
        assert output.numel() == 1, "A non-scalar output needs an explicit upstream gradient."
        grad_outputs = None
    else:
        assert isinstance(upstream, torch.Tensor)
        assert upstream.shape == output.shape
        grad_outputs = upstream

    names = list(inputs)
    tensors = list(inputs.values())
    gradients = torch.autograd.grad(
        output,
        tensors,
        grad_outputs=grad_outputs,
        retain_graph=retain_graph,
        create_graph=False,
        allow_unused=False,
    )
    reference = {
        "out": output.detach().clone(),
        "out_shape": tuple(output.shape),
    }
    for name, gradient in zip(names, gradients):
        reference[name] = gradient.detach().clone()
    _REFS[key] = reference


def _capture_path(key, field, output, input_tensor, upstream, retain_graph=False):
    """Store one private branch contribution using autograd rather than an answer formula."""
    gradient = torch.autograd.grad(
        output,
        input_tensor,
        grad_outputs=upstream,
        retain_graph=retain_graph,
        create_graph=False,
    )[0]
    _REFS[key][field] = gradient.detach().clone()


def _capture_product_path(key, field, left, right, upstream):
    """Store one operand's local product path on detached test-only leaves."""
    left_local = left.detach().requires_grad_()
    right_local = right.detach().requires_grad_()
    product_local = left_local * right_local
    gradient = torch.autograd.grad(product_local, left_local, grad_outputs=upstream)[0]
    _REFS[key][field] = gradient.detach().clone()


def _store_shape(key, field, tensor):
    """Add an intermediate shape to an existing reference record."""
    assert key in _REFS
    _REFS[key][field] = tuple(tensor.shape)


def _check_tensor(variable_name, key, field):
    """Check type, shape, dtype, and values without revealing expected values."""
    actual = globals().get(variable_name, _MISSING)
    assert actual is not _MISSING, f"Define `{variable_name}` in the answer cell first."
    assert isinstance(actual, torch.Tensor), f"`{variable_name}` must be a torch.Tensor."
    expected = _REFS[key][field]
    assert actual.shape == expected.shape, (
        f"`{variable_name}` has shape {tuple(actual.shape)}; "
        f"the gradient/value must have shape {tuple(expected.shape)}."
    )
    assert actual.dtype == expected.dtype, (
        f"`{variable_name}` has dtype {actual.dtype}; expected {expected.dtype}."
    )
    torch.testing.assert_close(actual, expected, rtol=1e-7, atol=1e-9)
    print(f"PASS: {variable_name}")


def _check_shape(variable_name, key, field="out_shape"):
    """Require an explicit Python shape tuple before checking its value."""
    actual = globals().get(variable_name, _MISSING)
    assert actual is not _MISSING, f"Define `{variable_name}` in the answer cell first."
    assert isinstance(actual, tuple), f"`{variable_name}` must be a Python tuple."
    expected = _REFS[key][field]
    assert actual == expected, f"`{variable_name}` is {actual}; reconsider the axes."
    print(f"PASS: {variable_name}")


print("Test helpers ready. Start at Exercise 001 and do not use autograd in answer cells.")


Test helpers ready. Start at Exercise 001 and do not use autograd in answer cells.


## 1. Tensor shape foundations — forward operations only

Before differentiating, learn to narrate every axis. For each exercise, compute the forward tensor and explicitly record its shape as a Python tuple. These first tasks contain no gradients.

Use this shape ledger:

1. Write each input shape.
2. Name what every axis represents.
3. Apply the operation's shape rule.
4. Check that the number of elements is preserved when reshaping.

A scalar tensor has shape `()`. A vector has one axis. A matrix has two axes.


### Exercise 001 — Scalar plus constant

**Purpose:** Learn that adding a number to a scalar tensor still produces a scalar tensor.

**Inputs:** `x` is a scalar tensor with shape `()`.

**Forward operation:** Compute `x + 3.0` manually in your answer cell.

**Derive/do:** Add the constant and record the result shape.

**Ingredients:** Tensor addition and `tuple(tensor.shape)`.

**Required outputs:**

- `ex001_out_shape`: a Python tuple containing the output shape.
- `ex001_out`: the resulting scalar tensor.

**Next concept:** Vector times scalar.


In [2]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor(2.0, dtype=DTYPE)
_store_forward("ex001", x + 3.0)
# Inspect only the supplied input shapes; compute the requested output yourself.
print("Input fixture ready.")


Input fixture ready.


In [3]:
# Exercise 001: derive manually; do not use autograd in this cell.
# Define `ex001_out_shape` — a Python tuple containing the output shape.
# Define `ex001_out` — the resulting scalar tensor.
# Write your tensor operations below these comments, then run the test cell.
print(tuple(x.shape))
ex001_out_shape = ()  # () is just a tensor with 1 number
ex001_out = x + 3.0


()


In [4]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex001_out_shape", "ex001", "out_shape")
_check_tensor("ex001_out", "ex001", "out")


PASS: ex001_out_shape
PASS: ex001_out


### Exercise 002 — Vector times scalar

**Purpose:** Learn that multiplying a vector by one number multiplies every element and keeps the vector's shape unchanged.

**Inputs:** `x` has shape `(4,)`.

**Forward operation:** Compute `x * 2.5` manually in your answer cell.

**Derive/do:** Multiply every entry by the scalar.

**Ingredients:** Elementwise multiplication with a scalar.

**Required outputs:**

- `ex002_out_shape`: a Python tuple containing the output shape.
- `ex002_out`: the scaled vector.

**Next concept:** Elementwise vector addition.


In [5]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([1.0, -2.0, 3.0, 4.0], dtype=DTYPE)
_store_forward("ex002", x * 2.5)
# Inspect only the supplied input shapes; compute the requested output yourself.
print("Input fixture ready.")


Input fixture ready.


In [6]:
# Exercise 002: derive manually; do not use autograd in this cell.
# Define `ex002_out_shape` — a Python tuple containing the output shape.
# Define `ex002_out` — the scaled vector.
# Write your tensor operations below these comments, then run the test cell.
ex002_out_shape = (4,)
ex002_out = x * 2.5

In [7]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex002_out_shape", "ex002", "out_shape")
_check_tensor("ex002_out", "ex002", "out")


PASS: ex002_out_shape
PASS: ex002_out


### Exercise 003 — Elementwise vector addition

**Purpose:** Learn that adding two vectors adds entries at matching positions: first with first, second with second, and so on.

**Inputs:** `x` and `y` both have shape `(4,)`.

**Forward operation:** Compute `x + y` manually in your answer cell.

**Derive/do:** Add corresponding entries.

**Ingredients:** Elementwise addition; this is not concatenation.

**Required outputs:**

- `ex003_out_shape`: a Python tuple containing the output shape.
- `ex003_out`: the elementwise sum.

**Next concept:** Elementwise vector multiplication.


In [8]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([1.0, 2.0, 3.0, 4.0], dtype=DTYPE)
y = torch.tensor([-1.0, 0.5, 2.0, 3.0], dtype=DTYPE)
_store_forward("ex003", x + y)
# Inspect only the supplied input shapes; compute the requested output yourself.
print("Input fixture ready.")


Input fixture ready.


In [9]:
# Exercise 003: derive manually; do not use autograd in this cell.
# Define `ex003_out_shape` — a Python tuple containing the output shape.
# Define `ex003_out` — the elementwise sum.
# Write your tensor operations below these comments, then run the test cell.
ex003_out_shape = (4,)
ex003_out = x + y

In [10]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex003_out_shape", "ex003", "out_shape")
_check_tensor("ex003_out", "ex003", "out")


PASS: ex003_out_shape
PASS: ex003_out


### Exercise 004 — Elementwise vector multiplication

**Purpose:** Learn that `*` multiplies matching vector entries and keeps them separate instead of combining them into one number.

**Inputs:** `x` and `y` both have shape `(3,)`.

**Forward operation:** Compute `x * y` manually in your answer cell.

**Derive/do:** Multiply corresponding entries without reducing.

**Ingredients:** The `*` operator.

**Required outputs:**

- `ex004_out_shape`: a Python tuple containing the output shape.
- `ex004_out`: the elementwise product.

**Next concept:** Matrix plus scalar.


In [11]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([2.0, -1.0, 4.0], dtype=DTYPE)
y = torch.tensor([3.0, 5.0, -2.0], dtype=DTYPE)
_store_forward("ex004", x * y)
# Inspect only the supplied input shapes; compute the requested output yourself.
print("Input fixture ready.")


Input fixture ready.


In [12]:
# Exercise 004: derive manually; do not use autograd in this cell.
# Define `ex004_out_shape` — a Python tuple containing the output shape.
# Define `ex004_out` — the elementwise product.
# Write your tensor operations below these comments, then run the test cell.
ex004_out_shape = (3,)
ex004_out = x * y

In [13]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex004_out_shape", "ex004", "out_shape")
_check_tensor("ex004_out", "ex004", "out")


PASS: ex004_out_shape
PASS: ex004_out


### Exercise 005 — Matrix plus scalar

**Purpose:** Learn that adding one number to a matrix adds that number to every matrix entry.

**Inputs:** `x` has shape `(2, 3)`.

**Forward operation:** Compute `x + 10.0` manually in your answer cell.

**Derive/do:** Add 10 to every matrix entry.

**Ingredients:** Scalar broadcasting.

**Required outputs:**

- `ex005_out_shape`: a Python tuple containing the output shape.
- `ex005_out`: the shifted matrix.

**Next concept:** Elementwise matrix product.


In [14]:
# Supplied fixture: run this before writing the manual answer.
x = torch.arange(6, dtype=DTYPE).reshape(2, 3)
_store_forward("ex005", x + 10.0)
# Inspect only the supplied input shapes; compute the requested output yourself.
print("Input fixture ready.")


Input fixture ready.


In [15]:
# Exercise 005: derive manually; do not use autograd in this cell.
# Define `ex005_out_shape` — a Python tuple containing the output shape.
# Define `ex005_out` — the shifted matrix.
# Write your tensor operations below these comments, then run the test cell.


In [16]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex005_out_shape", "ex005", "out_shape")
_check_tensor("ex005_out", "ex005", "out")


AssertionError: Define `ex005_out_shape` in the answer cell first.

### Exercise 006 — Elementwise matrix product

**Purpose:** Learn that `*` multiplies entries at matching row-column positions and keeps the matrix shape unchanged.

**Inputs:** `x` and `y` have shape `(2, 3)`.

**Forward operation:** Compute `x * y` manually in your answer cell.

**Derive/do:** Multiply entries at corresponding row-column coordinates.

**Ingredients:** The `*` operator, not `@`.

**Required outputs:**

- `ex006_out_shape`: a Python tuple containing the output shape.
- `ex006_out`: the elementwise matrix product.

**Next concept:** Reduce every axis.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.arange(1, 7, dtype=DTYPE).reshape(2, 3)
y = torch.tensor([[1.0, 0.0, -1.0], [2.0, 3.0, 4.0]], dtype=DTYPE)
_store_forward("ex006", x * y)
# Inspect only the supplied input shapes; compute the requested output yourself.
print("Input fixture ready.")


In [ ]:
# Exercise 006: derive manually; do not use autograd in this cell.
# Define `ex006_out_shape` — a Python tuple containing the output shape.
# Define `ex006_out` — the elementwise matrix product.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex006_out_shape", "ex006", "out_shape")
_check_tensor("ex006_out", "ex006", "out")


### Exercise 007 — Reduce every axis

**Purpose:** Learn that summing every entry of a matrix produces one scalar tensor.

**Inputs:** `x` has shape `(2, 3)`.

**Forward operation:** Compute `x.sum()` manually in your answer cell.

**Derive/do:** Sum all six entries.

**Ingredients:** `Tensor.sum()` with no dimension argument.

**Required outputs:**

- `ex007_out_shape`: a Python tuple containing the output shape.
- `ex007_out`: a scalar total.

**Next concept:** Reduce rows with dim 0.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.arange(1, 7, dtype=DTYPE).reshape(2, 3)
_store_forward("ex007", x.sum())
# Inspect only the supplied input shapes; compute the requested output yourself.
print("Input fixture ready.")


In [ ]:
# Exercise 007: derive manually; do not use autograd in this cell.
# Define `ex007_out_shape` — a Python tuple containing the output shape.
# Define `ex007_out` — a scalar total.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex007_out_shape", "ex007", "out_shape")
_check_tensor("ex007_out", "ex007", "out")


### Exercise 008 — Reduce rows with dim 0

**Purpose:** Learn that `dim=0` combines the rows and leaves one result for each column.

**Inputs:** `x` has shape `(2, 3)`.

**Forward operation:** Compute `x.sum(dim=0)` manually in your answer cell.

**Derive/do:** Sum down the rows, leaving one result per column.

**Ingredients:** `Tensor.sum(dim=0)`.

**Required outputs:**

- `ex008_out_shape`: a Python tuple containing the output shape.
- `ex008_out`: the column totals.

**Next concept:** Reduce columns with dim 1.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.arange(1, 7, dtype=DTYPE).reshape(2, 3)
_store_forward("ex008", x.sum(dim=0))
# Inspect only the supplied input shapes; compute the requested output yourself.
print("Input fixture ready.")


In [ ]:
# Exercise 008: derive manually; do not use autograd in this cell.
# Define `ex008_out_shape` — a Python tuple containing the output shape.
# Define `ex008_out` — the column totals.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex008_out_shape", "ex008", "out_shape")
_check_tensor("ex008_out", "ex008", "out")


### Exercise 009 — Reduce columns with dim 1

**Purpose:** Learn that `dim=1` combines the columns and leaves one result for each row.

**Inputs:** `x` has shape `(2, 3)`.

**Forward operation:** Compute `x.sum(dim=1)` manually in your answer cell.

**Derive/do:** Sum across columns, leaving one result per row.

**Ingredients:** `Tensor.sum(dim=1)`.

**Required outputs:**

- `ex009_out_shape`: a Python tuple containing the output shape.
- `ex009_out`: the row totals.

**Next concept:** Keep a reduced axis.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.arange(1, 7, dtype=DTYPE).reshape(2, 3)
_store_forward("ex009", x.sum(dim=1))
# Inspect only the supplied input shapes; compute the requested output yourself.
print("Input fixture ready.")


In [ ]:
# Exercise 009: derive manually; do not use autograd in this cell.
# Define `ex009_out_shape` — a Python tuple containing the output shape.
# Define `ex009_out` — the row totals.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex009_out_shape", "ex009", "out_shape")
_check_tensor("ex009_out", "ex009", "out")


### Exercise 010 — Keep a reduced axis

**Purpose:** Learn that `keepdim=True` leaves a reduced axis in the result with length 1.

**Inputs:** `x` has shape `(2, 3)`.

**Forward operation:** Compute `x.sum(dim=1, keepdim=True)` manually in your answer cell.

**Derive/do:** Sum each row while keeping dimension 1.

**Ingredients:** `Tensor.sum(dim=1, keepdim=True)`.

**Required outputs:**

- `ex010_out_shape`: a Python tuple containing the output shape.
- `ex010_out`: the row totals with shape `(2, 1)`.

**Next concept:** Reshape without changing element count.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.arange(1, 7, dtype=DTYPE).reshape(2, 3)
_store_forward("ex010", x.sum(dim=1, keepdim=True))
# Inspect only the supplied input shapes; compute the requested output yourself.
print("Input fixture ready.")


In [ ]:
# Exercise 010: derive manually; do not use autograd in this cell.
# Define `ex010_out_shape` — a Python tuple containing the output shape.
# Define `ex010_out` — the row totals with shape `(2, 1)`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex010_out_shape", "ex010", "out_shape")
_check_tensor("ex010_out", "ex010", "out")


### Exercise 011 — Reshape without changing element count

**Purpose:** Learn that reshaping changes how dimensions are grouped without changing the values or their total count.

**Inputs:** `x` has shape `(2, 3)` and six elements.

**Forward operation:** Compute `x.reshape(3, 2)` manually in your answer cell.

**Derive/do:** Reinterpret it as three rows and two columns.

**Ingredients:** `Tensor.reshape(3, 2)`.

**Required outputs:**

- `ex011_out_shape`: a Python tuple containing the output shape.
- `ex011_out`: the reshaped tensor.

**Next concept:** Flatten non-batch axes.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.arange(6, dtype=DTYPE).reshape(2, 3)
_store_forward("ex011", x.reshape(3, 2))
# Inspect only the supplied input shapes; compute the requested output yourself.
print("Input fixture ready.")


In [ ]:
# Exercise 011: derive manually; do not use autograd in this cell.
# Define `ex011_out_shape` — a Python tuple containing the output shape.
# Define `ex011_out` — the reshaped tensor.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex011_out_shape", "ex011", "out_shape")
_check_tensor("ex011_out", "ex011", "out")


### Exercise 012 — Flatten non-batch axes

**Purpose:** Learn how to keep batch examples separate while joining each example's remaining dimensions into one vector.

**Inputs:** `x` has shape `(2, 3, 2)`: two examples, three positions, two features.

**Forward operation:** Compute `x.reshape(x.shape[0], -1)` manually in your answer cell.

**Derive/do:** Keep the first axis and infer one flattened feature axis.

**Ingredients:** `reshape(x.shape[0], -1)` and element-count conservation.

**Required outputs:**

- `ex012_out_shape`: a Python tuple containing the output shape.
- `ex012_out`: a `(2, 6)` tensor.

**Next concept:** Matrix multiplication shape.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.arange(12, dtype=DTYPE).reshape(2, 3, 2)
_store_forward("ex012", x.reshape(x.shape[0], -1))
# Inspect only the supplied input shapes; compute the requested output yourself.
print("Input fixture ready.")


In [ ]:
# Exercise 012: derive manually; do not use autograd in this cell.
# Define `ex012_out_shape` — a Python tuple containing the output shape.
# Define `ex012_out` — a `(2, 6)` tensor.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex012_out_shape", "ex012", "out_shape")
_check_tensor("ex012_out", "ex012", "out")


### Exercise 013 — Matrix multiplication shape

**Purpose:** Learn how the two outside dimensions determine a matrix product's output shape after the shared inside dimension is combined.

**Inputs:** `left` has shape `(3, 4)` and `right` has shape `(4, 2)`.

**Forward operation:** Compute `left @ right` manually in your answer cell.

**Derive/do:** Multiply them with `@`.

**Ingredients:** The shared inner size 4 contracts; outer sizes 3 and 2 remain.

**Required outputs:**

- `ex013_out_shape`: a Python tuple containing the output shape.
- `ex013_out`: the matrix product.

**Next concept:** Embedding-table lookup shape.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
left = torch.arange(12, dtype=DTYPE).reshape(3, 4)
right = torch.arange(8, dtype=DTYPE).reshape(4, 2)
_store_forward("ex013", left @ right)
# Inspect only the supplied input shapes; compute the requested output yourself.
print("Input fixture ready.")


In [ ]:
# Exercise 013: derive manually; do not use autograd in this cell.
# Define `ex013_out_shape` — a Python tuple containing the output shape.
# Define `ex013_out` — the matrix product.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex013_out_shape", "ex013", "out_shape")
_check_tensor("ex013_out", "ex013", "out")


### Exercise 014 — Embedding-table lookup shape

**Purpose:** Learn that each integer ID is replaced by the corresponding row from an embedding table.

**Inputs:** `table` has shape `(5, 3)` and `ids` has shape `(2, 2)`.

**Forward operation:** Compute `table[ids]` manually in your answer cell.

**Derive/do:** Replace each integer ID with its length-3 table row.

**Ingredients:** Advanced integer indexing: index shape plus unindexed trailing source shape.

**Required outputs:**

- `ex014_out_shape`: a Python tuple containing the output shape.
- `ex014_out`: a grid of embedding vectors.

**Next concept:** Add a constant backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
table = torch.arange(15, dtype=DTYPE).reshape(5, 3)
ids = torch.tensor([[2, 0], [4, 1]], dtype=torch.long)
_store_forward("ex014", table[ids])
# Inspect only the supplied input shapes; compute the requested output yourself.
print("Input fixture ready.")


In [ ]:
# Exercise 014: derive manually; do not use autograd in this cell.
# Define `ex014_out_shape` — a Python tuple containing the output shape.
# Define `ex014_out` — a grid of embedding vectors.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex014_out_shape", "ex014", "out_shape")
_check_tensor("ex014_out", "ex014", "out")


## 2. Local derivatives with scalar and elementwise tensor operations

Now every supplied forward output receives an upstream gradient called `dout`. Derive gradients manually; do not call `.backward()`, `torch.autograd.grad`, or read `.grad`.

For an elementwise operation, each output position initially depends on the corresponding input position. The chain rule is

$$
\frac{\partial L}{\partial x}
=
\frac{\partial L}{\partial y}
\frac{\partial y}{\partial x}
$$

Here, `L` is the final scalar objective, `y` is this exercise's forward output, `x` is an input, and `dout` represents the first factor. The resulting gradient for an input must have exactly the same shape as that input.


### Exercise 015 — Add a constant backward

**Purpose:** Learn that adding a constant lets the incoming gradient pass back to `x` unchanged.

**Inputs:** `x` and `dout` have shape `(3,)`.

**Forward operation:** `y = x + 4.0`.

**Derive/do:** Derive the gradient of the scalar objective with respect to `x`.

**Ingredients:** The local slope of adding a constant and the incoming `dout`.

**Required outputs:**

- `ex015_out_shape`: a Python tuple containing the forward output shape.
- `ex015_dx`: the manual gradient with respect to `x`.

**Next concept:** Multiply by a constant backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([-2.0, 0.5, 3.0], dtype=DTYPE, requires_grad=True)
y = x + 4.0
dout = torch.tensor([0.2, -1.0, 3.0], dtype=DTYPE)
_capture("ex015", y, {"dx": x}, dout)
print("x", tuple(x.shape), "y", tuple(y.shape), "dout", tuple(dout.shape))


In [ ]:
# Exercise 015: derive manually; do not use autograd in this cell.
# Define `ex015_out_shape` — a Python tuple containing the forward output shape.
# Define `ex015_dx` — the manual gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex015_out_shape", "ex015", "out_shape")
_check_tensor("ex015_dx", "ex015", "dx")


### Exercise 016 — Multiply by a constant backward

**Purpose:** Learn that multiplying by a constant also multiplies the incoming gradient by that constant.

**Inputs:** `x` and `dout` have shape `(4,)`.

**Forward operation:** `y = -3.0 * x`.

**Derive/do:** Derive the gradient with respect to `x`.

**Ingredients:** The local derivative of multiplication by a fixed scalar.

**Required outputs:**

- `ex016_out_shape`: a Python tuple containing the forward output shape.
- `ex016_dx`: the manual gradient with respect to `x`.

**Next concept:** Negation backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([1.0, -2.0, 0.5, 4.0], dtype=DTYPE, requires_grad=True)
y = -3.0 * x
dout = torch.tensor([1.0, 2.0, -1.0, 0.25], dtype=DTYPE)
_capture("ex016", y, {"dx": x}, dout)
print("x", tuple(x.shape), "y", tuple(y.shape), "dout", tuple(dout.shape))


In [ ]:
# Exercise 016: derive manually; do not use autograd in this cell.
# Define `ex016_out_shape` — a Python tuple containing the forward output shape.
# Define `ex016_dx` — the manual gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex016_out_shape", "ex016", "out_shape")
_check_tensor("ex016_dx", "ex016", "dx")


### Exercise 017 — Negation backward

**Purpose:** Learn that a minus sign reverses the sign of the incoming gradient.

**Inputs:** `x` and `dout` have shape `(3,)`.

**Forward operation:** `y = -x`.

**Derive/do:** Derive `dx`.

**Ingredients:** The local slope of negation.

**Required outputs:**

- `ex017_out_shape`: a Python tuple containing the forward output shape.
- `ex017_dx`: the manual gradient with respect to `x`.

**Next concept:** Square backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([2.0, -1.0, 5.0], dtype=DTYPE, requires_grad=True)
y = -x
dout = torch.tensor([0.5, 2.0, -3.0], dtype=DTYPE)
_capture("ex017", y, {"dx": x}, dout)
print("Forward and upstream shapes:", tuple(y.shape), tuple(dout.shape))


In [ ]:
# Exercise 017: derive manually; do not use autograd in this cell.
# Define `ex017_out_shape` — a Python tuple containing the forward output shape.
# Define `ex017_dx` — the manual gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex017_out_shape", "ex017", "out_shape")
_check_tensor("ex017_dx", "ex017", "dx")


### Exercise 018 — Square backward

**Purpose:** Learn how the gradient of a square depends on the original input value.

**Inputs:** `x` and `dout` have shape `(4,)`.

**Forward operation:** `y = x**2`.

**Derive/do:** Derive `dx` for the supplied upstream gradient.

**Ingredients:** The power rule and elementwise chain rule.

**Required outputs:**

- `ex018_out_shape`: a Python tuple containing the forward output shape.
- `ex018_dx`: the manual gradient with respect to `x`.

**Next concept:** Cube backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([-2.0, -0.5, 1.5, 3.0], dtype=DTYPE, requires_grad=True)
y = x**2
dout = torch.tensor([1.0, -2.0, 0.5, 3.0], dtype=DTYPE)
_capture("ex018", y, {"dx": x}, dout)
print("Forward and upstream shapes:", tuple(y.shape), tuple(dout.shape))


In [ ]:
# Exercise 018: derive manually; do not use autograd in this cell.
# Define `ex018_out_shape` — a Python tuple containing the forward output shape.
# Define `ex018_dx` — the manual gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex018_out_shape", "ex018", "out_shape")
_check_tensor("ex018_dx", "ex018", "dx")


### Exercise 019 — Cube backward

**Purpose:** Learn to apply the same power-rule idea to a cube.

**Inputs:** `x` and `dout` have shape `(3,)`.

**Forward operation:** `y = x**3`.

**Derive/do:** Derive `dx`.

**Ingredients:** The power rule for exponent 3 and the upstream gradient.

**Required outputs:**

- `ex019_out_shape`: a Python tuple containing the forward output shape.
- `ex019_dx`: the manual gradient with respect to `x`.

**Next concept:** Reciprocal backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([-2.0, 0.5, 3.0], dtype=DTYPE, requires_grad=True)
y = x**3
dout = torch.tensor([0.25, -1.0, 2.0], dtype=DTYPE)
_capture("ex019", y, {"dx": x}, dout)
print("Forward and upstream shapes:", tuple(y.shape), tuple(dout.shape))


In [ ]:
# Exercise 019: derive manually; do not use autograd in this cell.
# Define `ex019_out_shape` — a Python tuple containing the forward output shape.
# Define `ex019_dx` — the manual gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex019_out_shape", "ex019", "out_shape")
_check_tensor("ex019_dx", "ex019", "dx")


### Exercise 020 — Reciprocal backward

**Purpose:** Learn how to differentiate `1 / x` by viewing it as a negative power.

**Inputs:** Positive `x` and `dout` have shape `(3,)`.

**Forward operation:** `y = x**-1`.

**Derive/do:** Derive `dx`.

**Ingredients:** Rewrite the reciprocal as a power, apply the power rule, then chain with `dout`.

**Required outputs:**

- `ex020_out_shape`: a Python tuple containing the forward output shape.
- `ex020_dx`: the manual gradient with respect to `x`.

**Next concept:** Inverse square root backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([0.5, 2.0, 4.0], dtype=DTYPE, requires_grad=True)
y = x**-1
dout = torch.tensor([1.0, -0.5, 3.0], dtype=DTYPE)
_capture("ex020", y, {"dx": x}, dout)
print("Forward and upstream shapes:", tuple(y.shape), tuple(dout.shape))


In [ ]:
# Exercise 020: derive manually; do not use autograd in this cell.
# Define `ex020_out_shape` — a Python tuple containing the forward output shape.
# Define `ex020_dx` — the manual gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex020_out_shape", "ex020", "out_shape")
_check_tensor("ex020_dx", "ex020", "dx")


### Exercise 021 — Inverse square root backward

**Purpose:** Learn how to differentiate the inverse square root used later in BatchNorm.

**Inputs:** Positive `x` and `dout` have shape `(3,)`.

**Forward operation:** `y = x**-0.5`.

**Derive/do:** Derive `dx`.

**Ingredients:** The power rule with exponent `-0.5`.

**Required outputs:**

- `ex021_out_shape`: a Python tuple containing the forward output shape.
- `ex021_dx`: the manual gradient with respect to `x`.

**Next concept:** Exponential backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([0.25, 1.0, 4.0], dtype=DTYPE, requires_grad=True)
y = x**-0.5
dout = torch.tensor([0.5, -2.0, 1.5], dtype=DTYPE)
_capture("ex021", y, {"dx": x}, dout)
print("Forward and upstream shapes:", tuple(y.shape), tuple(dout.shape))


In [ ]:
# Exercise 021: derive manually; do not use autograd in this cell.
# Define `ex021_out_shape` — a Python tuple containing the forward output shape.
# Define `ex021_dx` — the manual gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex021_out_shape", "ex021", "out_shape")
_check_tensor("ex021_dx", "ex021", "dx")


### Exercise 022 — Exponential backward

**Purpose:** Learn that the exponential's backward calculation can reuse its forward output.

**Inputs:** `x`, `y`, and `dout` have shape `(3,)`.

**Forward operation:** `y = x.exp()`.

**Derive/do:** Derive `dx`.

**Ingredients:** The derivative of the exponential and the available forward tensor `y`.

**Required outputs:**

- `ex022_out_shape`: a Python tuple containing the forward output shape.
- `ex022_dx`: the manual gradient with respect to `x`.

**Next concept:** Logarithm backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([-1.0, 0.0, 1.5], dtype=DTYPE, requires_grad=True)
y = x.exp()
dout = torch.tensor([2.0, -1.0, 0.5], dtype=DTYPE)
_capture("ex022", y, {"dx": x}, dout)
print("Forward and upstream shapes:", tuple(y.shape), tuple(dout.shape))


In [ ]:
# Exercise 022: derive manually; do not use autograd in this cell.
# Define `ex022_out_shape` — a Python tuple containing the forward output shape.
# Define `ex022_dx` — the manual gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex022_out_shape", "ex022", "out_shape")
_check_tensor("ex022_dx", "ex022", "dx")


### Exercise 023 — Logarithm backward

**Purpose:** Learn that the gradient through `log(x)` divides the incoming gradient by `x`.

**Inputs:** Positive `x`, `y`, and `dout` have shape `(3,)`.

**Forward operation:** `y = x.log()`.

**Derive/do:** Derive `dx`.

**Ingredients:** The derivative of natural logarithm and the elementwise chain rule.

**Required outputs:**

- `ex023_out_shape`: a Python tuple containing the forward output shape.
- `ex023_dx`: the manual gradient with respect to `x`.

**Next concept:** Tanh backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([0.5, 2.0, 5.0], dtype=DTYPE, requires_grad=True)
y = x.log()
dout = torch.tensor([-1.0, 0.25, 3.0], dtype=DTYPE)
_capture("ex023", y, {"dx": x}, dout)
print("Forward and upstream shapes:", tuple(y.shape), tuple(dout.shape))


In [ ]:
# Exercise 023: derive manually; do not use autograd in this cell.
# Define `ex023_out_shape` — a Python tuple containing the forward output shape.
# Define `ex023_dx` — the manual gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex023_out_shape", "ex023", "out_shape")
_check_tensor("ex023_dx", "ex023", "dx")


### Exercise 024 — Tanh backward

**Purpose:** Learn how an incoming gradient passes backward through `tanh`.

**Inputs:** `x`, `y`, and `dout` have shape `(4,)`.

**Forward operation:** `y = x.tanh()`.

**Derive/do:** Derive `dx`.

**Ingredients:** The tanh local derivative expressed using `y`, then the chain rule.

**Required outputs:**

- `ex024_out_shape`: a Python tuple containing the forward output shape.
- `ex024_dx`: the manual gradient with respect to `x`.

**Next concept:** Sigmoid backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([-2.0, -0.5, 0.5, 2.0], dtype=DTYPE, requires_grad=True)
y = x.tanh()
dout = torch.tensor([1.0, 2.0, -1.0, 0.5], dtype=DTYPE)
_capture("ex024", y, {"dx": x}, dout)
print("Forward and upstream shapes:", tuple(y.shape), tuple(dout.shape))


In [ ]:
# Exercise 024: derive manually; do not use autograd in this cell.
# Define `ex024_out_shape` — a Python tuple containing the forward output shape.
# Define `ex024_dx` — the manual gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex024_out_shape", "ex024", "out_shape")
_check_tensor("ex024_dx", "ex024", "dx")


### Exercise 025 — Sigmoid backward

**Purpose:** Learn how an incoming gradient passes backward through `sigmoid`.

**Inputs:** `x`, `y`, and `dout` have shape `(3,)`.

**Forward operation:** `y = x.sigmoid()`.

**Derive/do:** Derive `dx`.

**Ingredients:** The sigmoid local slope written with `y` and `1 - y`.

**Required outputs:**

- `ex025_out_shape`: a Python tuple containing the forward output shape.
- `ex025_dx`: the manual gradient with respect to `x`.

**Next concept:** ReLU backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([-1.5, 0.0, 2.0], dtype=DTYPE, requires_grad=True)
y = x.sigmoid()
dout = torch.tensor([2.0, -1.0, 0.25], dtype=DTYPE)
_capture("ex025", y, {"dx": x}, dout)
print("Forward and upstream shapes:", tuple(y.shape), tuple(dout.shape))


In [ ]:
# Exercise 025: derive manually; do not use autograd in this cell.
# Define `ex025_out_shape` — a Python tuple containing the forward output shape.
# Define `ex025_dx` — the manual gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [17]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex025_out_shape", "ex025", "out_shape")
_check_tensor("ex025_dx", "ex025", "dx")


AssertionError: Define `ex025_out_shape` in the answer cell first.

### Exercise 026 — ReLU backward

**Purpose:** Learn that ReLU passes gradients through positive inputs and blocks them at negative inputs.

**Inputs:** `x` contains no zero and has shape `(4,)`; `dout` matches it.

**Forward operation:** `y = x.relu()`.

**Derive/do:** Derive `dx`.

**Ingredients:** A Boolean mask for positive entries, converted by multiplication with `dout`.

**Required outputs:**

- `ex026_out_shape`: a Python tuple containing the forward output shape.
- `ex026_dx`: the manual gradient with respect to `x`.

**Next concept:** Elementwise addition backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([-3.0, -0.5, 1.0, 4.0], dtype=DTYPE, requires_grad=True)
y = x.relu()
dout = torch.tensor([1.0, 2.0, -2.0, 0.5], dtype=DTYPE)
_capture("ex026", y, {"dx": x}, dout)
print("Forward and upstream shapes:", tuple(y.shape), tuple(dout.shape))


In [ ]:
# Exercise 026: derive manually; do not use autograd in this cell.
# Define `ex026_out_shape` — a Python tuple containing the forward output shape.
# Define `ex026_dx` — the manual gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex026_out_shape", "ex026", "out_shape")
_check_tensor("ex026_dx", "ex026", "dx")


### Exercise 027 — Elementwise addition backward

**Purpose:** Learn that an addition sends the same incoming gradient to both of its inputs.

**Inputs:** `x`, `z`, and `dout` have shape `(3,)`.

**Forward operation:** `y = x + z`.

**Derive/do:** Derive both input gradients.

**Ingredients:** Treat each input as changing while the other is held fixed.

**Required outputs:**

- `ex027_out_shape`: a Python tuple containing the forward output shape.
- `ex027_dx`: the manual gradient with respect to `x`.
- `ex027_dz`: the manual gradient with respect to `z`.

**Next concept:** Elementwise subtraction backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([1.0, 2.0, 3.0], dtype=DTYPE, requires_grad=True)
z = torch.tensor([-2.0, 0.5, 4.0], dtype=DTYPE, requires_grad=True)
y = x + z
dout = torch.tensor([0.5, -1.0, 2.0], dtype=DTYPE)
_capture("ex027", y, {"dx": x, "dz": z}, dout)
print("All forward tensors have shape", tuple(y.shape))


In [ ]:
# Exercise 027: derive manually; do not use autograd in this cell.
# Define `ex027_out_shape` — a Python tuple containing the forward output shape.
# Define `ex027_dx` — the manual gradient with respect to `x`.
# Define `ex027_dz` — the manual gradient with respect to `z`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex027_out_shape", "ex027", "out_shape")
_check_tensor("ex027_dx", "ex027", "dx")
_check_tensor("ex027_dz", "ex027", "dz")


### Exercise 028 — Elementwise subtraction backward

**Purpose:** Learn that subtraction sends opposite-signed gradients to its left and right inputs.

**Inputs:** `x`, `z`, and `dout` have shape `(3,)`.

**Forward operation:** `y = x - z`.

**Derive/do:** Derive both input gradients.

**Ingredients:** The local slope with respect to the left input and with respect to the right input.

**Required outputs:**

- `ex028_out_shape`: a Python tuple containing the forward output shape.
- `ex028_dx`: the manual gradient with respect to `x`.
- `ex028_dz`: the manual gradient with respect to `z`.

**Next concept:** Elementwise multiplication backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([3.0, 1.0, -2.0], dtype=DTYPE, requires_grad=True)
z = torch.tensor([1.0, -4.0, 0.5], dtype=DTYPE, requires_grad=True)
y = x - z
dout = torch.tensor([2.0, -0.5, 1.0], dtype=DTYPE)
_capture("ex028", y, {"dx": x, "dz": z}, dout)
print("All forward tensors have shape", tuple(y.shape))


In [ ]:
# Exercise 028: derive manually; do not use autograd in this cell.
# Define `ex028_out_shape` — a Python tuple containing the forward output shape.
# Define `ex028_dx` — the manual gradient with respect to `x`.
# Define `ex028_dz` — the manual gradient with respect to `z`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex028_out_shape", "ex028", "out_shape")
_check_tensor("ex028_dx", "ex028", "dx")
_check_tensor("ex028_dz", "ex028", "dz")


### Exercise 029 — Elementwise multiplication backward

**Purpose:** Learn that each input of a multiplication uses the other input when computing its gradient.

**Inputs:** `x`, `z`, and `dout` have shape `(3,)`.

**Forward operation:** `y = x * z`.

**Derive/do:** Derive `dx` and `dz`.

**Ingredients:** For each input, identify the other forward factor, then apply `dout`.

**Required outputs:**

- `ex029_out_shape`: a Python tuple containing the forward output shape.
- `ex029_dx`: the manual gradient with respect to `x`.
- `ex029_dz`: the manual gradient with respect to `z`.

**Next concept:** Elementwise division backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([2.0, -1.0, 3.0], dtype=DTYPE, requires_grad=True)
z = torch.tensor([4.0, 5.0, -2.0], dtype=DTYPE, requires_grad=True)
y = x * z
dout = torch.tensor([0.5, -2.0, 1.5], dtype=DTYPE)
_capture("ex029", y, {"dx": x, "dz": z}, dout)
print("All forward tensors have shape", tuple(y.shape))


In [ ]:
# Exercise 029: derive manually; do not use autograd in this cell.
# Define `ex029_out_shape` — a Python tuple containing the forward output shape.
# Define `ex029_dx` — the manual gradient with respect to `x`.
# Define `ex029_dz` — the manual gradient with respect to `z`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex029_out_shape", "ex029", "out_shape")
_check_tensor("ex029_dx", "ex029", "dx")
_check_tensor("ex029_dz", "ex029", "dz")


### Exercise 030 — Elementwise division backward

**Purpose:** Learn that division gives different gradient rules for the numerator and denominator.

**Inputs:** Positive `z`; `x`, `z`, and `dout` have shape `(3,)`.

**Forward operation:** `y = x / z`.

**Derive/do:** Derive gradients for numerator and denominator.

**Ingredients:** Rewrite division as multiplication by `z**-1`; differentiate each input separately.

**Required outputs:**

- `ex030_out_shape`: a Python tuple containing the forward output shape.
- `ex030_dx`: the manual gradient with respect to `x`.
- `ex030_dz`: the manual gradient with respect to `z`.

**Next concept:** Fractional fixed power backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([2.0, -3.0, 5.0], dtype=DTYPE, requires_grad=True)
z = torch.tensor([0.5, 2.0, 4.0], dtype=DTYPE, requires_grad=True)
y = x / z
dout = torch.tensor([1.0, -0.5, 2.0], dtype=DTYPE)
_capture("ex030", y, {"dx": x, "dz": z}, dout)
print("All forward tensors have shape", tuple(y.shape))


In [ ]:
# Exercise 030: derive manually; do not use autograd in this cell.
# Define `ex030_out_shape` — a Python tuple containing the forward output shape.
# Define `ex030_dx` — the manual gradient with respect to `x`.
# Define `ex030_dz` — the manual gradient with respect to `z`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex030_out_shape", "ex030", "out_shape")
_check_tensor("ex030_dx", "ex030", "dx")
_check_tensor("ex030_dz", "ex030", "dz")


### Exercise 031 — Fractional fixed power backward

**Purpose:** Learn to differentiate a fixed power even when its exponent is not a whole number.

**Inputs:** Positive `x` and same-shape `dout` have shape `(3,)`.

**Forward operation:** `y = x**2.5`.

**Derive/do:** Derive `dx`.

**Ingredients:** The fixed-exponent power rule and `dout`.

**Required outputs:**

- `ex031_out_shape`: a Python tuple containing the forward output shape.
- `ex031_dx`: the manual gradient with respect to `x`.

**Next concept:** Elementwise affine operation.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([0.25, 1.0, 4.0], dtype=DTYPE, requires_grad=True)
y = x**2.5
dout = torch.tensor([2.0, -1.0, 0.5], dtype=DTYPE)
_capture("ex031", y, {"dx": x}, dout)
print("Forward and upstream shapes:", tuple(y.shape), tuple(dout.shape))


In [ ]:
# Exercise 031: derive manually; do not use autograd in this cell.
# Define `ex031_out_shape` — a Python tuple containing the forward output shape.
# Define `ex031_dx` — the manual gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex031_out_shape", "ex031", "out_shape")
_check_tensor("ex031_dx", "ex031", "dx")


### Exercise 032 — Elementwise affine operation

**Purpose:** Learn how `x * scale + shift` sends gradients to `x`, `scale`, and `shift`.

**Inputs:** `x`, `scale`, `shift`, and `dout` have shape `(3,)`.

**Forward operation:** `y = x * scale + shift`.

**Derive/do:** Derive gradients for all three inputs.

**Ingredients:** Backward through addition first, then multiplication; each operation is elementwise here.

**Required outputs:**

- `ex032_out_shape`: a Python tuple containing the forward output shape.
- `ex032_dx`: the manual gradient with respect to `x`.
- `ex032_dscale`: the manual gradient with respect to `scale`.
- `ex032_dshift`: the manual gradient with respect to `shift`.

**Next concept:** Vector sum backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([1.0, -2.0, 0.5], dtype=DTYPE, requires_grad=True)
scale = torch.tensor([2.0, 3.0, -4.0], dtype=DTYPE, requires_grad=True)
shift = torch.tensor([0.5, 1.0, -1.0], dtype=DTYPE, requires_grad=True)
y = x * scale + shift
dout = torch.tensor([1.0, -0.5, 2.0], dtype=DTYPE)
_capture("ex032", y, {"dx": x, "dscale": scale, "dshift": shift}, dout)
print("All tensors in this elementwise graph have shape", tuple(y.shape))


In [ ]:
# Exercise 032: derive manually; do not use autograd in this cell.
# Define `ex032_out_shape` — a Python tuple containing the forward output shape.
# Define `ex032_dx` — the manual gradient with respect to `x`.
# Define `ex032_dscale` — the manual gradient with respect to `scale`.
# Define `ex032_dshift` — the manual gradient with respect to `shift`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex032_out_shape", "ex032", "out_shape")
_check_tensor("ex032_dx", "ex032", "dx")
_check_tensor("ex032_dscale", "ex032", "dscale")
_check_tensor("ex032_dshift", "ex032", "dshift")


## 3. Reductions: backward broadcasts what forward removed

A reduction combines several input positions into fewer output positions. In backward, each reduced output gradient is distributed to every input position that contributed to it.

For a mean over `m` values, the local contribution to each input includes the factor

$$
\frac{1}{m}
$$

Here, `m` is the number of values in that particular reduction, not necessarily the batch size. Keep distinguishing totals from averages and always restore the input gradient's original shape.


### Exercise 033 — Vector sum backward

**Purpose:** Learn that a scalar gradient from a vector sum is copied back to every vector entry.

**Inputs:** `x` has shape `(4,)`; `y` and `dout` are scalar tensors.

**Forward operation:** `y = x.sum()`.

**Derive/do:** Derive `dx`.

**Ingredients:** Each input contributes once to the total.

**Required outputs:**

- `ex033_out_shape`: a Python tuple containing the forward output shape.
- `ex033_dx`: the length-4 gradient with respect to `x`.

**Next concept:** Vector mean backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([1.0, -2.0, 3.0, 4.0], dtype=DTYPE, requires_grad=True)
y = x.sum()
dout = torch.tensor(2.5, dtype=DTYPE)
_capture("ex033", y, {"dx": x}, dout)
print("x", tuple(x.shape), "y", tuple(y.shape), "dout", tuple(dout.shape))


In [ ]:
# Exercise 033: derive manually; do not use autograd in this cell.
# Define `ex033_out_shape` — a Python tuple containing the forward output shape.
# Define `ex033_dx` — the length-4 gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex033_out_shape", "ex033", "out_shape")
_check_tensor("ex033_dx", "ex033", "dx")


### Exercise 034 — Vector mean backward

**Purpose:** Learn that a vector mean shares the incoming gradient equally among all vector entries.

**Inputs:** `x` has four entries; `y` and `dout` are scalar tensors.

**Forward operation:** `y = x.mean()`.

**Derive/do:** Derive `dx` and identify the number of averaged values.

**Ingredients:** Mean equals sum divided by element count.

**Required outputs:**

- `ex034_out_shape`: a Python tuple containing the forward output shape.
- `ex034_dx`: the length-4 gradient with respect to `x`.

**Next concept:** Weighted sum backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([2.0, 4.0, -1.0, 3.0], dtype=DTYPE, requires_grad=True)
y = x.mean()
dout = torch.tensor(-2.0, dtype=DTYPE)
_capture("ex034", y, {"dx": x}, dout)
print("x", tuple(x.shape), "y", tuple(y.shape), "dout", tuple(dout.shape))


In [ ]:
# Exercise 034: derive manually; do not use autograd in this cell.
# Define `ex034_out_shape` — a Python tuple containing the forward output shape.
# Define `ex034_dx` — the length-4 gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex034_out_shape", "ex034", "out_shape")
_check_tensor("ex034_dx", "ex034", "dx")


### Exercise 035 — Weighted sum backward

**Purpose:** Learn to reverse a weighted sum by undoing the sum first and the elementwise multiplication second.

**Inputs:** `x` and `weights` have shape `(4,)`; the output is scalar.

**Forward operation:** `y = (x * weights).sum()`.

**Derive/do:** Derive both gradients.

**Ingredients:** Reverse the sum, then reverse the elementwise product.

**Required outputs:**

- `ex035_out_shape`: a Python tuple containing the forward output shape.
- `ex035_dx`: the gradient with respect to `x`.
- `ex035_dweights`: the gradient with respect to `weights`.

**Next concept:** Matrix total backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([1.0, 2.0, -1.0, 3.0], dtype=DTYPE, requires_grad=True)
weights = torch.tensor([0.5, -2.0, 4.0, 1.5], dtype=DTYPE, requires_grad=True)
y = (x * weights).sum()
dout = torch.tensor(1.25, dtype=DTYPE)
_capture("ex035", y, {"dx": x, "dweights": weights}, dout)
print("Inputs", tuple(x.shape), "output", tuple(y.shape))


In [ ]:
# Exercise 035: derive manually; do not use autograd in this cell.
# Define `ex035_out_shape` — a Python tuple containing the forward output shape.
# Define `ex035_dx` — the gradient with respect to `x`.
# Define `ex035_dweights` — the gradient with respect to `weights`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex035_out_shape", "ex035", "out_shape")
_check_tensor("ex035_dx", "ex035", "dx")
_check_tensor("ex035_dweights", "ex035", "dweights")


### Exercise 036 — Matrix total backward

**Purpose:** Learn that the gradient of a matrix total must return to every row-column position.

**Inputs:** `x` has shape `(2, 3)`; the output and upstream gradient are scalar.

**Forward operation:** `y = x.sum()`.

**Derive/do:** Derive `dx` with shape `(2, 3)`.

**Ingredients:** Every matrix entry contributes once to the scalar total.

**Required outputs:**

- `ex036_out_shape`: a Python tuple containing the forward output shape.
- `ex036_dx`: the matrix gradient with respect to `x`.

**Next concept:** Matrix mean backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.arange(1, 7, dtype=DTYPE).reshape(2, 3).requires_grad_()
y = x.sum()
dout = torch.tensor(-0.75, dtype=DTYPE)
_capture("ex036", y, {"dx": x}, dout)
print("x", tuple(x.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 036: derive manually; do not use autograd in this cell.
# Define `ex036_out_shape` — a Python tuple containing the forward output shape.
# Define `ex036_dx` — the matrix gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex036_out_shape", "ex036", "out_shape")
_check_tensor("ex036_dx", "ex036", "dx")


### Exercise 037 — Matrix mean backward

**Purpose:** Learn that a matrix mean divides the incoming gradient among every matrix entry.

**Inputs:** `x` has six entries in shape `(2, 3)`.

**Forward operation:** `y = x.mean()`.

**Derive/do:** Derive `dx`; use the total number of averaged entries.

**Ingredients:** Mean reduction over all axes and a scalar upstream gradient.

**Required outputs:**

- `ex037_out_shape`: a Python tuple containing the forward output shape.
- `ex037_dx`: the matrix gradient with respect to `x`.

**Next concept:** Column sums backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.arange(1, 7, dtype=DTYPE).reshape(2, 3).requires_grad_()
y = x.mean()
dout = torch.tensor(3.0, dtype=DTYPE)
_capture("ex037", y, {"dx": x}, dout)
print("x", tuple(x.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 037: derive manually; do not use autograd in this cell.
# Define `ex037_out_shape` — a Python tuple containing the forward output shape.
# Define `ex037_dx` — the matrix gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex037_out_shape", "ex037", "out_shape")
_check_tensor("ex037_dx", "ex037", "dx")


### Exercise 038 — Column sums backward

**Purpose:** Learn that each column-sum gradient is copied back down all rows of that column.

**Inputs:** `x` has shape `(2, 3)`; `y` and `dout` have shape `(3,)`.

**Forward operation:** `y = x.sum(dim=0)`.

**Derive/do:** Derive `dx`.

**Ingredients:** Dimension 0 was reduced; determine which upstream entry belongs to each input column.

**Required outputs:**

- `ex038_out_shape`: a Python tuple containing the forward output shape.
- `ex038_dx`: the `(2, 3)` gradient with respect to `x`.

**Next concept:** Row sums backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.arange(1, 7, dtype=DTYPE).reshape(2, 3).requires_grad_()
y = x.sum(dim=0)
dout = torch.tensor([1.0, -2.0, 0.5], dtype=DTYPE)
_capture("ex038", y, {"dx": x}, dout)
print("x", tuple(x.shape), "y", tuple(y.shape), "dout", tuple(dout.shape))


In [ ]:
# Exercise 038: derive manually; do not use autograd in this cell.
# Define `ex038_out_shape` — a Python tuple containing the forward output shape.
# Define `ex038_dx` — the `(2, 3)` gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex038_out_shape", "ex038", "out_shape")
_check_tensor("ex038_dx", "ex038", "dx")


### Exercise 039 — Row sums backward

**Purpose:** Learn that each row-sum gradient is copied back across all columns of that row.

**Inputs:** `x` has shape `(2, 3)`; `y` and `dout` have shape `(2,)`.

**Forward operation:** `y = x.sum(dim=1)`.

**Derive/do:** Derive `dx`.

**Ingredients:** Dimension 1 was reduced; introduce an axis if needed to broadcast row gradients.

**Required outputs:**

- `ex039_out_shape`: a Python tuple containing the forward output shape.
- `ex039_dx`: the `(2, 3)` gradient with respect to `x`.

**Next concept:** Row sums with keepdim backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.arange(1, 7, dtype=DTYPE).reshape(2, 3).requires_grad_()
y = x.sum(dim=1)
dout = torch.tensor([2.0, -1.0], dtype=DTYPE)
_capture("ex039", y, {"dx": x}, dout)
print("x", tuple(x.shape), "y", tuple(y.shape), "dout", tuple(dout.shape))


In [ ]:
# Exercise 039: derive manually; do not use autograd in this cell.
# Define `ex039_out_shape` — a Python tuple containing the forward output shape.
# Define `ex039_dx` — the `(2, 3)` gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex039_out_shape", "ex039", "out_shape")
_check_tensor("ex039_dx", "ex039", "dx")


### Exercise 040 — Row sums with keepdim backward

**Purpose:** Learn how keeping a reduced axis at length 1 makes the backward shape easier to reuse.

**Inputs:** `x` has shape `(2, 3)`; `y` and `dout` have shape `(2, 1)`.

**Forward operation:** `y = x.sum(dim=1, keepdim=True)`.

**Derive/do:** Derive `dx`.

**Ingredients:** The upstream tensor already has the singleton column axis needed for expansion.

**Required outputs:**

- `ex040_out_shape`: a Python tuple containing the forward output shape.
- `ex040_dx`: the `(2, 3)` gradient with respect to `x`.

**Next concept:** Column means backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.arange(1, 7, dtype=DTYPE).reshape(2, 3).requires_grad_()
y = x.sum(dim=1, keepdim=True)
dout = torch.tensor([[2.0], [-1.0]], dtype=DTYPE)
_capture("ex040", y, {"dx": x}, dout)
print("x", tuple(x.shape), "y", tuple(y.shape), "dout", tuple(dout.shape))


In [ ]:
# Exercise 040: derive manually; do not use autograd in this cell.
# Define `ex040_out_shape` — a Python tuple containing the forward output shape.
# Define `ex040_dx` — the `(2, 3)` gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex040_out_shape", "ex040", "out_shape")
_check_tensor("ex040_dx", "ex040", "dx")


### Exercise 041 — Column means backward

**Purpose:** Learn that each column-mean gradient is copied down the column and divided by the number of rows.

**Inputs:** `x` has shape `(4, 3)`; `y` and `dout` have shape `(3,)`.

**Forward operation:** `y = x.mean(dim=0)`.

**Derive/do:** Derive `dx`.

**Ingredients:** There are four training-example rows in each column mean.

**Required outputs:**

- `ex041_out_shape`: a Python tuple containing the forward output shape.
- `ex041_dx`: the `(4, 3)` gradient with respect to `x`.

**Next concept:** Row means with keepdim backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.arange(12, dtype=DTYPE).reshape(4, 3).requires_grad_()
y = x.mean(dim=0)
dout = torch.tensor([1.0, -2.0, 0.5], dtype=DTYPE)
_capture("ex041", y, {"dx": x}, dout)
print("x", tuple(x.shape), "y", tuple(y.shape), "dout", tuple(dout.shape))


In [ ]:
# Exercise 041: derive manually; do not use autograd in this cell.
# Define `ex041_out_shape` — a Python tuple containing the forward output shape.
# Define `ex041_dx` — the `(4, 3)` gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex041_out_shape", "ex041", "out_shape")
_check_tensor("ex041_dx", "ex041", "dx")


### Exercise 042 — Row means with keepdim backward

**Purpose:** Learn that each row-mean gradient is copied across the row and divided by the number of columns.

**Inputs:** `x` has shape `(3, 4)`; `y` and `dout` have shape `(3, 1)`.

**Forward operation:** `y = x.mean(dim=1, keepdim=True)`.

**Derive/do:** Derive `dx`.

**Ingredients:** Each row mean averages four feature entries.

**Required outputs:**

- `ex042_out_shape`: a Python tuple containing the forward output shape.
- `ex042_dx`: the `(3, 4)` gradient with respect to `x`.

**Next concept:** Sum of squares.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.arange(12, dtype=DTYPE).reshape(3, 4).requires_grad_()
y = x.mean(dim=1, keepdim=True)
dout = torch.tensor([[1.0], [-2.0], [0.5]], dtype=DTYPE)
_capture("ex042", y, {"dx": x}, dout)
print("x", tuple(x.shape), "y", tuple(y.shape), "dout", tuple(dout.shape))


In [ ]:
# Exercise 042: derive manually; do not use autograd in this cell.
# Define `ex042_out_shape` — a Python tuple containing the forward output shape.
# Define `ex042_dx` — the `(3, 4)` gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex042_out_shape", "ex042", "out_shape")
_check_tensor("ex042_dx", "ex042", "dx")


### Exercise 043 — Sum of squares

**Purpose:** Learn to reverse a scalar sum of squared tensor entries.

**Inputs:** `x` has shape `(4,)`; `y` is scalar.

**Forward operation:** `y = (x**2).sum()`.

**Derive/do:** Derive `dx` by reversing the sum and square.

**Ingredients:** A scalar upstream gradient and the square local derivative.

**Required outputs:**

- `ex043_out_shape`: a Python tuple containing the forward output shape.
- `ex043_dx`: the gradient with respect to `x`.

**Next concept:** Row-wise sums of squares.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([-2.0, -0.5, 1.0, 3.0], dtype=DTYPE, requires_grad=True)
y = (x**2).sum()
dout = torch.tensor(1.5, dtype=DTYPE)
_capture("ex043", y, {"dx": x}, dout)
print("x", tuple(x.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 043: derive manually; do not use autograd in this cell.
# Define `ex043_out_shape` — a Python tuple containing the forward output shape.
# Define `ex043_dx` — the gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex043_out_shape", "ex043", "out_shape")
_check_tensor("ex043_dx", "ex043", "dx")


### Exercise 044 — Row-wise sums of squares

**Purpose:** Learn to reverse one separate sum of squares for each row.

**Inputs:** `x` has shape `(2, 3)`; `y` and `dout` have shape `(2,)`.

**Forward operation:** `y = (x**2).sum(dim=1)`.

**Derive/do:** Derive `dx`.

**Ingredients:** Expand each row's upstream gradient, then reverse the square.

**Required outputs:**

- `ex044_out_shape`: a Python tuple containing the forward output shape.
- `ex044_dx`: the `(2, 3)` gradient with respect to `x`.

**Next concept:** Column means of squares.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([[-2.0, 1.0, 3.0], [0.5, -1.0, 2.0]], dtype=DTYPE, requires_grad=True)
y = (x**2).sum(dim=1)
dout = torch.tensor([2.0, -0.5], dtype=DTYPE)
_capture("ex044", y, {"dx": x}, dout)
print("x", tuple(x.shape), "y", tuple(y.shape), "dout", tuple(dout.shape))


In [ ]:
# Exercise 044: derive manually; do not use autograd in this cell.
# Define `ex044_out_shape` — a Python tuple containing the forward output shape.
# Define `ex044_dx` — the `(2, 3)` gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex044_out_shape", "ex044", "out_shape")
_check_tensor("ex044_dx", "ex044", "dx")


### Exercise 045 — Column means of squares

**Purpose:** Learn to reverse a square followed by one mean for each column.

**Inputs:** `x` has shape `(4, 3)`; `y` and `dout` have shape `(3,)`.

**Forward operation:** `y = (x**2).mean(dim=0)`.

**Derive/do:** Derive `dx`.

**Ingredients:** Reverse the mean over four rows, then reverse the square.

**Required outputs:**

- `ex045_out_shape`: a Python tuple containing the forward output shape.
- `ex045_dx`: the `(4, 3)` gradient with respect to `x`.

**Next concept:** Unique vector maximum backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.linspace(-2.0, 3.5, steps=12, dtype=DTYPE).reshape(4, 3).requires_grad_()
y = (x**2).mean(dim=0)
dout = torch.tensor([1.0, -1.5, 0.25], dtype=DTYPE)
_capture("ex045", y, {"dx": x}, dout)
print("x", tuple(x.shape), "y", tuple(y.shape), "dout", tuple(dout.shape))


In [ ]:
# Exercise 045: derive manually; do not use autograd in this cell.
# Define `ex045_out_shape` — a Python tuple containing the forward output shape.
# Define `ex045_dx` — the `(4, 3)` gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex045_out_shape", "ex045", "out_shape")
_check_tensor("ex045_dx", "ex045", "dx")


### Exercise 046 — Unique vector maximum backward

**Purpose:** Learn that only the entry that won a unique maximum receives the gradient.

**Inputs:** `x` has shape `(4,)` with one unique maximum; `y` is scalar.

**Forward operation:** `y = x.max()`.

**Derive/do:** Derive `dx`.

**Ingredients:** Build a mask for the unique maximum and apply the scalar upstream gradient.

**Required outputs:**

- `ex046_out_shape`: a Python tuple containing the forward output shape.
- `ex046_dx`: the gradient with respect to `x`.

**Next concept:** Unique row maxima backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([-1.0, 3.0, 2.0, 0.5], dtype=DTYPE, requires_grad=True)
y = x.max()
dout = torch.tensor(2.0, dtype=DTYPE)
_capture("ex046", y, {"dx": x}, dout)
print("x", tuple(x.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 046: derive manually; do not use autograd in this cell.
# Define `ex046_out_shape` — a Python tuple containing the forward output shape.
# Define `ex046_dx` — the gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex046_out_shape", "ex046", "out_shape")
_check_tensor("ex046_dx", "ex046", "dx")


### Exercise 047 — Unique row maxima backward

**Purpose:** Learn that a row-wise maximum sends each row's gradient only to that row's winning entry.

**Inputs:** `x` has shape `(3, 4)` and each row has one unique maximum; `y` has shape `(3,)`.

**Forward operation:** `y = x.max(dim=1).values`.

**Derive/do:** Derive `dx`.

**Ingredients:** One maximum mask per row and one upstream value per row.

**Required outputs:**

- `ex047_out_shape`: a Python tuple containing the forward output shape.
- `ex047_dx`: the `(3, 4)` gradient with respect to `x`.

**Next concept:** Reduction chain.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([[1.0, 4.0, 2.0, 0.0], [3.0, -1.0, 5.0, 2.0], [7.0, 1.0, 0.0, 6.0]], dtype=DTYPE, requires_grad=True)
y = x.max(dim=1).values
dout = torch.tensor([1.0, -2.0, 0.5], dtype=DTYPE)
_capture("ex047", y, {"dx": x}, dout)
print("x", tuple(x.shape), "y", tuple(y.shape), "dout", tuple(dout.shape))


In [ ]:
# Exercise 047: derive manually; do not use autograd in this cell.
# Define `ex047_out_shape` — a Python tuple containing the forward output shape.
# Define `ex047_dx` — the `(3, 4)` gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex047_out_shape", "ex047", "out_shape")
_check_tensor("ex047_dx", "ex047", "dx")


### Exercise 048 — Reduction chain

**Purpose:** Learn to reverse two reductions in the opposite order from the forward pass.

**Inputs:** `x` has shape `(2, 3)`; the final output is scalar.

**Forward operation:** `columns = x.sum(dim=0)` and `y = columns.mean()`.

**Derive/do:** Derive gradients for `columns` and `x`.

**Ingredients:** Reverse the mean over three column totals, then reverse the row reduction.

**Required outputs:**

- `ex048_out_shape`: a Python tuple containing the forward output shape.
- `ex048_dcolumns`: the gradient with respect to the three column totals.
- `ex048_dx`: the gradient with respect to `x`.

**Next concept:** Add a row bias.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.arange(1, 7, dtype=DTYPE).reshape(2, 3).requires_grad_()
columns = x.sum(dim=0)
y = columns.mean()
dout = torch.tensor(-1.5, dtype=DTYPE)
_capture("ex048", y, {"dcolumns": columns, "dx": x}, dout)
print("x", tuple(x.shape), "columns", tuple(columns.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 048: derive manually; do not use autograd in this cell.
# Define `ex048_out_shape` — a Python tuple containing the forward output shape.
# Define `ex048_dcolumns` — the gradient with respect to the three column totals.
# Define `ex048_dx` — the gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex048_out_shape", "ex048", "out_shape")
_check_tensor("ex048_dcolumns", "ex048", "dcolumns")
_check_tensor("ex048_dx", "ex048", "dx")


## 4. Broadcasting: backward must unbroadcast

Broadcasting reuses a smaller tensor at many output positions. Each reuse creates a path to the objective. Backward must add all path contributions until the gradient has the original input shape.

Use this rule:

> If an axis had size 1 before forward broadcasting, sum the gradient over that expanded axis in backward and keep the axis when the original input kept it.

Never stop at a broadcast output shape when computing the gradient of a smaller input.


### Exercise 049 — Add a row bias

**Purpose:** Learn that one feature bias is reused by every example, so its gradient adds contributions from all rows.

**Inputs:** `x` has shape `(3, 4)`, `bias` has shape `(4,)`, and `dout` has shape `(3, 4)`.

**Forward operation:** `y = x + bias`.

**Derive/do:** Derive `dx` and `dbias`.

**Ingredients:** The bias is reused down the three-row axis; its gradient must return to shape `(4,)`.

**Required outputs:**

- `ex049_out_shape`: a Python tuple containing the forward output shape.
- `ex049_dx`: the gradient with respect to `x`.
- `ex049_dbias`: the unbroadcast gradient with respect to `bias`.

**Next concept:** Add a column bias.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.arange(12, dtype=DTYPE).reshape(3, 4).requires_grad_()
bias = torch.tensor([0.5, -1.0, 2.0, 3.0], dtype=DTYPE, requires_grad=True)
y = x + bias
dout = torch.linspace(-1.0, 1.2, steps=12, dtype=DTYPE).reshape(3, 4)
_capture("ex049", y, {"dx": x, "dbias": bias}, dout)
print("x", tuple(x.shape), "bias", tuple(bias.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 049: derive manually; do not use autograd in this cell.
# Define `ex049_out_shape` — a Python tuple containing the forward output shape.
# Define `ex049_dx` — the gradient with respect to `x`.
# Define `ex049_dbias` — the unbroadcast gradient with respect to `bias`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex049_out_shape", "ex049", "out_shape")
_check_tensor("ex049_dx", "ex049", "dx")
_check_tensor("ex049_dbias", "ex049", "dbias")


### Exercise 050 — Add a column bias

**Purpose:** Learn that a `(3, 1)` value reused across columns receives one summed gradient per row.

**Inputs:** `x` has shape `(3, 4)`, `bias` has shape `(3, 1)`, and `dout` has shape `(3, 4)`.

**Forward operation:** `y = x + bias`.

**Derive/do:** Derive `dx` and `dbias`.

**Ingredients:** The size-one column axis expanded from 1 to 4; restore it in backward.

**Required outputs:**

- `ex050_out_shape`: a Python tuple containing the forward output shape.
- `ex050_dx`: the gradient with respect to `x`.
- `ex050_dbias`: the `(3, 1)` gradient with respect to `bias`.

**Next concept:** Add a scalar parameter.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.arange(12, dtype=DTYPE).reshape(3, 4).requires_grad_()
bias = torch.tensor([[0.5], [-1.0], [2.0]], dtype=DTYPE, requires_grad=True)
y = x + bias
dout = torch.linspace(-1.0, 1.2, steps=12, dtype=DTYPE).reshape(3, 4)
_capture("ex050", y, {"dx": x, "dbias": bias}, dout)
print("x", tuple(x.shape), "bias", tuple(bias.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 050: derive manually; do not use autograd in this cell.
# Define `ex050_out_shape` — a Python tuple containing the forward output shape.
# Define `ex050_dx` — the gradient with respect to `x`.
# Define `ex050_dbias` — the `(3, 1)` gradient with respect to `bias`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex050_out_shape", "ex050", "out_shape")
_check_tensor("ex050_dx", "ex050", "dx")
_check_tensor("ex050_dbias", "ex050", "dbias")


### Exercise 051 — Add a scalar parameter

**Purpose:** Learn that a scalar reused at every matrix position receives all of those gradient contributions added together.

**Inputs:** `x` has shape `(2, 3)`, `shift` has shape `()`, and `dout` has shape `(2, 3)`.

**Forward operation:** `y = x + shift`.

**Derive/do:** Derive `dx` and scalar `dshift`.

**Ingredients:** The scalar was reused at all six output positions.

**Required outputs:**

- `ex051_out_shape`: a Python tuple containing the forward output shape.
- `ex051_dx`: the gradient with respect to `x`.
- `ex051_dshift`: the scalar gradient with respect to `shift`.

**Next concept:** Multiply by a row scale.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.arange(6, dtype=DTYPE).reshape(2, 3).requires_grad_()
shift = torch.tensor(1.5, dtype=DTYPE, requires_grad=True)
y = x + shift
dout = torch.tensor([[1.0, -2.0, 0.5], [3.0, 1.5, -1.0]], dtype=DTYPE)
_capture("ex051", y, {"dx": x, "dshift": shift}, dout)
print("x", tuple(x.shape), "shift", tuple(shift.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 051: derive manually; do not use autograd in this cell.
# Define `ex051_out_shape` — a Python tuple containing the forward output shape.
# Define `ex051_dx` — the gradient with respect to `x`.
# Define `ex051_dshift` — the scalar gradient with respect to `shift`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex051_out_shape", "ex051", "out_shape")
_check_tensor("ex051_dx", "ex051", "dx")
_check_tensor("ex051_dshift", "ex051", "dshift")


### Exercise 052 — Multiply by a row scale

**Purpose:** Learn that one feature scale reused by every row receives contributions from all rows.

**Inputs:** `x` has shape `(3, 4)`, `scale` has shape `(4,)`, and `dout` has shape `(3, 4)`.

**Forward operation:** `y = x * scale`.

**Derive/do:** Derive `dx` and `dscale`.

**Ingredients:** First derive per-output product contributions; then sum scale contributions over examples.

**Required outputs:**

- `ex052_out_shape`: a Python tuple containing the forward output shape.
- `ex052_dx`: the gradient with respect to `x`.
- `ex052_dscale`: the unbroadcast gradient with respect to `scale`.

**Next concept:** Multiply by a column scale.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.arange(1, 13, dtype=DTYPE).reshape(3, 4).requires_grad_()
scale = torch.tensor([0.5, -1.0, 2.0, 3.0], dtype=DTYPE, requires_grad=True)
y = x * scale
dout = torch.linspace(-0.5, 1.7, steps=12, dtype=DTYPE).reshape(3, 4)
_capture("ex052", y, {"dx": x, "dscale": scale}, dout)
print("x", tuple(x.shape), "scale", tuple(scale.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 052: derive manually; do not use autograd in this cell.
# Define `ex052_out_shape` — a Python tuple containing the forward output shape.
# Define `ex052_dx` — the gradient with respect to `x`.
# Define `ex052_dscale` — the unbroadcast gradient with respect to `scale`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex052_out_shape", "ex052", "out_shape")
_check_tensor("ex052_dx", "ex052", "dx")
_check_tensor("ex052_dscale", "ex052", "dscale")


### Exercise 053 — Multiply by a column scale

**Purpose:** Learn that one scale per row receives contributions from every column in that row.

**Inputs:** `x` has shape `(3, 4)`, `scale` has shape `(3, 1)`, and `dout` has shape `(3, 4)`.

**Forward operation:** `y = x * scale`.

**Derive/do:** Derive `dx` and `dscale`.

**Ingredients:** Sum scale contributions over the four-column axis with `keepdim=True`.

**Required outputs:**

- `ex053_out_shape`: a Python tuple containing the forward output shape.
- `ex053_dx`: the gradient with respect to `x`.
- `ex053_dscale`: the `(3, 1)` gradient with respect to `scale`.

**Next concept:** Two-way broadcasted addition.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.arange(1, 13, dtype=DTYPE).reshape(3, 4).requires_grad_()
scale = torch.tensor([[0.5], [-1.0], [2.0]], dtype=DTYPE, requires_grad=True)
y = x * scale
dout = torch.linspace(-0.5, 1.7, steps=12, dtype=DTYPE).reshape(3, 4)
_capture("ex053", y, {"dx": x, "dscale": scale}, dout)
print("x", tuple(x.shape), "scale", tuple(scale.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 053: derive manually; do not use autograd in this cell.
# Define `ex053_out_shape` — a Python tuple containing the forward output shape.
# Define `ex053_dx` — the gradient with respect to `x`.
# Define `ex053_dscale` — the `(3, 1)` gradient with respect to `scale`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex053_out_shape", "ex053", "out_shape")
_check_tensor("ex053_dx", "ex053", "dx")
_check_tensor("ex053_dscale", "ex053", "dscale")


### Exercise 054 — Two-way broadcasted addition

**Purpose:** Learn how tensors shaped `(3, 1)` and `(1, 4)` are reused to make a `(3, 4)` sum, and how their gradients shrink back.

**Inputs:** `left` has shape `(3, 1)`, `right` has shape `(1, 4)`, and `y` has shape `(3, 4)`.

**Forward operation:** `y = left + right`.

**Derive/do:** Derive gradients with each operand's original shape.

**Ingredients:** `left` repeats across columns; `right` repeats across rows.

**Required outputs:**

- `ex054_out_shape`: a Python tuple containing the forward output shape.
- `ex054_dleft`: the `(3, 1)` gradient with respect to `left`.
- `ex054_dright`: the `(1, 4)` gradient with respect to `right`.

**Next concept:** Two-way broadcasted multiplication.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
left = torch.tensor([[1.0], [2.0], [3.0]], dtype=DTYPE, requires_grad=True)
right = torch.tensor([[0.5, -1.0, 2.0, 4.0]], dtype=DTYPE, requires_grad=True)
y = left + right
dout = torch.linspace(-1.0, 1.2, steps=12, dtype=DTYPE).reshape(3, 4)
_capture("ex054", y, {"dleft": left, "dright": right}, dout)
print("left", tuple(left.shape), "right", tuple(right.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 054: derive manually; do not use autograd in this cell.
# Define `ex054_out_shape` — a Python tuple containing the forward output shape.
# Define `ex054_dleft` — the `(3, 1)` gradient with respect to `left`.
# Define `ex054_dright` — the `(1, 4)` gradient with respect to `right`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex054_out_shape", "ex054", "out_shape")
_check_tensor("ex054_dleft", "ex054", "dleft")
_check_tensor("ex054_dright", "ex054", "dright")


### Exercise 055 — Two-way broadcasted multiplication

**Purpose:** Learn how two differently shaped tensors are reused in multiplication and how each gradient returns to its original shape.

**Inputs:** `left` has shape `(3, 1)`, `right` has shape `(1, 4)`, and `dout` has shape `(3, 4)`.

**Forward operation:** `y = left * right`.

**Derive/do:** Derive both gradients.

**Ingredients:** Use the other factor locally, then reduce the axis on which each input was reused.

**Required outputs:**

- `ex055_out_shape`: a Python tuple containing the forward output shape.
- `ex055_dleft`: the `(3, 1)` gradient with respect to `left`.
- `ex055_dright`: the `(1, 4)` gradient with respect to `right`.

**Next concept:** Broadcast a channel bias in rank 3.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
left = torch.tensor([[1.0], [-2.0], [0.5]], dtype=DTYPE, requires_grad=True)
right = torch.tensor([[0.5, -1.0, 2.0, 4.0]], dtype=DTYPE, requires_grad=True)
y = left * right
dout = torch.linspace(-1.0, 1.2, steps=12, dtype=DTYPE).reshape(3, 4)
_capture("ex055", y, {"dleft": left, "dright": right}, dout)
print("left", tuple(left.shape), "right", tuple(right.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 055: derive manually; do not use autograd in this cell.
# Define `ex055_out_shape` — a Python tuple containing the forward output shape.
# Define `ex055_dleft` — the `(3, 1)` gradient with respect to `left`.
# Define `ex055_dright` — the `(1, 4)` gradient with respect to `right`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex055_out_shape", "ex055", "out_shape")
_check_tensor("ex055_dleft", "ex055", "dleft")
_check_tensor("ex055_dright", "ex055", "dright")


### Exercise 056 — Broadcast a channel bias in rank 3

**Purpose:** Learn to track batch, channel, and position axes when one channel bias is reused in a rank-3 tensor.

**Inputs:** `x` has shape `(2, 3, 4)` and `bias` has shape `(1, 3, 1)`.

**Forward operation:** `y = x + bias`.

**Derive/do:** Derive `dx` and `dbias`.

**Ingredients:** The bias is reused over batch and position, but not over channel.

**Required outputs:**

- `ex056_out_shape`: a Python tuple containing the forward output shape.
- `ex056_dx`: the gradient with respect to `x`.
- `ex056_dbias`: the `(1, 3, 1)` channel-bias gradient.

**Next concept:** Broadcast a feature scale in rank 3.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.arange(24, dtype=DTYPE).reshape(2, 3, 4).requires_grad_()
bias = torch.tensor([[[0.5], [-1.0], [2.0]]], dtype=DTYPE, requires_grad=True)
y = x + bias
dout = torch.linspace(-1.0, 1.3, steps=24, dtype=DTYPE).reshape(2, 3, 4)
_capture("ex056", y, {"dx": x, "dbias": bias}, dout)
print("x", tuple(x.shape), "bias", tuple(bias.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 056: derive manually; do not use autograd in this cell.
# Define `ex056_out_shape` — a Python tuple containing the forward output shape.
# Define `ex056_dx` — the gradient with respect to `x`.
# Define `ex056_dbias` — the `(1, 3, 1)` channel-bias gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex056_out_shape", "ex056", "out_shape")
_check_tensor("ex056_dx", "ex056", "dx")
_check_tensor("ex056_dbias", "ex056", "dbias")


### Exercise 057 — Broadcast a feature scale in rank 3

**Purpose:** Learn that a feature scale on the last axis is reused at every batch and position location.

**Inputs:** `x` has shape `(2, 3, 4)` and `scale` has shape `(4,)`.

**Forward operation:** `y = x * scale`.

**Derive/do:** Derive `dx` and `dscale`.

**Ingredients:** The feature scale is reused over both leading axes.

**Required outputs:**

- `ex057_out_shape`: a Python tuple containing the forward output shape.
- `ex057_dx`: the gradient with respect to `x`.
- `ex057_dscale`: the length-4 feature-scale gradient.

**Next concept:** Broadcasted bias followed by a total.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.linspace(-2.0, 3.0, steps=24, dtype=DTYPE).reshape(2, 3, 4).requires_grad_()
scale = torch.tensor([0.5, -1.0, 2.0, 3.0], dtype=DTYPE, requires_grad=True)
y = x * scale
dout = torch.linspace(1.0, -1.3, steps=24, dtype=DTYPE).reshape(2, 3, 4)
_capture("ex057", y, {"dx": x, "dscale": scale}, dout)
print("x", tuple(x.shape), "scale", tuple(scale.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 057: derive manually; do not use autograd in this cell.
# Define `ex057_out_shape` — a Python tuple containing the forward output shape.
# Define `ex057_dx` — the gradient with respect to `x`.
# Define `ex057_dscale` — the length-4 feature-scale gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex057_out_shape", "ex057", "out_shape")
_check_tensor("ex057_dx", "ex057", "dx")
_check_tensor("ex057_dscale", "ex057", "dscale")


### Exercise 058 — Broadcasted bias followed by a total

**Purpose:** Learn why summing all biased outputs makes the bias gradient count every example that used it.

**Inputs:** `x` has shape `(3, 4)` and `bias` has shape `(4,)`.

**Forward operation:** `pre = x + bias` and `y = pre.sum()`.

**Derive/do:** Derive `dpre`, `dx`, and `dbias`.

**Ingredients:** Reverse the scalar reduction before unbroadcasting the bias path.

**Required outputs:**

- `ex058_out_shape`: a Python tuple containing the forward output shape.
- `ex058_dpre`: the gradient entering the broadcasted addition.
- `ex058_dx`: the gradient with respect to `x`.
- `ex058_dbias`: the accumulated bias gradient.

**Next concept:** Broadcasted scale followed by a mean.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.arange(12, dtype=DTYPE).reshape(3, 4).requires_grad_()
bias = torch.tensor([0.5, -1.0, 2.0, 3.0], dtype=DTYPE, requires_grad=True)
pre = x + bias
y = pre.sum()
dout = torch.tensor(1.75, dtype=DTYPE)
_capture("ex058", y, {"dpre": pre, "dx": x, "dbias": bias}, dout)
print("x", tuple(x.shape), "bias", tuple(bias.shape), "pre", tuple(pre.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 058: derive manually; do not use autograd in this cell.
# Define `ex058_out_shape` — a Python tuple containing the forward output shape.
# Define `ex058_dpre` — the gradient entering the broadcasted addition.
# Define `ex058_dx` — the gradient with respect to `x`.
# Define `ex058_dbias` — the accumulated bias gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex058_out_shape", "ex058", "out_shape")
_check_tensor("ex058_dpre", "ex058", "dpre")
_check_tensor("ex058_dx", "ex058", "dx")
_check_tensor("ex058_dbias", "ex058", "dbias")


### Exercise 059 — Broadcasted scale followed by a mean

**Purpose:** Learn how a final mean changes gradients before they return through a reused scale.

**Inputs:** `x` has shape `(3, 4)` and `scale` has shape `(4,)`.

**Forward operation:** `pre = x * scale` and `y = pre.mean()`.

**Derive/do:** Derive `dpre`, `dx`, and `dscale`.

**Ingredients:** The mean averages all 12 output entries before product backward.

**Required outputs:**

- `ex059_out_shape`: a Python tuple containing the forward output shape.
- `ex059_dpre`: the gradient entering the broadcasted multiplication.
- `ex059_dx`: the gradient with respect to `x`.
- `ex059_dscale`: the accumulated scale gradient.

**Next concept:** Center columns by their means.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.arange(1, 13, dtype=DTYPE).reshape(3, 4).requires_grad_()
scale = torch.tensor([0.5, -1.0, 2.0, 3.0], dtype=DTYPE, requires_grad=True)
pre = x * scale
y = pre.mean()
dout = torch.tensor(-2.0, dtype=DTYPE)
_capture("ex059", y, {"dpre": pre, "dx": x, "dscale": scale}, dout)
print("x", tuple(x.shape), "scale", tuple(scale.shape), "pre", tuple(pre.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 059: derive manually; do not use autograd in this cell.
# Define `ex059_out_shape` — a Python tuple containing the forward output shape.
# Define `ex059_dpre` — the gradient entering the broadcasted multiplication.
# Define `ex059_dx` — the gradient with respect to `x`.
# Define `ex059_dscale` — the accumulated scale gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex059_out_shape", "ex059", "out_shape")
_check_tensor("ex059_dpre", "ex059", "dpre")
_check_tensor("ex059_dx", "ex059", "dx")
_check_tensor("ex059_dscale", "ex059", "dscale")


### Exercise 060 — Center columns by their means

**Purpose:** Learn that `x` affects centered values both directly and indirectly through the mean computed from `x`.

**Inputs:** `x` has shape `(4, 3)`; `mean` has shape `(1, 3)`; `y` and `dout` have shape `(4, 3)`.

**Forward operation:** `mean = x.mean(dim=0, keepdim=True)` and `y = x - mean`.

**Derive/do:** Derive `dmean` and the total `dx` from both paths.

**Ingredients:** Backward through subtraction gives a direct `x` path and a mean path; add contributions at `x`.

**Required outputs:**

- `ex060_out_shape`: a Python tuple containing the forward output shape.
- `ex060_dmean`: the `(1, 3)` gradient with respect to the column means.
- `ex060_dx`: the total gradient with respect to `x`.

**Next concept:** Divide by a row scale.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.linspace(-2.0, 3.5, steps=12, dtype=DTYPE).reshape(4, 3).requires_grad_()
mean = x.mean(dim=0, keepdim=True)
y = x - mean
dout = torch.linspace(1.0, -1.2, steps=12, dtype=DTYPE).reshape(4, 3)
_capture("ex060", y, {"dmean": mean, "dx": x}, dout)
print("x", tuple(x.shape), "mean", tuple(mean.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 060: derive manually; do not use autograd in this cell.
# Define `ex060_out_shape` — a Python tuple containing the forward output shape.
# Define `ex060_dmean` — the `(1, 3)` gradient with respect to the column means.
# Define `ex060_dx` — the total gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex060_out_shape", "ex060", "out_shape")
_check_tensor("ex060_dmean", "ex060", "dmean")
_check_tensor("ex060_dx", "ex060", "dx")


### Exercise 061 — Divide by a row scale

**Purpose:** Learn how a denominator reused across a row collects gradient contributions from every column.

**Inputs:** `x` has shape `(3, 4)`, positive `scale` has shape `(3, 1)`, and `dout` has shape `(3, 4)`.

**Forward operation:** `y = x / scale`.

**Derive/do:** Derive `dx` and `dscale`.

**Ingredients:** Use the division local derivatives, then sum denominator contributions across columns.

**Required outputs:**

- `ex061_out_shape`: a Python tuple containing the forward output shape.
- `ex061_dx`: the gradient with respect to `x`.
- `ex061_dscale`: the `(3, 1)` denominator gradient.

**Next concept:** Normalize rows by their sums.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.arange(1, 13, dtype=DTYPE).reshape(3, 4).requires_grad_()
scale = torch.tensor([[1.0], [2.0], [4.0]], dtype=DTYPE, requires_grad=True)
y = x / scale
dout = torch.linspace(-1.0, 1.2, steps=12, dtype=DTYPE).reshape(3, 4)
_capture("ex061", y, {"dx": x, "dscale": scale}, dout)
print("x", tuple(x.shape), "scale", tuple(scale.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 061: derive manually; do not use autograd in this cell.
# Define `ex061_out_shape` — a Python tuple containing the forward output shape.
# Define `ex061_dx` — the gradient with respect to `x`.
# Define `ex061_dscale` — the `(3, 1)` denominator gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex061_out_shape", "ex061", "out_shape")
_check_tensor("ex061_dx", "ex061", "dx")
_check_tensor("ex061_dscale", "ex061", "dscale")


### Exercise 062 — Normalize rows by their sums

**Purpose:** Learn that each entry affects row normalization both as a numerator and as part of the shared row sum.

**Inputs:** Positive `x` has shape `(3, 4)`; `row_sum` has shape `(3, 1)`.

**Forward operation:** `row_sum = x.sum(dim=1, keepdim=True)` and `y = x / row_sum`.

**Derive/do:** Derive `drow_sum` and total `dx`.

**Ingredients:** Each `x` entry has a direct numerator path and an indirect path through its row total.

**Required outputs:**

- `ex062_out_shape`: a Python tuple containing the forward output shape.
- `ex062_drow_sum`: the `(3, 1)` gradient of the shared row denominators.
- `ex062_dx`: the total gradient with respect to `x`.

**Next concept:** Broadcasted where with a scalar fallback.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([[1.0, 2.0, 3.0, 4.0], [2.0, 1.0, 4.0, 3.0], [3.0, 5.0, 2.0, 1.0]], dtype=DTYPE, requires_grad=True)
row_sum = x.sum(dim=1, keepdim=True)
y = x / row_sum
dout = torch.linspace(-1.0, 1.2, steps=12, dtype=DTYPE).reshape(3, 4)
_capture("ex062", y, {"drow_sum": row_sum, "dx": x}, dout)
print("x", tuple(x.shape), "row_sum", tuple(row_sum.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 062: derive manually; do not use autograd in this cell.
# Define `ex062_out_shape` — a Python tuple containing the forward output shape.
# Define `ex062_drow_sum` — the `(3, 1)` gradient of the shared row denominators.
# Define `ex062_dx` — the total gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex062_out_shape", "ex062", "out_shape")
_check_tensor("ex062_drow_sum", "ex062", "drow_sum")
_check_tensor("ex062_dx", "ex062", "dx")


### Exercise 063 — Broadcasted where with a scalar fallback

**Purpose:** Learn that `where` sends each output gradient only through the branch chosen by its Boolean mask.

**Inputs:** `x` has shape `(2, 3)`, `fallback` is scalar, and `mask` is Boolean `(2, 3)`.

**Forward operation:** `y = torch.where(mask, x, fallback)`.

**Derive/do:** Derive `dx` and scalar `dfallback`.

**Ingredients:** The mask chooses one active source at each output; the scalar is reused at every false position.

**Required outputs:**

- `ex063_out_shape`: a Python tuple containing the forward output shape.
- `ex063_dx`: the masked gradient with respect to `x`.
- `ex063_dfallback`: the accumulated scalar fallback gradient.

**Next concept:** A shared bias on two branches.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.arange(1, 7, dtype=DTYPE).reshape(2, 3).requires_grad_()
fallback = torch.tensor(-2.0, dtype=DTYPE, requires_grad=True)
mask = torch.tensor([[True, False, True], [False, False, True]])
y = torch.where(mask, x, fallback)
dout = torch.tensor([[1.0, -2.0, 0.5], [3.0, 1.5, -1.0]], dtype=DTYPE)
_capture("ex063", y, {"dx": x, "dfallback": fallback}, dout)
print("x", tuple(x.shape), "fallback", tuple(fallback.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 063: derive manually; do not use autograd in this cell.
# Define `ex063_out_shape` — a Python tuple containing the forward output shape.
# Define `ex063_dx` — the masked gradient with respect to `x`.
# Define `ex063_dfallback` — the accumulated scalar fallback gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex063_out_shape", "ex063", "out_shape")
_check_tensor("ex063_dx", "ex063", "dx")
_check_tensor("ex063_dfallback", "ex063", "dfallback")


### Exercise 064 — A shared bias on two branches

**Purpose:** Learn that when one bias is used in two branches, its final gradient is the sum of both branch contributions.

**Inputs:** `x` has shape `(2, 3)` and `bias` has shape `(3,)`.

**Forward operation:** `left = x + bias`, `right = 2 * bias`, and `y = left + right`.

**Derive/do:** Derive total `dbias` and `dx`.

**Ingredients:** The bias receives a broadcasted path through `left` and a separate path through `right`; add them.

**Required outputs:**

- `ex064_out_shape`: a Python tuple containing the forward output shape.
- `ex064_dx`: the gradient with respect to `x`.
- `ex064_dbias`: the total gradient from both bias uses.

**Next concept:** Backward through expand.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.arange(6, dtype=DTYPE).reshape(2, 3).requires_grad_()
bias = torch.tensor([0.5, -1.0, 2.0], dtype=DTYPE, requires_grad=True)
left = x + bias
right = 2.0 * bias
y = left + right
dout = torch.tensor([[1.0, -2.0, 0.5], [3.0, 1.5, -1.0]], dtype=DTYPE)
_capture("ex064", y, {"dx": x, "dbias": bias}, dout)
print("x", tuple(x.shape), "bias", tuple(bias.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 064: derive manually; do not use autograd in this cell.
# Define `ex064_out_shape` — a Python tuple containing the forward output shape.
# Define `ex064_dx` — the gradient with respect to `x`.
# Define `ex064_dbias` — the total gradient from both bias uses.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex064_out_shape", "ex064", "out_shape")
_check_tensor("ex064_dx", "ex064", "dx")
_check_tensor("ex064_dbias", "ex064", "dbias")


### Exercise 065 — Backward through expand

**Purpose:** Learn that expanding a tensor reuses its values, so backward adds gradients back into the smaller original shape.

**Inputs:** `bias` has shape `(1, 3)` and `expanded` has shape `(2, 3)`.

**Forward operation:** `expanded = bias.expand(2, 3)` and `y = expanded * x`.

**Derive/do:** Derive `dexpanded`, `dbias`, and `dx`.

**Ingredients:** Reverse multiplication, then sum expanded contributions back to the original size-one row axis.

**Required outputs:**

- `ex065_out_shape`: a Python tuple containing the forward output shape.
- `ex065_dexpanded`: the gradient at the expanded view.
- `ex065_dbias`: the unexpanded gradient with respect to `bias`.
- `ex065_dx`: the gradient with respect to `x`.

**Next concept:** Broadcast, square, then backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
bias = torch.tensor([[0.5, -1.0, 2.0]], dtype=DTYPE, requires_grad=True)
x = torch.arange(1, 7, dtype=DTYPE).reshape(2, 3).requires_grad_()
expanded = bias.expand(2, 3)
y = expanded * x
dout = torch.tensor([[1.0, -2.0, 0.5], [3.0, 1.5, -1.0]], dtype=DTYPE)
_capture("ex065", y, {"dexpanded": expanded, "dbias": bias, "dx": x}, dout)
print("bias", tuple(bias.shape), "expanded", tuple(expanded.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 065: derive manually; do not use autograd in this cell.
# Define `ex065_out_shape` — a Python tuple containing the forward output shape.
# Define `ex065_dexpanded` — the gradient at the expanded view.
# Define `ex065_dbias` — the unexpanded gradient with respect to `bias`.
# Define `ex065_dx` — the gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex065_out_shape", "ex065", "out_shape")
_check_tensor("ex065_dexpanded", "ex065", "dexpanded")
_check_tensor("ex065_dbias", "ex065", "dbias")
_check_tensor("ex065_dx", "ex065", "dx")


### Exercise 066 — Broadcast, square, then backward

**Purpose:** Learn to reverse a square first and then reverse the broadcasted addition that produced its input.

**Inputs:** `x` has shape `(3, 4)`, `bias` has shape `(4,)`, and `pre` has shape `(3, 4)`.

**Forward operation:** `pre = x + bias` and `y = pre**2`.

**Derive/do:** Derive `dpre`, `dx`, and `dbias`.

**Ingredients:** Reverse the square first; then split addition paths and unbroadcast the bias path.

**Required outputs:**

- `ex066_out_shape`: a Python tuple containing the forward output shape.
- `ex066_dpre`: the gradient after reversing the square.
- `ex066_dx`: the gradient with respect to `x`.
- `ex066_dbias`: the unbroadcast bias gradient.

**Next concept:** Vector dot product.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.linspace(-2.0, 3.5, steps=12, dtype=DTYPE).reshape(3, 4).requires_grad_()
bias = torch.tensor([0.5, -1.0, 2.0, 3.0], dtype=DTYPE, requires_grad=True)
pre = x + bias
y = pre**2
dout = torch.linspace(1.0, -1.2, steps=12, dtype=DTYPE).reshape(3, 4)
_capture("ex066", y, {"dpre": pre, "dx": x, "dbias": bias}, dout)
print("x", tuple(x.shape), "bias", tuple(bias.shape), "pre", tuple(pre.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 066: derive manually; do not use autograd in this cell.
# Define `ex066_out_shape` — a Python tuple containing the forward output shape.
# Define `ex066_dpre` — the gradient after reversing the square.
# Define `ex066_dx` — the gradient with respect to `x`.
# Define `ex066_dbias` — the unbroadcast bias gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex066_out_shape", "ex066", "out_shape")
_check_tensor("ex066_dpre", "ex066", "dpre")
_check_tensor("ex066_dx", "ex066", "dx")
_check_tensor("ex066_dbias", "ex066", "dbias")


## 5. Matrix multiplication and affine layers

Matrix multiplication contracts an inner axis. Its backward pass must restore each operand's original shape.

For a batch matrix `X`, weights `W`, and output `Y`, the forward relationship is

$$
Y = XW
$$

Here, `X` has shape `(B, I)`, `W` has shape `(I, O)`, and `Y` has shape `(B, O)`. In backward, reason with transposes and verify that every matrix product's inner dimensions match. A bias added to `Y` is broadcast across the `B` examples, so its gradient accumulates over the batch axis.


### Exercise 067 — Vector dot product

**Purpose:** Learn how gradients flow through a dot product of two vectors.

**Inputs:** `x` and `w` have shape `(4,)`; `y = x @ w` is scalar.

**Forward operation:** `y = x @ w`.

**Derive/do:** Derive `dx` and `dw`.

**Ingredients:** Expand the dot product as a sum of elementwise products.

**Required outputs:**

- `ex067_out_shape`: a Python tuple containing the forward output shape.
- `ex067_dx`: the gradient with respect to `x`.
- `ex067_dw`: the gradient with respect to `w`.

**Next concept:** Matrix-vector product.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([1.0, -2.0, 3.0, 0.5], dtype=DTYPE, requires_grad=True)
w = torch.tensor([0.5, 4.0, -1.0, 2.0], dtype=DTYPE, requires_grad=True)
y = x @ w
dout = torch.tensor(1.5, dtype=DTYPE)
_capture("ex067", y, {"dx": x, "dw": w}, dout)
print("x", tuple(x.shape), "w", tuple(w.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 067: derive manually; do not use autograd in this cell.
# Define `ex067_out_shape` — a Python tuple containing the forward output shape.
# Define `ex067_dx` — the gradient with respect to `x`.
# Define `ex067_dw` — the gradient with respect to `w`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex067_out_shape", "ex067", "out_shape")
_check_tensor("ex067_dx", "ex067", "dx")
_check_tensor("ex067_dw", "ex067", "dw")


### Exercise 068 — Matrix-vector product

**Purpose:** Learn how a matrix-vector product sends gradients back to both the matrix and the vector.

**Inputs:** `A` has shape `(3, 4)`, `x` has shape `(4,)`, and `y` has shape `(3,)`.

**Forward operation:** `y = A @ x`.

**Derive/do:** Derive `dA` and `dx`.

**Ingredients:** Write one output as a dot product, account for all three outputs, and use shape checks.

**Required outputs:**

- `ex068_out_shape`: a Python tuple containing the forward output shape.
- `ex068_dA`: the `(3, 4)` gradient with respect to `A`.
- `ex068_dx`: the length-4 gradient with respect to `x`.

**Next concept:** Row vector times weight matrix.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
A = torch.arange(1, 13, dtype=DTYPE).reshape(3, 4).requires_grad_()
x = torch.tensor([0.5, -1.0, 2.0, 3.0], dtype=DTYPE, requires_grad=True)
y = A @ x
dout = torch.tensor([1.0, -2.0, 0.5], dtype=DTYPE)
_capture("ex068", y, {"dA": A, "dx": x}, dout)
print("A", tuple(A.shape), "x", tuple(x.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 068: derive manually; do not use autograd in this cell.
# Define `ex068_out_shape` — a Python tuple containing the forward output shape.
# Define `ex068_dA` — the `(3, 4)` gradient with respect to `A`.
# Define `ex068_dx` — the length-4 gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex068_out_shape", "ex068", "out_shape")
_check_tensor("ex068_dA", "ex068", "dA")
_check_tensor("ex068_dx", "ex068", "dx")


### Exercise 069 — Row vector times weight matrix

**Purpose:** Learn how one input vector produces several outputs through a weight matrix and receives gradients from all of them.

**Inputs:** `x` has shape `(3,)`, `W` has shape `(3, 2)`, and `y` has shape `(2,)`.

**Forward operation:** `y = x @ W`.

**Derive/do:** Derive `dx` and `dW`.

**Ingredients:** Each output is a dot product with one weight column; use transposes to satisfy shapes.

**Required outputs:**

- `ex069_out_shape`: a Python tuple containing the forward output shape.
- `ex069_dx`: the length-3 gradient with respect to `x`.
- `ex069_dW`: the `(3, 2)` gradient with respect to `W`.

**Next concept:** Matrix-matrix product.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([1.0, -2.0, 0.5], dtype=DTYPE, requires_grad=True)
W = torch.tensor([[0.5, 1.0], [-1.0, 2.0], [3.0, -0.5]], dtype=DTYPE, requires_grad=True)
y = x @ W
dout = torch.tensor([2.0, -1.5], dtype=DTYPE)
_capture("ex069", y, {"dx": x, "dW": W}, dout)
print("x", tuple(x.shape), "W", tuple(W.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 069: derive manually; do not use autograd in this cell.
# Define `ex069_out_shape` — a Python tuple containing the forward output shape.
# Define `ex069_dx` — the length-3 gradient with respect to `x`.
# Define `ex069_dW` — the `(3, 2)` gradient with respect to `W`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex069_out_shape", "ex069", "out_shape")
_check_tensor("ex069_dx", "ex069", "dx")
_check_tensor("ex069_dW", "ex069", "dW")


### Exercise 070 — Matrix-matrix product

**Purpose:** Learn how a matrix-matrix product sends gradients back to both input matrices.

**Inputs:** `A` has shape `(2, 3)`, `B` has shape `(3, 4)`, and `dout` has shape `(2, 4)`.

**Forward operation:** `Y = A @ B`.

**Derive/do:** Derive `dA` and `dB`.

**Ingredients:** Use the upstream matrix and the appropriate transpose of the other operand; audit all inner dimensions.

**Required outputs:**

- `ex070_out_shape`: a Python tuple containing the forward output shape.
- `ex070_dA`: the `(2, 3)` gradient with respect to `A`.
- `ex070_dB`: the `(3, 4)` gradient with respect to `B`.

**Next concept:** A batch through one weight matrix.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
A = torch.arange(1, 7, dtype=DTYPE).reshape(2, 3).requires_grad_()
B = torch.linspace(-1.0, 2.3, steps=12, dtype=DTYPE).reshape(3, 4).requires_grad_()
Y = A @ B
dout = torch.linspace(0.5, -1.0, steps=8, dtype=DTYPE).reshape(2, 4)
_capture("ex070", Y, {"dA": A, "dB": B}, dout)
print("A", tuple(A.shape), "B", tuple(B.shape), "Y", tuple(Y.shape))


In [ ]:
# Exercise 070: derive manually; do not use autograd in this cell.
# Define `ex070_out_shape` — a Python tuple containing the forward output shape.
# Define `ex070_dA` — the `(2, 3)` gradient with respect to `A`.
# Define `ex070_dB` — the `(3, 4)` gradient with respect to `B`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex070_out_shape", "ex070", "out_shape")
_check_tensor("ex070_dA", "ex070", "dA")
_check_tensor("ex070_dB", "ex070", "dB")


### Exercise 071 — A batch through one weight matrix

**Purpose:** Learn that a weight matrix shared by all batch rows collects a contribution from every training example.

**Inputs:** `X` has shape `(4, 3)`, `W` has shape `(3, 2)`, and `Y` has shape `(4, 2)`.

**Forward operation:** `Y = X @ W`.

**Derive/do:** Derive `dX` and `dW`.

**Ingredients:** The same `W` is reused for all four rows; matrix multiplication performs the needed accumulation.

**Required outputs:**

- `ex071_out_shape`: a Python tuple containing the forward output shape.
- `ex071_dX`: the `(4, 3)` gradient with respect to `X`.
- `ex071_dW`: the `(3, 2)` gradient accumulated over the batch.

**Next concept:** Output bias backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
X = torch.linspace(-2.0, 3.5, steps=12, dtype=DTYPE).reshape(4, 3).requires_grad_()
W = torch.tensor([[0.5, 1.0], [-1.0, 2.0], [3.0, -0.5]], dtype=DTYPE, requires_grad=True)
Y = X @ W
dout = torch.linspace(1.0, -0.75, steps=8, dtype=DTYPE).reshape(4, 2)
_capture("ex071", Y, {"dX": X, "dW": W}, dout)
print("X", tuple(X.shape), "W", tuple(W.shape), "Y", tuple(Y.shape))


In [ ]:
# Exercise 071: derive manually; do not use autograd in this cell.
# Define `ex071_out_shape` — a Python tuple containing the forward output shape.
# Define `ex071_dX` — the `(4, 3)` gradient with respect to `X`.
# Define `ex071_dW` — the `(3, 2)` gradient accumulated over the batch.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex071_out_shape", "ex071", "out_shape")
_check_tensor("ex071_dX", "ex071", "dX")
_check_tensor("ex071_dW", "ex071", "dW")


### Exercise 072 — Output bias backward

**Purpose:** Learn that an output bias shared by all examples adds its gradient contributions across the batch rows.

**Inputs:** `Z` has shape `(4, 2)`, `bias` has shape `(2,)`, and `dout` has shape `(4, 2)`.

**Forward operation:** `Y = Z + bias`.

**Derive/do:** Derive `dZ` and `dbias`.

**Ingredients:** The bias is reused for four examples; sum only the example axis.

**Required outputs:**

- `ex072_out_shape`: a Python tuple containing the forward output shape.
- `ex072_dZ`: the gradient with respect to `Z`.
- `ex072_dbias`: the length-2 bias gradient.

**Next concept:** Complete affine layer.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
Z = torch.linspace(-2.0, 1.5, steps=8, dtype=DTYPE).reshape(4, 2).requires_grad_()
bias = torch.tensor([0.5, -1.0], dtype=DTYPE, requires_grad=True)
Y = Z + bias
dout = torch.linspace(1.0, -0.75, steps=8, dtype=DTYPE).reshape(4, 2)
_capture("ex072", Y, {"dZ": Z, "dbias": bias}, dout)
print("Z", tuple(Z.shape), "bias", tuple(bias.shape), "Y", tuple(Y.shape))


In [ ]:
# Exercise 072: derive manually; do not use autograd in this cell.
# Define `ex072_out_shape` — a Python tuple containing the forward output shape.
# Define `ex072_dZ` — the gradient with respect to `Z`.
# Define `ex072_dbias` — the length-2 bias gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex072_out_shape", "ex072", "out_shape")
_check_tensor("ex072_dZ", "ex072", "dZ")
_check_tensor("ex072_dbias", "ex072", "dbias")


### Exercise 073 — Complete affine layer

**Purpose:** Learn to backpropagate through the complete operation `X @ W + bias`.

**Inputs:** `X` is `(4, 3)`, `W` is `(3, 2)`, `bias` is `(2,)`, and `Y` is `(4, 2)`.

**Forward operation:** `Y = X @ W + bias`.

**Derive/do:** Derive `dX`, `dW`, and `dbias`.

**Ingredients:** Start from `dout`; handle the addition and matrix product while checking each gradient shape.

**Required outputs:**

- `ex073_out_shape`: a Python tuple containing the forward output shape.
- `ex073_dX`: the input gradient.
- `ex073_dW`: the weight gradient.
- `ex073_dbias`: the bias gradient.

**Next concept:** Affine layer followed by a total.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
X = torch.linspace(-2.0, 3.5, steps=12, dtype=DTYPE).reshape(4, 3).requires_grad_()
W = torch.tensor([[0.5, 1.0], [-1.0, 2.0], [3.0, -0.5]], dtype=DTYPE, requires_grad=True)
bias = torch.tensor([0.25, -0.75], dtype=DTYPE, requires_grad=True)
Y = X @ W + bias
dout = torch.linspace(1.0, -0.75, steps=8, dtype=DTYPE).reshape(4, 2)
_capture("ex073", Y, {"dX": X, "dW": W, "dbias": bias}, dout)
print("X", tuple(X.shape), "W", tuple(W.shape), "bias", tuple(bias.shape), "Y", tuple(Y.shape))


In [19]:
# Exercise 073: derive manually; do not use autograd in this cell.
# Define `ex073_out_shape` — a Python tuple containing the forward output shape.
# Define `ex073_dX` — the input gradient.
# Define `ex073_dW` — the weight gradient.
# Define `ex073_dbias` — the bias gradient.
# Write your tensor operations below these comments, then run the test cell.


In [20]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex073_out_shape", "ex073", "out_shape")
_check_tensor("ex073_dX", "ex073", "dX")
_check_tensor("ex073_dW", "ex073", "dW")
_check_tensor("ex073_dbias", "ex073", "dbias")


AssertionError: Define `ex073_out_shape` in the answer cell first.

### Exercise 074 — Affine layer followed by a total

**Purpose:** Learn how a scalar sum creates the first gradient tensor for an affine layer's outputs.

**Inputs:** `Y = X @ W + bias` has shape `(3, 2)`; `loss = Y.sum()` is scalar.

**Forward operation:** Compute `Y`, then `loss = Y.sum()`.

**Derive/do:** Derive `dY`, `dX`, `dW`, and `dbias`.

**Ingredients:** Reverse the sum to create `dY`, then apply affine backward.

**Required outputs:**

- `ex074_out_shape`: a Python tuple containing the forward output shape.
- `ex074_dY`: the gradient after reversing the total.
- `ex074_dX`: the input gradient.
- `ex074_dW`: the weight gradient.
- `ex074_dbias`: the bias gradient.

**Next concept:** Affine layer followed by a mean.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
X = torch.linspace(-1.0, 2.0, steps=9, dtype=DTYPE).reshape(3, 3).requires_grad_()
W = torch.tensor([[0.5, 1.0], [-1.0, 2.0], [3.0, -0.5]], dtype=DTYPE, requires_grad=True)
bias = torch.tensor([0.25, -0.75], dtype=DTYPE, requires_grad=True)
Y = X @ W + bias
loss_local = Y.sum()
dout = torch.tensor(1.5, dtype=DTYPE)
_capture("ex074", loss_local, {"dY": Y, "dX": X, "dW": W, "dbias": bias}, dout)
print("X", tuple(X.shape), "Y", tuple(Y.shape), "loss", tuple(loss_local.shape))


In [ ]:
# Exercise 074: derive manually; do not use autograd in this cell.
# Define `ex074_out_shape` — a Python tuple containing the forward output shape.
# Define `ex074_dY` — the gradient after reversing the total.
# Define `ex074_dX` — the input gradient.
# Define `ex074_dW` — the weight gradient.
# Define `ex074_dbias` — the bias gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex074_out_shape", "ex074", "out_shape")
_check_tensor("ex074_dY", "ex074", "dY")
_check_tensor("ex074_dX", "ex074", "dX")
_check_tensor("ex074_dW", "ex074", "dW")
_check_tensor("ex074_dbias", "ex074", "dbias")


### Exercise 075 — Affine layer followed by a mean

**Purpose:** Learn how averaging affine outputs changes the gradient scale compared with summing them.

**Inputs:** `Y` has shape `(3, 2)`, so its mean averages six values.

**Forward operation:** `Y = X @ W + bias` and `loss = Y.mean()`.

**Derive/do:** Derive `dY`, `dX`, `dW`, and `dbias`.

**Ingredients:** Seed all six `Y` entries with the scalar upstream gradient divided by six.

**Required outputs:**

- `ex075_out_shape`: a Python tuple containing the forward output shape.
- `ex075_dY`: the gradient after reversing the six-value mean.
- `ex075_dX`: the input gradient.
- `ex075_dW`: the weight gradient.
- `ex075_dbias`: the bias gradient.

**Next concept:** Squared affine objective.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
X = torch.linspace(-1.0, 2.0, steps=9, dtype=DTYPE).reshape(3, 3).requires_grad_()
W = torch.tensor([[0.5, 1.0], [-1.0, 2.0], [3.0, -0.5]], dtype=DTYPE, requires_grad=True)
bias = torch.tensor([0.25, -0.75], dtype=DTYPE, requires_grad=True)
Y = X @ W + bias
loss_local = Y.mean()
dout = torch.tensor(-2.0, dtype=DTYPE)
_capture("ex075", loss_local, {"dY": Y, "dX": X, "dW": W, "dbias": bias}, dout)
print("X", tuple(X.shape), "Y", tuple(Y.shape), "loss", tuple(loss_local.shape))


In [ ]:
# Exercise 075: derive manually; do not use autograd in this cell.
# Define `ex075_out_shape` — a Python tuple containing the forward output shape.
# Define `ex075_dY` — the gradient after reversing the six-value mean.
# Define `ex075_dX` — the input gradient.
# Define `ex075_dW` — the weight gradient.
# Define `ex075_dbias` — the bias gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex075_out_shape", "ex075", "out_shape")
_check_tensor("ex075_dY", "ex075", "dY")
_check_tensor("ex075_dX", "ex075", "dX")
_check_tensor("ex075_dW", "ex075", "dW")
_check_tensor("ex075_dbias", "ex075", "dbias")


### Exercise 076 — Squared affine objective

**Purpose:** Learn to reverse a mean of squared affine outputs before computing parameter gradients.

**Inputs:** `Y` has shape `(3, 2)` and `loss = (Y**2).mean()` is scalar.

**Forward operation:** `Y = X @ W + bias`; average the six squared outputs.

**Derive/do:** Derive `dY`, `dX`, `dW`, and `dbias`.

**Ingredients:** Reverse mean, square, addition, and matrix multiplication in that order.

**Required outputs:**

- `ex076_out_shape`: a Python tuple containing the forward output shape.
- `ex076_dY`: the gradient with respect to affine outputs.
- `ex076_dX`: the input gradient.
- `ex076_dW`: the weight gradient.
- `ex076_dbias`: the bias gradient.

**Next concept:** Transpose backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
X = torch.linspace(-1.0, 2.0, steps=9, dtype=DTYPE).reshape(3, 3).requires_grad_()
W = torch.tensor([[0.5, 1.0], [-1.0, 2.0], [3.0, -0.5]], dtype=DTYPE, requires_grad=True)
bias = torch.tensor([0.25, -0.75], dtype=DTYPE, requires_grad=True)
Y = X @ W + bias
loss_local = (Y**2).mean()
dout = torch.tensor(1.0, dtype=DTYPE)
_capture("ex076", loss_local, {"dY": Y, "dX": X, "dW": W, "dbias": bias}, dout)
print("X", tuple(X.shape), "Y", tuple(Y.shape), "loss", tuple(loss_local.shape))


In [ ]:
# Exercise 076: derive manually; do not use autograd in this cell.
# Define `ex076_out_shape` — a Python tuple containing the forward output shape.
# Define `ex076_dY` — the gradient with respect to affine outputs.
# Define `ex076_dX` — the input gradient.
# Define `ex076_dW` — the weight gradient.
# Define `ex076_dbias` — the bias gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex076_out_shape", "ex076", "out_shape")
_check_tensor("ex076_dY", "ex076", "dY")
_check_tensor("ex076_dX", "ex076", "dX")
_check_tensor("ex076_dW", "ex076", "dW")
_check_tensor("ex076_dbias", "ex076", "dbias")


### Exercise 077 — Transpose backward

**Purpose:** Learn that backward through a transpose swaps the axes back to their original order.

**Inputs:** `X` has shape `(2, 3)`, `Y = X.T` has shape `(3, 2)`.

**Forward operation:** `Y = X.T`.

**Derive/do:** Derive `dX` from a `(3, 2)` upstream tensor.

**Ingredients:** A transpose only swaps axes; backward must restore their original order.

**Required outputs:**

- `ex077_out_shape`: a Python tuple containing the forward output shape.
- `ex077_dX`: the `(2, 3)` gradient with respect to `X`.

**Next concept:** Outer product.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
X = torch.arange(6, dtype=DTYPE).reshape(2, 3).requires_grad_()
Y = X.T
dout = torch.tensor([[1.0, -1.0], [2.0, 0.5], [-2.0, 3.0]], dtype=DTYPE)
_capture("ex077", Y, {"dX": X}, dout)
print("X", tuple(X.shape), "Y", tuple(Y.shape), "dout", tuple(dout.shape))


In [ ]:
# Exercise 077: derive manually; do not use autograd in this cell.
# Define `ex077_out_shape` — a Python tuple containing the forward output shape.
# Define `ex077_dX` — the `(2, 3)` gradient with respect to `X`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex077_out_shape", "ex077", "out_shape")
_check_tensor("ex077_dX", "ex077", "dX")


### Exercise 078 — Outer product

**Purpose:** Learn how every pair of entries from two vectors forms a matrix and contributes to both vector gradients.

**Inputs:** `x` has shape `(3,)`, `z` has shape `(4,)`, and `Y` has shape `(3, 4)`.

**Forward operation:** `Y = x[:, None] * z[None, :]`.

**Derive/do:** Derive `dx` and `dz`.

**Ingredients:** This is two-way broadcasted multiplication; reduce the axis introduced for each vector.

**Required outputs:**

- `ex078_out_shape`: a Python tuple containing the forward output shape.
- `ex078_dx`: the length-3 gradient with respect to `x`.
- `ex078_dz`: the length-4 gradient with respect to `z`.

**Next concept:** Batched matrix multiplication.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([1.0, -2.0, 0.5], dtype=DTYPE, requires_grad=True)
z = torch.tensor([0.5, -1.0, 2.0, 3.0], dtype=DTYPE, requires_grad=True)
Y = x[:, None] * z[None, :]
dout = torch.linspace(-1.0, 1.2, steps=12, dtype=DTYPE).reshape(3, 4)
_capture("ex078", Y, {"dx": x, "dz": z}, dout)
print("x", tuple(x.shape), "z", tuple(z.shape), "Y", tuple(Y.shape))


In [ ]:
# Exercise 078: derive manually; do not use autograd in this cell.
# Define `ex078_out_shape` — a Python tuple containing the forward output shape.
# Define `ex078_dx` — the length-3 gradient with respect to `x`.
# Define `ex078_dz` — the length-4 gradient with respect to `z`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex078_out_shape", "ex078", "out_shape")
_check_tensor("ex078_dx", "ex078", "dx")
_check_tensor("ex078_dz", "ex078", "dz")


### Exercise 079 — Batched matrix multiplication

**Purpose:** Learn to backpropagate through several independent matrix products stored along a batch axis.

**Inputs:** `A` has shape `(2, 2, 3)`, `B` has shape `(2, 3, 2)`, and `Y` has shape `(2, 2, 2)`.

**Forward operation:** `Y = torch.bmm(A, B)`.

**Derive/do:** Derive `dA` and `dB`.

**Ingredients:** Apply matrix-product backward within each of the two batch entries; `transpose(1, 2)` swaps matrix axes only.

**Required outputs:**

- `ex079_out_shape`: a Python tuple containing the forward output shape.
- `ex079_dA`: the `(2, 2, 3)` gradient with respect to `A`.
- `ex079_dB`: the `(2, 3, 2)` gradient with respect to `B`.

**Next concept:** Transform the last axis of rank 3.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
A = torch.linspace(-1.0, 2.3, steps=12, dtype=DTYPE).reshape(2, 2, 3).requires_grad_()
B = torch.linspace(0.5, -1.8, steps=12, dtype=DTYPE).reshape(2, 3, 2).requires_grad_()
Y = torch.bmm(A, B)
dout = torch.linspace(1.0, -0.4, steps=8, dtype=DTYPE).reshape(2, 2, 2)
_capture("ex079", Y, {"dA": A, "dB": B}, dout)
print("A", tuple(A.shape), "B", tuple(B.shape), "Y", tuple(Y.shape))


In [ ]:
# Exercise 079: derive manually; do not use autograd in this cell.
# Define `ex079_out_shape` — a Python tuple containing the forward output shape.
# Define `ex079_dA` — the `(2, 2, 3)` gradient with respect to `A`.
# Define `ex079_dB` — the `(2, 3, 2)` gradient with respect to `B`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex079_out_shape", "ex079", "out_shape")
_check_tensor("ex079_dA", "ex079", "dA")
_check_tensor("ex079_dB", "ex079", "dB")


### Exercise 080 — Transform the last axis of rank 3

**Purpose:** Learn that `X @ W` transforms the last axis of a rank-3 tensor at every leading position.

**Inputs:** `X` has shape `(2, 3, 4)`, `W` has shape `(4, 2)`, and `Y` has shape `(2, 3, 2)`.

**Forward operation:** `Y = X @ W`.

**Derive/do:** Derive `dX` and `dW`.

**Ingredients:** Flatten leading positions conceptually into six examples, or use last-two-axis matrix rules and sum shared-weight contributions.

**Required outputs:**

- `ex080_out_shape`: a Python tuple containing the forward output shape.
- `ex080_dX`: the `(2, 3, 4)` gradient with respect to `X`.
- `ex080_dW`: the `(4, 2)` shared-weight gradient.

**Next concept:** Shared weights on two input branches.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
X = torch.linspace(-2.0, 3.5, steps=24, dtype=DTYPE).reshape(2, 3, 4).requires_grad_()
W = torch.linspace(-1.0, 1.5, steps=8, dtype=DTYPE).reshape(4, 2).requires_grad_()
Y = X @ W
dout = torch.linspace(1.0, -0.7, steps=12, dtype=DTYPE).reshape(2, 3, 2)
_capture("ex080", Y, {"dX": X, "dW": W}, dout)
print("X", tuple(X.shape), "W", tuple(W.shape), "Y", tuple(Y.shape))


In [ ]:
# Exercise 080: derive manually; do not use autograd in this cell.
# Define `ex080_out_shape` — a Python tuple containing the forward output shape.
# Define `ex080_dX` — the `(2, 3, 4)` gradient with respect to `X`.
# Define `ex080_dW` — the `(4, 2)` shared-weight gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex080_out_shape", "ex080", "out_shape")
_check_tensor("ex080_dX", "ex080", "dX")
_check_tensor("ex080_dW", "ex080", "dW")


### Exercise 081 — Shared weights on two input branches

**Purpose:** Learn that weights used by two matrix-product branches receive the sum of both branches' gradients.

**Inputs:** `X1` and `X2` are `(2, 3)`; shared `W` is `(3, 2)`.

**Forward operation:** `Y = X1 @ W + X2 @ W`.

**Derive/do:** Derive `dX1`, `dX2`, and total `dW`.

**Ingredients:** Reverse the final addition, backpropagate through both products, then add the two weight contributions.

**Required outputs:**

- `ex081_out_shape`: a Python tuple containing the forward output shape.
- `ex081_dX1`: the first input gradient.
- `ex081_dX2`: the second input gradient.
- `ex081_dW`: the total shared-weight gradient.

**Next concept:** Weight gradient under a summed objective.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
X1 = torch.linspace(-1.0, 1.5, steps=6, dtype=DTYPE).reshape(2, 3).requires_grad_()
X2 = torch.linspace(2.0, -0.5, steps=6, dtype=DTYPE).reshape(2, 3).requires_grad_()
W = torch.tensor([[0.5, 1.0], [-1.0, 2.0], [3.0, -0.5]], dtype=DTYPE, requires_grad=True)
Y = X1 @ W + X2 @ W
dout = torch.tensor([[1.0, -2.0], [0.5, 3.0]], dtype=DTYPE)
_capture("ex081", Y, {"dX1": X1, "dX2": X2, "dW": W}, dout)
print("X1", tuple(X1.shape), "X2", tuple(X2.shape), "W", tuple(W.shape), "Y", tuple(Y.shape))


In [ ]:
# Exercise 081: derive manually; do not use autograd in this cell.
# Define `ex081_out_shape` — a Python tuple containing the forward output shape.
# Define `ex081_dX1` — the first input gradient.
# Define `ex081_dX2` — the second input gradient.
# Define `ex081_dW` — the total shared-weight gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex081_out_shape", "ex081", "out_shape")
_check_tensor("ex081_dX1", "ex081", "dX1")
_check_tensor("ex081_dX2", "ex081", "dX2")
_check_tensor("ex081_dW", "ex081", "dW")


### Exercise 082 — Weight gradient under a summed objective

**Purpose:** Learn how a total over all outputs makes every output contribute equally to the weight gradient.

**Inputs:** `X` is `(4, 3)`, `W` is `(3, 2)`, and the objective totals all eight outputs.

**Forward operation:** `Y = X @ W` and `loss = Y.sum()`.

**Derive/do:** Derive `dY` and `dW`.

**Ingredients:** A total seeds each output with one times the scalar upstream gradient; then apply matrix backward.

**Required outputs:**

- `ex082_out_shape`: a Python tuple containing the forward output shape.
- `ex082_dY`: the gradient at all eight outputs.
- `ex082_dW`: the summed-objective weight gradient.

**Next concept:** Weight gradient under an averaged objective.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
X = torch.linspace(-1.0, 2.0, steps=12, dtype=DTYPE).reshape(4, 3).requires_grad_()
W = torch.linspace(-0.5, 1.0, steps=6, dtype=DTYPE).reshape(3, 2).requires_grad_()
Y = X @ W
loss_local = Y.sum()
dout = torch.tensor(1.0, dtype=DTYPE)
_capture("ex082", loss_local, {"dY": Y, "dW": W}, dout)
print("Y", tuple(Y.shape), "loss", tuple(loss_local.shape))


In [ ]:
# Exercise 082: derive manually; do not use autograd in this cell.
# Define `ex082_out_shape` — a Python tuple containing the forward output shape.
# Define `ex082_dY` — the gradient at all eight outputs.
# Define `ex082_dW` — the summed-objective weight gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex082_out_shape", "ex082", "out_shape")
_check_tensor("ex082_dY", "ex082", "dY")
_check_tensor("ex082_dW", "ex082", "dW")


### Exercise 083 — Weight gradient under an averaged objective

**Purpose:** Learn how replacing that total with a mean divides the output and weight gradients by the number of outputs.

**Inputs:** `X` is `(4, 3)`, `W` is `(3, 2)`, and the objective averages all eight outputs.

**Forward operation:** `Y = X @ W` and `loss = Y.mean()`.

**Derive/do:** Derive `dY` and `dW`.

**Ingredients:** Determine the exact number of averaged output entries before matrix backward.

**Required outputs:**

- `ex083_out_shape`: a Python tuple containing the forward output shape.
- `ex083_dY`: the gradient at all eight averaged outputs.
- `ex083_dW`: the averaged-objective weight gradient.

**Next concept:** Two linear layers.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
X = torch.linspace(-1.0, 2.0, steps=12, dtype=DTYPE).reshape(4, 3).requires_grad_()
W = torch.linspace(-0.5, 1.0, steps=6, dtype=DTYPE).reshape(3, 2).requires_grad_()
Y = X @ W
loss_local = Y.mean()
dout = torch.tensor(1.0, dtype=DTYPE)
_capture("ex083", loss_local, {"dY": Y, "dW": W}, dout)
print("Y", tuple(Y.shape), "loss", tuple(loss_local.shape))


In [ ]:
# Exercise 083: derive manually; do not use autograd in this cell.
# Define `ex083_out_shape` — a Python tuple containing the forward output shape.
# Define `ex083_dY` — the gradient at all eight averaged outputs.
# Define `ex083_dW` — the averaged-objective weight gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex083_out_shape", "ex083", "out_shape")
_check_tensor("ex083_dY", "ex083", "dY")
_check_tensor("ex083_dW", "ex083", "dW")


### Exercise 084 — Two linear layers

**Purpose:** Learn to work backward through two linear layers in the correct reverse order.

**Inputs:** `X` is `(2, 3)`, `W1` is `(3, 4)`, hidden `H` is `(2, 4)`, and `W2` is `(4, 2)`.

**Forward operation:** `H = X @ W1` and `Y = H @ W2`.

**Derive/do:** Derive `dH`, `dX`, `dW1`, and `dW2`.

**Ingredients:** Reverse the second product first; use its `dH` as the upstream gradient for the first product.

**Required outputs:**

- `ex084_out_shape`: a Python tuple containing the forward output shape.
- `ex084_dH`: the hidden-state gradient.
- `ex084_dX`: the input gradient.
- `ex084_dW1`: the first weight gradient.
- `ex084_dW2`: the second weight gradient.

**Next concept:** Multiply then add.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
X = torch.linspace(-1.0, 1.5, steps=6, dtype=DTYPE).reshape(2, 3).requires_grad_()
W1 = torch.linspace(-0.8, 1.4, steps=12, dtype=DTYPE).reshape(3, 4).requires_grad_()
H = X @ W1
W2 = torch.linspace(0.7, -1.0, steps=8, dtype=DTYPE).reshape(4, 2).requires_grad_()
Y = H @ W2
dout = torch.tensor([[1.0, -2.0], [0.5, 3.0]], dtype=DTYPE)
_capture("ex084", Y, {"dH": H, "dX": X, "dW1": W1, "dW2": W2}, dout)
print("X", tuple(X.shape), "H", tuple(H.shape), "Y", tuple(Y.shape))


In [ ]:
# Exercise 084: derive manually; do not use autograd in this cell.
# Define `ex084_out_shape` — a Python tuple containing the forward output shape.
# Define `ex084_dH` — the hidden-state gradient.
# Define `ex084_dX` — the input gradient.
# Define `ex084_dW1` — the first weight gradient.
# Define `ex084_dW2` — the second weight gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex084_out_shape", "ex084", "out_shape")
_check_tensor("ex084_dH", "ex084", "dH")
_check_tensor("ex084_dX", "ex084", "dX")
_check_tensor("ex084_dW1", "ex084", "dW1")
_check_tensor("ex084_dW2", "ex084", "dW2")


## 6. Shallow chains, deeper chains, and fan-out

A chain passes the upstream gradient through operations in reverse order. A branch sends one value along multiple paths; backward adds the contributions when those paths meet again.

For a branch from `x` to intermediate values `u` and `v`, the total gradient is

$$
\frac{\partial L}{\partial x}
=
\frac{\partial L}{\partial u}
\frac{\partial u}{\partial x}
+
\frac{\partial L}{\partial v}
\frac{\partial v}{\partial x}
$$

Here, both terms represent distinct paths from `x` to the same scalar objective `L`. A zero gradient blocks only its own path; another path may still contribute.


### Exercise 085 — Multiply then add

**Purpose:** Learn to reverse a two-step chain by undoing the last operation first.

**Inputs:** `x`, `a`, and `y` all have shape `(3,)`.

**Forward operation:** `a = 3 * x` and `y = a + 2`.

**Derive/do:** Derive `da` and `dx`.

**Ingredients:** Start from `dout`, reverse addition, then reverse multiplication.

**Required outputs:**

- `ex085_out_shape`: a Python tuple containing the forward output shape.
- `ex085_da`: the intermediate gradient with respect to `a`.
- `ex085_dx`: the gradient with respect to `x`.

**Next concept:** Square then sum.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([-2.0, 0.5, 3.0], dtype=DTYPE, requires_grad=True)
a = 3.0 * x
y = a + 2.0
dout = torch.tensor([1.0, -2.0, 0.5], dtype=DTYPE)
_capture("ex085", y, {"da": a, "dx": x}, dout)
print("x", tuple(x.shape), "a", tuple(a.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 085: derive manually; do not use autograd in this cell.
# Define `ex085_out_shape` — a Python tuple containing the forward output shape.
# Define `ex085_da` — the intermediate gradient with respect to `a`.
# Define `ex085_dx` — the gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex085_out_shape", "ex085", "out_shape")
_check_tensor("ex085_da", "ex085", "da")
_check_tensor("ex085_dx", "ex085", "dx")


### Exercise 086 — Square then sum

**Purpose:** Learn to reverse a scalar sum first and then the square that produced its entries.

**Inputs:** `x` and `squared` have shape `(4,)`; `y` is scalar.

**Forward operation:** `squared = x**2` and `y = squared.sum()`.

**Derive/do:** Derive `dsquared` and `dx`.

**Ingredients:** Reverse the total first, then the square.

**Required outputs:**

- `ex086_out_shape`: a Python tuple containing the forward output shape.
- `ex086_dsquared`: the gradient after reversing the sum.
- `ex086_dx`: the gradient after also reversing the square.

**Next concept:** Exponential then sum.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([-2.0, -0.5, 1.0, 3.0], dtype=DTYPE, requires_grad=True)
squared = x**2
y = squared.sum()
dout = torch.tensor(1.5, dtype=DTYPE)
_capture("ex086", y, {"dsquared": squared, "dx": x}, dout)
print("x", tuple(x.shape), "squared", tuple(squared.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 086: derive manually; do not use autograd in this cell.
# Define `ex086_out_shape` — a Python tuple containing the forward output shape.
# Define `ex086_dsquared` — the gradient after reversing the sum.
# Define `ex086_dx` — the gradient after also reversing the square.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex086_out_shape", "ex086", "out_shape")
_check_tensor("ex086_dsquared", "ex086", "dsquared")
_check_tensor("ex086_dx", "ex086", "dx")


### Exercise 087 — Exponential then sum

**Purpose:** Learn to reverse a sum and then use the saved exponential values to continue backward.

**Inputs:** `x` and `exp_x` have shape `(3,)`; `y` is scalar.

**Forward operation:** `exp_x = x.exp()` and `y = exp_x.sum()`.

**Derive/do:** Derive `dexp_x` and `dx`.

**Ingredients:** Reverse the sum, then use the exponential forward value.

**Required outputs:**

- `ex087_out_shape`: a Python tuple containing the forward output shape.
- `ex087_dexp_x`: the gradient with respect to `exp_x`.
- `ex087_dx`: the gradient with respect to `x`.

**Next concept:** Log then weighted sum.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([-1.0, 0.0, 1.5], dtype=DTYPE, requires_grad=True)
exp_x = x.exp()
y = exp_x.sum()
dout = torch.tensor(-0.75, dtype=DTYPE)
_capture("ex087", y, {"dexp_x": exp_x, "dx": x}, dout)
print("x", tuple(x.shape), "exp_x", tuple(exp_x.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 087: derive manually; do not use autograd in this cell.
# Define `ex087_out_shape` — a Python tuple containing the forward output shape.
# Define `ex087_dexp_x` — the gradient with respect to `exp_x`.
# Define `ex087_dx` — the gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex087_out_shape", "ex087", "out_shape")
_check_tensor("ex087_dexp_x", "ex087", "dexp_x")
_check_tensor("ex087_dx", "ex087", "dx")


### Exercise 088 — Log then weighted sum

**Purpose:** Learn to reverse a weighted sum before passing its gradient through a logarithm.

**Inputs:** Positive `x`, `log_x`, and `weights` have shape `(3,)`; `y` is scalar.

**Forward operation:** `log_x = x.log()` and `y = (log_x * weights).sum()`.

**Derive/do:** Derive `dlog_x` and `dx`.

**Ingredients:** Reverse sum, multiplication by weights, and logarithm.

**Required outputs:**

- `ex088_out_shape`: a Python tuple containing the forward output shape.
- `ex088_dlog_x`: the gradient entering the logarithm.
- `ex088_dx`: the gradient with respect to `x`.

**Next concept:** Tanh then weighted sum.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([0.5, 2.0, 5.0], dtype=DTYPE, requires_grad=True)
weights = torch.tensor([2.0, -1.0, 0.5], dtype=DTYPE)
log_x = x.log()
y = (log_x * weights).sum()
dout = torch.tensor(1.25, dtype=DTYPE)
_capture("ex088", y, {"dlog_x": log_x, "dx": x}, dout)
print("x", tuple(x.shape), "log_x", tuple(log_x.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 088: derive manually; do not use autograd in this cell.
# Define `ex088_out_shape` — a Python tuple containing the forward output shape.
# Define `ex088_dlog_x` — the gradient entering the logarithm.
# Define `ex088_dx` — the gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex088_out_shape", "ex088", "out_shape")
_check_tensor("ex088_dlog_x", "ex088", "dlog_x")
_check_tensor("ex088_dx", "ex088", "dx")


### Exercise 089 — Tanh then weighted sum

**Purpose:** Learn to pass a weighted-sum gradient backward through the MLP's `tanh` activation.

**Inputs:** `x`, `h`, and `weights` have shape `(4,)`; `y` is scalar.

**Forward operation:** `h = x.tanh()` and `y = (h * weights).sum()`.

**Derive/do:** Derive `dh` and `dx`.

**Ingredients:** Reverse the weighted sum, then apply tanh's local derivative using `h`.

**Required outputs:**

- `ex089_out_shape`: a Python tuple containing the forward output shape.
- `ex089_dh`: the gradient entering tanh.
- `ex089_dx`: the gradient with respect to `x`.

**Next concept:** Product, square, and sum.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([-2.0, -0.5, 0.5, 2.0], dtype=DTYPE, requires_grad=True)
weights = torch.tensor([1.0, -2.0, 0.5, 3.0], dtype=DTYPE)
h = x.tanh()
y = (h * weights).sum()
dout = torch.tensor(-1.5, dtype=DTYPE)
_capture("ex089", y, {"dh": h, "dx": x}, dout)
print("x", tuple(x.shape), "h", tuple(h.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 089: derive manually; do not use autograd in this cell.
# Define `ex089_out_shape` — a Python tuple containing the forward output shape.
# Define `ex089_dh` — the gradient entering tanh.
# Define `ex089_dx` — the gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex089_out_shape", "ex089", "out_shape")
_check_tensor("ex089_dh", "ex089", "dh")
_check_tensor("ex089_dx", "ex089", "dx")


### Exercise 090 — Product, square, and sum

**Purpose:** Learn to track two inputs backward through multiplication, squaring, and summation.

**Inputs:** `x`, `z`, and `product` have shape `(3,)`; `y` is scalar.

**Forward operation:** `product = x * z`, `squared = product**2`, and `y = squared.sum()`.

**Derive/do:** Derive `dsquared`, `dproduct`, `dx`, and `dz`.

**Ingredients:** Reverse one operation at a time; product backward sends gradients to both inputs.

**Required outputs:**

- `ex090_out_shape`: a Python tuple containing the forward output shape.
- `ex090_dsquared`: the gradient after reversing the sum.
- `ex090_dproduct`: the gradient after reversing the square.
- `ex090_dx`: the gradient with respect to `x`.
- `ex090_dz`: the gradient with respect to `z`.

**Next concept:** Square and linear fan-out.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([2.0, -1.0, 3.0], dtype=DTYPE, requires_grad=True)
z = torch.tensor([0.5, 4.0, -2.0], dtype=DTYPE, requires_grad=True)
product = x * z
squared = product**2
y = squared.sum()
dout = torch.tensor(0.75, dtype=DTYPE)
_capture("ex090", y, {"dsquared": squared, "dproduct": product, "dx": x, "dz": z}, dout)
print("product", tuple(product.shape), "squared", tuple(squared.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 090: derive manually; do not use autograd in this cell.
# Define `ex090_out_shape` — a Python tuple containing the forward output shape.
# Define `ex090_dsquared` — the gradient after reversing the sum.
# Define `ex090_dproduct` — the gradient after reversing the square.
# Define `ex090_dx` — the gradient with respect to `x`.
# Define `ex090_dz` — the gradient with respect to `z`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex090_out_shape", "ex090", "out_shape")
_check_tensor("ex090_dsquared", "ex090", "dsquared")
_check_tensor("ex090_dproduct", "ex090", "dproduct")
_check_tensor("ex090_dx", "ex090", "dx")
_check_tensor("ex090_dz", "ex090", "dz")


### Exercise 091 — Square and linear fan-out

**Purpose:** Learn that when `x` reaches the output through a square branch and a linear branch, both gradients must be added.

**Inputs:** `x`, `square`, `linear`, and `y` have shape `(3,)`.

**Forward operation:** `square = x**2`, `linear = 3*x`, and `y = square + linear`.

**Derive/do:** Derive `dsquare`, `dlinear`, and total `dx`.

**Ingredients:** Reverse the final addition; compute each branch contribution to `x`; add them.

**Required outputs:**

- `ex091_out_shape`: a Python tuple containing the forward output shape.
- `ex091_dsquare`: the upstream gradient on the square branch.
- `ex091_dlinear`: the upstream gradient on the linear branch.
- `ex091_dx`: the sum of both contributions at `x`.

**Next concept:** Exponential and square fan-out.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([-2.0, 0.5, 3.0], dtype=DTYPE, requires_grad=True)
square = x**2
linear = 3.0 * x
y = square + linear
dout = torch.tensor([1.0, -2.0, 0.5], dtype=DTYPE)
_capture("ex091", y, {"dsquare": square, "dlinear": linear, "dx": x}, dout)
print("All forward branch tensors have shape", tuple(y.shape))


In [ ]:
# Exercise 091: derive manually; do not use autograd in this cell.
# Define `ex091_out_shape` — a Python tuple containing the forward output shape.
# Define `ex091_dsquare` — the upstream gradient on the square branch.
# Define `ex091_dlinear` — the upstream gradient on the linear branch.
# Define `ex091_dx` — the sum of both contributions at `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex091_out_shape", "ex091", "out_shape")
_check_tensor("ex091_dsquare", "ex091", "dsquare")
_check_tensor("ex091_dlinear", "ex091", "dlinear")
_check_tensor("ex091_dx", "ex091", "dx")


### Exercise 092 — Exponential and square fan-out

**Purpose:** Learn to add the two gradients that return to `x` through exponential and square branches.

**Inputs:** `x`, both branches, and `y` have shape `(3,)`.

**Forward operation:** `left = x.exp()`, `right = x**2`, and `y = left + right`.

**Derive/do:** Derive branch gradients and total `dx`.

**Ingredients:** Reverse addition; use each branch's local derivative; add at the shared source.

**Required outputs:**

- `ex092_out_shape`: a Python tuple containing the forward output shape.
- `ex092_dleft`: the upstream gradient on the exponential branch.
- `ex092_dright`: the upstream gradient on the square branch.
- `ex092_dx`: the total gradient with respect to `x`.

**Next concept:** One tensor in both product slots.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([-1.0, 0.5, 2.0], dtype=DTYPE, requires_grad=True)
left = x.exp()
right = x**2
y = left + right
dout = torch.tensor([2.0, -1.0, 0.25], dtype=DTYPE)
_capture("ex092", y, {"dleft": left, "dright": right, "dx": x}, dout)
print("All forward branch tensors have shape", tuple(y.shape))


In [ ]:
# Exercise 092: derive manually; do not use autograd in this cell.
# Define `ex092_out_shape` — a Python tuple containing the forward output shape.
# Define `ex092_dleft` — the upstream gradient on the exponential branch.
# Define `ex092_dright` — the upstream gradient on the square branch.
# Define `ex092_dx` — the total gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex092_out_shape", "ex092", "out_shape")
_check_tensor("ex092_dleft", "ex092", "dleft")
_check_tensor("ex092_dright", "ex092", "dright")
_check_tensor("ex092_dx", "ex092", "dx")


### Exercise 093 — One tensor in both product slots

**Purpose:** Learn that `x * x` uses `x` twice, so backward has two contributions even though both inputs have the same name.

**Inputs:** `x` and `y` have shape `(3,)`.

**Forward operation:** `y = x * x`.

**Derive/do:** Derive total `dx` without treating the two uses as one path.

**Ingredients:** Imagine temporarily naming the left and right copies separately; compute and add both contributions.

**Required outputs:**

- `ex093_out_shape`: a Python tuple containing the forward output shape.
- `ex093_dx`: the total gradient from both uses of `x`.

**Next concept:** Broadcast, square, and average.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([-2.0, 0.5, 3.0], dtype=DTYPE, requires_grad=True)
y = x * x
dout = torch.tensor([1.0, -2.0, 0.5], dtype=DTYPE)
_capture("ex093", y, {"dx": x}, dout)
print("x", tuple(x.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 093: derive manually; do not use autograd in this cell.
# Define `ex093_out_shape` — a Python tuple containing the forward output shape.
# Define `ex093_dx` — the total gradient from both uses of `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex093_out_shape", "ex093", "out_shape")
_check_tensor("ex093_dx", "ex093", "dx")


### Exercise 094 — Broadcast, square, and average

**Purpose:** Learn to combine a mean, a square, and a reused bias in one backward chain.

**Inputs:** `x` is `(3, 4)`, `bias` is `(4,)`, and the scalar loss averages 12 squared entries.

**Forward operation:** `pre = x + bias`, `squared = pre**2`, and `loss = squared.mean()`.

**Derive/do:** Derive `dsquared`, `dpre`, `dx`, and `dbias`.

**Ingredients:** Reverse mean, square, and broadcast addition in order.

**Required outputs:**

- `ex094_out_shape`: a Python tuple containing the forward output shape.
- `ex094_dsquared`: the gradient after reversing the mean.
- `ex094_dpre`: the gradient after reversing the square.
- `ex094_dx`: the input gradient.
- `ex094_dbias`: the accumulated bias gradient.

**Next concept:** Row totals, square, then total.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.linspace(-2.0, 3.5, steps=12, dtype=DTYPE).reshape(3, 4).requires_grad_()
bias = torch.tensor([0.5, -1.0, 2.0, 3.0], dtype=DTYPE, requires_grad=True)
pre = x + bias
squared = pre**2
loss_local = squared.mean()
dout = torch.tensor(1.0, dtype=DTYPE)
_capture("ex094", loss_local, {"dsquared": squared, "dpre": pre, "dx": x, "dbias": bias}, dout)
print("x", tuple(x.shape), "pre", tuple(pre.shape), "loss", tuple(loss_local.shape))


In [ ]:
# Exercise 094: derive manually; do not use autograd in this cell.
# Define `ex094_out_shape` — a Python tuple containing the forward output shape.
# Define `ex094_dsquared` — the gradient after reversing the mean.
# Define `ex094_dpre` — the gradient after reversing the square.
# Define `ex094_dx` — the input gradient.
# Define `ex094_dbias` — the accumulated bias gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex094_out_shape", "ex094", "out_shape")
_check_tensor("ex094_dsquared", "ex094", "dsquared")
_check_tensor("ex094_dpre", "ex094", "dpre")
_check_tensor("ex094_dx", "ex094", "dx")
_check_tensor("ex094_dbias", "ex094", "dbias")


### Exercise 095 — Row totals, square, then total

**Purpose:** Learn to restore matrix-shaped gradients after row sums are squared and then summed again.

**Inputs:** `x` is `(3, 4)`, `rows` is `(3,)`, and the final output is scalar.

**Forward operation:** `rows = x.sum(dim=1)`, `squared = rows**2`, and `y = squared.sum()`.

**Derive/do:** Derive `dsquared`, `drows`, and `dx`.

**Ingredients:** Reverse the final total, square, then row reduction; restore four columns per row.

**Required outputs:**

- `ex095_out_shape`: a Python tuple containing the forward output shape.
- `ex095_dsquared`: the gradient after reversing the final sum.
- `ex095_drows`: the gradient with respect to row totals.
- `ex095_dx`: the matrix gradient with respect to `x`.

**Next concept:** Affine then tanh.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.arange(1, 13, dtype=DTYPE).reshape(3, 4).requires_grad_()
rows = x.sum(dim=1)
squared = rows**2
y = squared.sum()
dout = torch.tensor(-0.5, dtype=DTYPE)
_capture("ex095", y, {"dsquared": squared, "drows": rows, "dx": x}, dout)
print("x", tuple(x.shape), "rows", tuple(rows.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 095: derive manually; do not use autograd in this cell.
# Define `ex095_out_shape` — a Python tuple containing the forward output shape.
# Define `ex095_dsquared` — the gradient after reversing the final sum.
# Define `ex095_drows` — the gradient with respect to row totals.
# Define `ex095_dx` — the matrix gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex095_out_shape", "ex095", "out_shape")
_check_tensor("ex095_dsquared", "ex095", "dsquared")
_check_tensor("ex095_drows", "ex095", "drows")
_check_tensor("ex095_dx", "ex095", "dx")


### Exercise 096 — Affine then tanh

**Purpose:** Learn to reverse `tanh` and then the matrix multiplication and bias addition before it.

**Inputs:** `X` is `(3, 2)`, `W` is `(2, 4)`, `bias` is `(4,)`, and `h` is `(3, 4)`.

**Forward operation:** `pre = X @ W + bias` and `h = pre.tanh()`.

**Derive/do:** Derive `dpre`, `dX`, `dW`, and `dbias` from supplied `dout`.

**Ingredients:** Reverse tanh first, then the affine layer; preserve batch and hidden axes.

**Required outputs:**

- `ex096_out_shape`: a Python tuple containing the forward output shape.
- `ex096_dpre`: the gradient entering the affine operation.
- `ex096_dX`: the input gradient.
- `ex096_dW`: the weight gradient.
- `ex096_dbias`: the bias gradient.

**Next concept:** Affine then ReLU then total.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
X = torch.linspace(-1.0, 1.5, steps=6, dtype=DTYPE).reshape(3, 2).requires_grad_()
W = torch.linspace(-0.8, 1.3, steps=8, dtype=DTYPE).reshape(2, 4).requires_grad_()
bias = torch.tensor([0.25, -0.5, 1.0, -1.5], dtype=DTYPE, requires_grad=True)
pre = X @ W + bias
h = pre.tanh()
dout = torch.linspace(1.0, -1.2, steps=12, dtype=DTYPE).reshape(3, 4)
_capture("ex096", h, {"dpre": pre, "dX": X, "dW": W, "dbias": bias}, dout)
print("X", tuple(X.shape), "pre", tuple(pre.shape), "h", tuple(h.shape))


In [ ]:
# Exercise 096: derive manually; do not use autograd in this cell.
# Define `ex096_out_shape` — a Python tuple containing the forward output shape.
# Define `ex096_dpre` — the gradient entering the affine operation.
# Define `ex096_dX` — the input gradient.
# Define `ex096_dW` — the weight gradient.
# Define `ex096_dbias` — the bias gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex096_out_shape", "ex096", "out_shape")
_check_tensor("ex096_dpre", "ex096", "dpre")
_check_tensor("ex096_dX", "ex096", "dX")
_check_tensor("ex096_dW", "ex096", "dW")
_check_tensor("ex096_dbias", "ex096", "dbias")


### Exercise 097 — Affine then ReLU then total

**Purpose:** Learn to reverse a scalar total, then a ReLU mask, and then an affine layer.

**Inputs:** `pre` and `h` are `(3, 4)`; no pre-activation is exactly zero.

**Forward operation:** `pre = X @ W + bias`, `h = pre.relu()`, and `loss = h.sum()`.

**Derive/do:** Derive `dh`, `dpre`, `dX`, `dW`, and `dbias`.

**Ingredients:** Reverse total, ReLU mask, and affine layer.

**Required outputs:**

- `ex097_out_shape`: a Python tuple containing the forward output shape.
- `ex097_dh`: the gradient after reversing the total.
- `ex097_dpre`: the gradient after reversing ReLU.
- `ex097_dX`: the input gradient.
- `ex097_dW`: the weight gradient.
- `ex097_dbias`: the bias gradient.

**Next concept:** Exp, sum, then log.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
X = torch.tensor([[-1.0, 0.5], [1.5, -2.0], [0.25, 3.0]], dtype=DTYPE, requires_grad=True)
W = torch.tensor([[0.7, -1.2, 2.1, 0.3], [-0.4, 1.3, 0.6, -2.0]], dtype=DTYPE, requires_grad=True)
bias = torch.tensor([0.11, -0.37, 0.83, -1.19], dtype=DTYPE, requires_grad=True)
pre = X @ W + bias
h = pre.relu()
loss_local = h.sum()
dout = torch.tensor(0.75, dtype=DTYPE)
_capture("ex097", loss_local, {"dh": h, "dpre": pre, "dX": X, "dW": W, "dbias": bias}, dout)
print("pre", tuple(pre.shape), "h", tuple(h.shape), "loss", tuple(loss_local.shape))


In [ ]:
# Exercise 097: derive manually; do not use autograd in this cell.
# Define `ex097_out_shape` — a Python tuple containing the forward output shape.
# Define `ex097_dh` — the gradient after reversing the total.
# Define `ex097_dpre` — the gradient after reversing ReLU.
# Define `ex097_dX` — the input gradient.
# Define `ex097_dW` — the weight gradient.
# Define `ex097_dbias` — the bias gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex097_out_shape", "ex097", "out_shape")
_check_tensor("ex097_dh", "ex097", "dh")
_check_tensor("ex097_dpre", "ex097", "dpre")
_check_tensor("ex097_dX", "ex097", "dX")
_check_tensor("ex097_dW", "ex097", "dW")
_check_tensor("ex097_dbias", "ex097", "dbias")


### Exercise 098 — Exp, sum, then log

**Purpose:** Learn to work backward through exponential, sum, and logarithm operations one step at a time.

**Inputs:** `x` and `exp_x` are `(4,)`; `total` and `y` are scalar.

**Forward operation:** `exp_x = x.exp()`, `total = exp_x.sum()`, and `y = total.log()`.

**Derive/do:** Derive `dtotal`, `dexp_x`, and `dx`.

**Ingredients:** Reverse log, scalar sum, and exp in order.

**Required outputs:**

- `ex098_out_shape`: a Python tuple containing the forward output shape.
- `ex098_dtotal`: the gradient entering the sum.
- `ex098_dexp_x`: the gradient entering exp.
- `ex098_dx`: the gradient with respect to `x`.

**Next concept:** Subtract a computed maximum.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([-1.0, 0.0, 0.5, 1.5], dtype=DTYPE, requires_grad=True)
exp_x = x.exp()
total = exp_x.sum()
y = total.log()
dout = torch.tensor(1.25, dtype=DTYPE)
_capture("ex098", y, {"dtotal": total, "dexp_x": exp_x, "dx": x}, dout)
print("x", tuple(x.shape), "total", tuple(total.shape), "y", tuple(y.shape))


In [ ]:
# Exercise 098: derive manually; do not use autograd in this cell.
# Define `ex098_out_shape` — a Python tuple containing the forward output shape.
# Define `ex098_dtotal` — the gradient entering the sum.
# Define `ex098_dexp_x` — the gradient entering exp.
# Define `ex098_dx` — the gradient with respect to `x`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex098_out_shape", "ex098", "out_shape")
_check_tensor("ex098_dtotal", "ex098", "dtotal")
_check_tensor("ex098_dexp_x", "ex098", "dexp_x")
_check_tensor("ex098_dx", "ex098", "dx")


### Exercise 099 — Subtract a computed maximum

**Purpose:** Learn that subtracting a maximum gives `x` a direct path and another path through the winning maximum position.

**Inputs:** `x` is `(4,)`, `maximum` is scalar, and `shifted` is `(4,)`; the maximum is unique.

**Forward operation:** `maximum = x.max()` and `shifted = x - maximum`.

**Derive/do:** Derive `dmaximum` and total `dx`.

**Ingredients:** The direct subtraction path reaches every `x`; the maximum path returns only to the winning index; add them.

**Required outputs:**

- `ex099_out_shape`: a Python tuple containing the forward output shape.
- `ex099_dmaximum`: the scalar gradient with respect to the computed maximum.
- `ex099_dx`: the total gradient from direct and maximum paths.

**Next concept:** Two scalar objectives from one tensor.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
x = torch.tensor([-1.0, 3.0, 2.0, 0.5], dtype=DTYPE, requires_grad=True)
maximum = x.max()
shifted = x - maximum
dout = torch.tensor([1.0, -2.0, 0.5, 3.0], dtype=DTYPE)
_capture("ex099", shifted, {"dmaximum": maximum, "dx": x}, dout)
print("x", tuple(x.shape), "maximum", tuple(maximum.shape), "shifted", tuple(shifted.shape))


In [ ]:
# Exercise 099: derive manually; do not use autograd in this cell.
# Define `ex099_out_shape` — a Python tuple containing the forward output shape.
# Define `ex099_dmaximum` — the scalar gradient with respect to the computed maximum.
# Define `ex099_dx` — the total gradient from direct and maximum paths.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex099_out_shape", "ex099", "out_shape")
_check_tensor("ex099_dmaximum", "ex099", "dmaximum")
_check_tensor("ex099_dx", "ex099", "dx")


### Exercise 100 — Two scalar objectives from one tensor

**Purpose:** Learn that one tensor used by two scalar objectives receives the sum of both objectives' gradients.

**Inputs:** `x` has shape `(2, 3)`; both branch outputs are scalar.

**Forward operation:** `left = (x**2).sum()`, `right = x.mean()`, and `y = left + right`.

**Derive/do:** Derive `dleft`, `dright`, and total `dx`.

**Ingredients:** Reverse the scalar addition; independently backpropagate each branch; add both `x` contributions.

**Required outputs:**

- `ex100_out_shape`: a Python tuple containing the forward output shape.
- `ex100_dleft`: the scalar gradient entering the square-sum branch.
- `ex100_dright`: the scalar gradient entering the mean branch.
- `ex100_dx`: the total gradient from both branches.

**Next concept:** One embedding row.


In [21]:
# Supplied fixture: run this before writing the manual answer.
x = torch.linspace(-2.0, 3.0, steps=6, dtype=DTYPE).reshape(2, 3).requires_grad_()
left = (x**2).sum()
right = x.mean()
y = left + right
dout = torch.tensor(-0.75, dtype=DTYPE)
_capture("ex100", y, {"dleft": left, "dright": right, "dx": x}, dout)
print("x", tuple(x.shape), "left", tuple(left.shape), "right", tuple(right.shape), "y", tuple(y.shape))


x (2, 3) left () right () y ()


In [22]:
# Exercise 100: derive manually; do not use autograd in this cell.
# Define `ex100_out_shape` — a Python tuple containing the forward output shape.
# Define `ex100_dleft` — the scalar gradient entering the square-sum branch.
# Define `ex100_dright` — the scalar gradient entering the mean branch.
# Define `ex100_dx` — the total gradient from both branches.
# Write your tensor operations below these comments, then run the test cell.


In [23]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex100_out_shape", "ex100", "out_shape")
_check_tensor("ex100_dleft", "ex100", "dleft")
_check_tensor("ex100_dright", "ex100", "dright")
_check_tensor("ex100_dx", "ex100", "dx")


AssertionError: Define `ex100_out_shape` in the answer cell first.

## 7. Embedding lookups and indexed gradient accumulation

An embedding lookup uses integer IDs to select rows. Forward indexing replaces each ID with a vector. Backward sends each selected vector gradient back to its source row. Unselected rows receive zero. If an ID occurs more than once, all of its gradient contributions add into the same embedding-table row.

This is the backward counterpart of the lookup shape rule practiced in the indexing workbook.


### Exercise 101 — One embedding row

**Purpose:** Learn that selecting one embedding row sends a vector gradient to that row and zero to every unselected row.

**Inputs:** `table` is `(5, 3)` and `token_id` is one integer.

**Forward operation:** `emb = table[token_id]`.

**Derive/do:** Derive the full table gradient `dtable`.

**Ingredients:** Start with zeros shaped like the table and place the upstream vector at the selected row.

**Required outputs:**

- `ex101_out_shape`: a Python tuple containing the forward output shape.
- `ex101_dtable`: the `(5, 3)` table gradient.

**Next concept:** Several unique embedding rows.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
table = (torch.arange(15, dtype=DTYPE).reshape(5, 3) / 10).requires_grad_()
token_id = 2
emb = table[token_id]
dout = torch.tensor([1.0, -2.0, 0.5], dtype=DTYPE)
_capture("ex101", emb, {"dtable": table}, dout)
print("table", tuple(table.shape), "emb", tuple(emb.shape), "dout", tuple(dout.shape))


In [ ]:
# Exercise 101: derive manually; do not use autograd in this cell.
# Define `ex101_out_shape` — a Python tuple containing the forward output shape.
# Define `ex101_dtable` — the `(5, 3)` table gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex101_out_shape", "ex101", "out_shape")
_check_tensor("ex101_dtable", "ex101", "dtable")


### Exercise 102 — Several unique embedding rows

**Purpose:** Learn to send several lookup gradients back to their corresponding embedding-table rows.

**Inputs:** `table` is `(6, 3)` and `ids = [4, 1, 3]` has shape `(3,)`.

**Forward operation:** `emb = table[ids]`.

**Derive/do:** Derive `dtable`.

**Ingredients:** Each row of `dout` belongs to the corresponding ID; all other table rows stay zero.

**Required outputs:**

- `ex102_out_shape`: a Python tuple containing the forward output shape.
- `ex102_dtable`: the `(6, 3)` table gradient.

**Next concept:** Repeated embedding IDs.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
table = (torch.arange(18, dtype=DTYPE).reshape(6, 3) / 10).requires_grad_()
ids = torch.tensor([4, 1, 3], dtype=torch.long)
emb = table[ids]
dout = torch.tensor([[1.0, 0.5, -1.0], [-2.0, 3.0, 0.25], [0.5, -0.75, 2.0]], dtype=DTYPE)
_capture("ex102", emb, {"dtable": table}, dout)
print("table", tuple(table.shape), "ids", tuple(ids.shape), "emb", tuple(emb.shape))


In [ ]:
# Exercise 102: derive manually; do not use autograd in this cell.
# Define `ex102_out_shape` — a Python tuple containing the forward output shape.
# Define `ex102_dtable` — the `(6, 3)` table gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex102_out_shape", "ex102", "out_shape")
_check_tensor("ex102_dtable", "ex102", "dtable")


### Exercise 103 — Repeated embedding IDs

**Purpose:** Learn that repeated token IDs make several gradients add into the same embedding-table row.

**Inputs:** `table` is `(6, 3)` and `ids = [2, 4, 2, 2]`.

**Forward operation:** `emb = table[ids]`.

**Derive/do:** Derive `dtable`, paying special attention to row 2.

**Ingredients:** Use an accumulating indexed write such as `index_add_`; plain replacement loses repeated contributions.

**Required outputs:**

- `ex103_out_shape`: a Python tuple containing the forward output shape.
- `ex103_dtable`: the table gradient with repeated-ID accumulation.

**Next concept:** A rank-2 grid of IDs.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
table = (torch.arange(18, dtype=DTYPE).reshape(6, 3) / 10).requires_grad_()
ids = torch.tensor([2, 4, 2, 2], dtype=torch.long)
emb = table[ids]
dout = torch.tensor([[1.0, 0.5, -1.0], [-2.0, 3.0, 0.25], [0.5, -0.75, 2.0], [3.0, 1.0, -0.5]], dtype=DTYPE)
_capture("ex103", emb, {"dtable": table}, dout)
print("table", tuple(table.shape), "ids", tuple(ids.shape), "emb", tuple(emb.shape))


In [ ]:
# Exercise 103: derive manually; do not use autograd in this cell.
# Define `ex103_out_shape` — a Python tuple containing the forward output shape.
# Define `ex103_dtable` — the table gradient with repeated-ID accumulation.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex103_out_shape", "ex103", "out_shape")
_check_tensor("ex103_dtable", "ex103", "dtable")


### Exercise 104 — A rank-2 grid of IDs

**Purpose:** Learn to map a two-dimensional grid of lookup gradients back into embedding-table rows.

**Inputs:** `table` is `(6, 2)`, `ids` is `(2, 3)`, and `emb` is `(2, 3, 2)`.

**Forward operation:** `emb = table[ids]`.

**Derive/do:** Derive `dtable` across all six batch-position lookups.

**Ingredients:** Flatten ID positions and embedding gradients conceptually, then accumulate by ID.

**Required outputs:**

- `ex104_out_shape`: a Python tuple containing the forward output shape.
- `ex104_dtable`: the `(6, 2)` table gradient.

**Next concept:** Weighted embedding outputs.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
table = (torch.arange(12, dtype=DTYPE).reshape(6, 2) / 10).requires_grad_()
ids = torch.tensor([[2, 1, 2], [4, 0, 1]], dtype=torch.long)
emb = table[ids]
dout = torch.linspace(-1.0, 1.2, steps=12, dtype=DTYPE).reshape(2, 3, 2)
_capture("ex104", emb, {"dtable": table}, dout)
print("table", tuple(table.shape), "ids", tuple(ids.shape), "emb", tuple(emb.shape))


In [ ]:
# Exercise 104: derive manually; do not use autograd in this cell.
# Define `ex104_out_shape` — a Python tuple containing the forward output shape.
# Define `ex104_dtable` — the `(6, 2)` table gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex104_out_shape", "ex104", "out_shape")
_check_tensor("ex104_dtable", "ex104", "dtable")


### Exercise 105 — Weighted embedding outputs

**Purpose:** Learn that every lookup position can send a different vector gradient back to its selected table row.

**Inputs:** `table` is `(7, 3)`, `ids` is `(2, 2)`, and `emb` is `(2, 2, 3)`.

**Forward operation:** `emb = table[ids]` with the supplied `dout`.

**Derive/do:** Derive `dtable`.

**Ingredients:** Every trailing length-3 vector is one lookup contribution; accumulate repeated IDs.

**Required outputs:**

- `ex105_out_shape`: a Python tuple containing the forward output shape.
- `ex105_dtable`: the table gradient from arbitrary lookup-vector gradients.

**Next concept:** Lookup then flatten.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
table = (torch.arange(21, dtype=DTYPE).reshape(7, 3) / 10).requires_grad_()
ids = torch.tensor([[5, 2], [5, 1]], dtype=torch.long)
emb = table[ids]
dout = torch.linspace(-1.0, 1.2, steps=12, dtype=DTYPE).reshape(2, 2, 3)
_capture("ex105", emb, {"dtable": table}, dout)
print("table", tuple(table.shape), "ids", tuple(ids.shape), "emb", tuple(emb.shape))


In [ ]:
# Exercise 105: derive manually; do not use autograd in this cell.
# Define `ex105_out_shape` — a Python tuple containing the forward output shape.
# Define `ex105_dtable` — the table gradient from arbitrary lookup-vector gradients.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex105_out_shape", "ex105", "out_shape")
_check_tensor("ex105_dtable", "ex105", "dtable")


### Exercise 106 — Lookup then flatten

**Purpose:** Learn to restore an embedding tensor's shape before sending its gradients back through the lookup.

**Inputs:** `ids` is `(2, 3)`, `emb` is `(2, 3, 2)`, and `flat` is `(2, 6)`.

**Forward operation:** `emb = table[ids]` and `flat = emb.reshape(2, 6)`.

**Derive/do:** Derive `demb` and `dtable` from the supplied `(2, 6)` upstream tensor.

**Ingredients:** A reshape backward restores the old shape without changing element order; then accumulate by ID.

**Required outputs:**

- `ex106_out_shape`: a Python tuple containing the forward output shape.
- `ex106_demb`: the upstream gradient restored to `(2, 3, 2)`.
- `ex106_dtable`: the accumulated embedding-table gradient.

**Next concept:** Pool context embeddings by sum.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
table = (torch.arange(12, dtype=DTYPE).reshape(6, 2) / 10).requires_grad_()
ids = torch.tensor([[2, 1, 2], [4, 0, 1]], dtype=torch.long)
emb = table[ids]
flat = emb.reshape(2, 6)
dout = torch.linspace(-1.0, 1.2, steps=12, dtype=DTYPE).reshape(2, 6)
_capture("ex106", flat, {"demb": emb, "dtable": table}, dout)
print("emb", tuple(emb.shape), "flat", tuple(flat.shape))


In [ ]:
# Exercise 106: derive manually; do not use autograd in this cell.
# Define `ex106_out_shape` — a Python tuple containing the forward output shape.
# Define `ex106_demb` — the upstream gradient restored to `(2, 3, 2)`.
# Define `ex106_dtable` — the accumulated embedding-table gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex106_out_shape", "ex106", "out_shape")
_check_tensor("ex106_demb", "ex106", "demb")
_check_tensor("ex106_dtable", "ex106", "dtable")


### Exercise 107 — Pool context embeddings by sum

**Purpose:** Learn to copy each pooled-sum gradient back to every context position before updating embedding rows.

**Inputs:** `emb` is `(2, 3, 2)` and `pooled = emb.sum(dim=1)` is `(2, 2)`.

**Forward operation:** Look up the IDs, then sum three context positions per example.

**Derive/do:** Derive `demb` and `dtable`.

**Ingredients:** Broadcast each example's pooled gradient across its three positions, then accumulate rows by token ID.

**Required outputs:**

- `ex107_out_shape`: a Python tuple containing the forward output shape.
- `ex107_demb`: the `(2, 3, 2)` gradient before lookup backward.
- `ex107_dtable`: the accumulated table gradient.

**Next concept:** Pool context embeddings by mean.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
table = (torch.arange(12, dtype=DTYPE).reshape(6, 2) / 10).requires_grad_()
ids = torch.tensor([[2, 1, 2], [4, 0, 1]], dtype=torch.long)
emb = table[ids]
pooled = emb.sum(dim=1)
dout = torch.tensor([[1.0, -2.0], [0.5, 3.0]], dtype=DTYPE)
_capture("ex107", pooled, {"demb": emb, "dtable": table}, dout)
print("emb", tuple(emb.shape), "pooled", tuple(pooled.shape))


In [ ]:
# Exercise 107: derive manually; do not use autograd in this cell.
# Define `ex107_out_shape` — a Python tuple containing the forward output shape.
# Define `ex107_demb` — the `(2, 3, 2)` gradient before lookup backward.
# Define `ex107_dtable` — the accumulated table gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex107_out_shape", "ex107", "out_shape")
_check_tensor("ex107_demb", "ex107", "demb")
_check_tensor("ex107_dtable", "ex107", "dtable")


### Exercise 108 — Pool context embeddings by mean

**Purpose:** Learn that mean pooling divides each example's gradient among its context positions before lookup backward.

**Inputs:** `emb` is `(2, 3, 2)` and `pooled = emb.mean(dim=1)` is `(2, 2)`.

**Forward operation:** Average the three context embeddings per example.

**Derive/do:** Derive `demb` and `dtable`.

**Ingredients:** Each pooled gradient is divided across three positions before ID accumulation.

**Required outputs:**

- `ex108_out_shape`: a Python tuple containing the forward output shape.
- `ex108_demb`: the position-wise gradient after reversing the mean.
- `ex108_dtable`: the accumulated table gradient.

**Next concept:** Lookup, flatten, then linear.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
table = (torch.arange(12, dtype=DTYPE).reshape(6, 2) / 10).requires_grad_()
ids = torch.tensor([[2, 1, 2], [4, 0, 1]], dtype=torch.long)
emb = table[ids]
pooled = emb.mean(dim=1)
dout = torch.tensor([[1.0, -2.0], [0.5, 3.0]], dtype=DTYPE)
_capture("ex108", pooled, {"demb": emb, "dtable": table}, dout)
print("emb", tuple(emb.shape), "pooled", tuple(pooled.shape))


In [ ]:
# Exercise 108: derive manually; do not use autograd in this cell.
# Define `ex108_out_shape` — a Python tuple containing the forward output shape.
# Define `ex108_demb` — the position-wise gradient after reversing the mean.
# Define `ex108_dtable` — the accumulated table gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex108_out_shape", "ex108", "out_shape")
_check_tensor("ex108_demb", "ex108", "demb")
_check_tensor("ex108_dtable", "ex108", "dtable")


### Exercise 109 — Lookup, flatten, then linear

**Purpose:** Learn to reverse a linear layer, restore the embedding shape, and then accumulate embedding-table gradients.

**Inputs:** `flat` is `(2, 6)`, `W` is `(6, 4)`, and `Y` is `(2, 4)`.

**Forward operation:** `emb = table[ids]`, `flat = emb.reshape(2, 6)`, and `Y = flat @ W`.

**Derive/do:** Derive `dW`, `dflat`, `demb`, and `dtable`.

**Ingredients:** Reverse matrix multiplication, restore the embedding shape, then accumulate repeated IDs.

**Required outputs:**

- `ex109_out_shape`: a Python tuple containing the forward output shape.
- `ex109_dW`: the projection-weight gradient.
- `ex109_dflat`: the flattened embedding gradient.
- `ex109_demb`: the restored embedding-grid gradient.
- `ex109_dtable`: the accumulated table gradient.

**Next concept:** Paired target selection backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
table = (torch.arange(12, dtype=DTYPE).reshape(6, 2) / 10).requires_grad_()
ids = torch.tensor([[2, 1, 2], [4, 0, 1]], dtype=torch.long)
emb = table[ids]
flat = emb.reshape(2, 6)
W = torch.linspace(-1.0, 1.3, steps=24, dtype=DTYPE).reshape(6, 4).requires_grad_()
Y = flat @ W
dout = torch.linspace(1.0, -0.7, steps=8, dtype=DTYPE).reshape(2, 4)
_capture("ex109", Y, {"dW": W, "dflat": flat, "demb": emb, "dtable": table}, dout)
print("emb", tuple(emb.shape), "flat", tuple(flat.shape), "W", tuple(W.shape), "Y", tuple(Y.shape))


In [ ]:
# Exercise 109: derive manually; do not use autograd in this cell.
# Define `ex109_out_shape` — a Python tuple containing the forward output shape.
# Define `ex109_dW` — the projection-weight gradient.
# Define `ex109_dflat` — the flattened embedding gradient.
# Define `ex109_demb` — the restored embedding-grid gradient.
# Define `ex109_dtable` — the accumulated table gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex109_out_shape", "ex109", "out_shape")
_check_tensor("ex109_dW", "ex109", "dW")
_check_tensor("ex109_dflat", "ex109", "dflat")
_check_tensor("ex109_demb", "ex109", "demb")
_check_tensor("ex109_dtable", "ex109", "dtable")


### Exercise 110 — Paired target selection backward

**Purpose:** Learn that selecting one target candidate per example creates one nonzero direct gradient in each row.

**Inputs:** `log_values` is `(3, 5)` and `targets` is `(3,)`, one candidate index per example.

**Forward operation:** `selected = log_values[range(3), targets]`.

**Derive/do:** Derive the full `(3, 5)` gradient `dlog_values` from a length-3 upstream gradient.

**Ingredients:** Start with zeros and place one upstream entry at each paired example-target coordinate.

**Required outputs:**

- `ex110_out_shape`: a Python tuple containing the forward output shape.
- `ex110_dlog_values`: the sparse `(3, 5)` gradient at paired target positions.

**Next concept:** Stable row maxima.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
log_values = torch.linspace(-3.0, -0.1, steps=15, dtype=DTYPE).reshape(3, 5).requires_grad_()
targets = torch.tensor([2, 0, 4], dtype=torch.long)
selected = log_values[range(3), targets]
dout = torch.tensor([1.0, -2.0, 0.5], dtype=DTYPE)
_capture("ex110", selected, {"dlog_values": log_values}, dout)
print("log_values", tuple(log_values.shape), "targets", tuple(targets.shape), "selected", tuple(selected.shape))


In [ ]:
# Exercise 110: derive manually; do not use autograd in this cell.
# Define `ex110_out_shape` — a Python tuple containing the forward output shape.
# Define `ex110_dlog_values` — the sparse `(3, 5)` gradient at paired target positions.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex110_out_shape", "ex110", "out_shape")
_check_tensor("ex110_dlog_values", "ex110", "dlog_values")


## 8. Stable softmax and mean negative log-likelihood

For each training example `i`, the model produces one logit for every candidate `j`. Stable softmax is

$$
p_{i,j}
=
\frac{\exp(z_{i,j}-m_i)}{\sum_k \exp(z_{i,k}-m_i)}
$$

Here, `z` is `logits`, `m_i` is the maximum logit in example row `i`, `k` ranges over all candidates, and `p` is `probs`. Subtracting the row maximum changes numerical scale but not probabilities.

For one example, negative log-likelihood is

$$
\ell_i
=
-\log p_{i,y_i}
$$

Here, `y_i` is the expected target candidate. For a batch of `B` examples, the notebook uses the average

$$
L
=
\frac{1}{B}\sum_{i=1}^{B}\ell_i
$$

Only the indexed target log-probability contributes directly to each example's loss. Other candidates still influence that probability through the shared softmax denominator.

The lecture's variable name `counts` does **not** mean observed character occurrences here. It is simply `norm_logits.exp()`: positive, unnormalized candidate weights. Likewise, `counts_sum_inv` is the reciprocal of one row total.


### Exercise 111 — Stable row maxima

**Purpose:** Learn to find one maximum per example so exponentiation can use safer numbers.

**Inputs:** `logits` has shape `(3, 5)`: three examples and five candidates.

**Forward operation:** Compute `logits.max(dim=1, keepdim=True).values` manually in your answer cell.

**Derive/do:** Find the maximum of each row while preserving the candidate axis as size one.

**Ingredients:** `max(dim=1, keepdim=True).values`.

**Required outputs:**

- `ex111_out_shape`: a Python tuple containing the output shape.
- `ex111_out`: the `(3, 1)` row maxima.

**Next concept:** Subtract an independent row maximum.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
logits = torch.tensor([[1.0, 3.0, -2.0, 0.5, 2.0], [4.0, -1.0, 2.0, 0.0, 1.0], [-2.0, 0.5, 3.5, 1.0, 2.0]], dtype=DTYPE)
_store_forward("ex111", logits.max(dim=1, keepdim=True).values)
# Inspect only the supplied input shapes; compute the requested output yourself.
print("Input fixture ready.")


In [ ]:
# Exercise 111: derive manually; do not use autograd in this cell.
# Define `ex111_out_shape` — a Python tuple containing the output shape.
# Define `ex111_out` — the `(3, 1)` row maxima.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex111_out_shape", "ex111", "out_shape")
_check_tensor("ex111_out", "ex111", "out")


### Exercise 112 — Subtract an independent row maximum

**Purpose:** Learn how subtracting one row value from every candidate sends a summed gradient back to that row value.

**Inputs:** `logits` is `(3, 5)`, independent `row_max` is `(3, 1)`, and `dout` is `(3, 5)`.

**Forward operation:** `norm = logits - row_max`.

**Derive/do:** Derive `dlogits` and `drow_max`.

**Ingredients:** The maximum tensor is reused across five candidates in its row.

**Required outputs:**

- `ex112_out_shape`: a Python tuple containing the forward output shape.
- `ex112_dlogits`: the direct gradient with respect to `logits`.
- `ex112_drow_max`: the `(3, 1)` unbroadcast subtraction gradient.

**Next concept:** Exponentiate normalized logits.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
logits = torch.linspace(-2.0, 2.2, steps=15, dtype=DTYPE).reshape(3, 5).requires_grad_()
row_max = torch.tensor([[2.2], [1.1], [3.4]], dtype=DTYPE, requires_grad=True)
norm = logits - row_max
dout = torch.linspace(1.0, -1.4, steps=15, dtype=DTYPE).reshape(3, 5)
_capture("ex112", norm, {"dlogits": logits, "drow_max": row_max}, dout)
print("logits", tuple(logits.shape), "row_max", tuple(row_max.shape), "norm", tuple(norm.shape))


In [ ]:
# Exercise 112: derive manually; do not use autograd in this cell.
# Define `ex112_out_shape` — a Python tuple containing the forward output shape.
# Define `ex112_dlogits` — the direct gradient with respect to `logits`.
# Define `ex112_drow_max` — the `(3, 1)` unbroadcast subtraction gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex112_out_shape", "ex112", "out_shape")
_check_tensor("ex112_dlogits", "ex112", "dlogits")
_check_tensor("ex112_drow_max", "ex112", "drow_max")


### Exercise 113 — Exponentiate normalized logits

**Purpose:** Learn to pass gradients backward through the exponentiation that creates positive candidate weights.

**Inputs:** `norm_logits`, `counts`, and `dout` have shape `(3, 5)`.

**Forward operation:** `counts = norm_logits.exp()`.

**Derive/do:** Derive `dnorm_logits`.

**Ingredients:** Use the stored `counts` forward value as exp's local derivative.

**Required outputs:**

- `ex113_out_shape`: a Python tuple containing the forward output shape.
- `ex113_dnorm_logits`: the gradient with respect to normalized logits.

**Next concept:** Sum candidate weights per row.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
norm_logits = torch.linspace(-4.0, 0.0, steps=15, dtype=DTYPE).reshape(3, 5).requires_grad_()
counts = norm_logits.exp()
dout = torch.linspace(1.0, -1.4, steps=15, dtype=DTYPE).reshape(3, 5)
_capture("ex113", counts, {"dnorm_logits": norm_logits}, dout)
print("norm_logits", tuple(norm_logits.shape), "counts", tuple(counts.shape))


In [ ]:
# Exercise 113: derive manually; do not use autograd in this cell.
# Define `ex113_out_shape` — a Python tuple containing the forward output shape.
# Define `ex113_dnorm_logits` — the gradient with respect to normalized logits.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex113_out_shape", "ex113", "out_shape")
_check_tensor("ex113_dnorm_logits", "ex113", "dnorm_logits")


### Exercise 114 — Sum candidate weights per row

**Purpose:** Learn that one row-total gradient is copied back to every candidate weight in that row.

**Inputs:** `counts` is `(3, 5)` and `counts_sum` is `(3, 1)`.

**Forward operation:** `counts_sum = counts.sum(dim=1, keepdim=True)`.

**Derive/do:** Derive `dcounts` from `(3, 1)` upstream gradients.

**Ingredients:** Each row total depends once on all five candidate weights in that row.

**Required outputs:**

- `ex114_out_shape`: a Python tuple containing the forward output shape.
- `ex114_dcounts`: the `(3, 5)` gradient after reversing the row sums.

**Next concept:** Reciprocal row totals.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
counts = torch.linspace(0.1, 1.5, steps=15, dtype=DTYPE).reshape(3, 5).requires_grad_()
counts_sum = counts.sum(dim=1, keepdim=True)
dout = torch.tensor([[1.0], [-2.0], [0.5]], dtype=DTYPE)
_capture("ex114", counts_sum, {"dcounts": counts}, dout)
print("counts", tuple(counts.shape), "counts_sum", tuple(counts_sum.shape))


In [ ]:
# Exercise 114: derive manually; do not use autograd in this cell.
# Define `ex114_out_shape` — a Python tuple containing the forward output shape.
# Define `ex114_dcounts` — the `(3, 5)` gradient after reversing the row sums.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex114_out_shape", "ex114", "out_shape")
_check_tensor("ex114_dcounts", "ex114", "dcounts")


### Exercise 115 — Reciprocal row totals

**Purpose:** Learn to pass gradients backward through the reciprocal of each row total.

**Inputs:** Positive `counts_sum` and `counts_sum_inv` have shape `(3, 1)`.

**Forward operation:** `counts_sum_inv = counts_sum**-1`.

**Derive/do:** Derive `dcounts_sum`.

**Ingredients:** Use the fixed-power rule with exponent `-1`.

**Required outputs:**

- `ex115_out_shape`: a Python tuple containing the forward output shape.
- `ex115_dcounts_sum`: the `(3, 1)` gradient with respect to row totals.

**Next concept:** Normalize weights by reciprocal totals.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
counts_sum = torch.tensor([[2.0], [4.0], [0.5]], dtype=DTYPE, requires_grad=True)
counts_sum_inv = counts_sum**-1
dout = torch.tensor([[1.0], [-2.0], [0.5]], dtype=DTYPE)
_capture("ex115", counts_sum_inv, {"dcounts_sum": counts_sum}, dout)
print("counts_sum", tuple(counts_sum.shape), "counts_sum_inv", tuple(counts_sum_inv.shape))


In [ ]:
# Exercise 115: derive manually; do not use autograd in this cell.
# Define `ex115_out_shape` — a Python tuple containing the forward output shape.
# Define `ex115_dcounts_sum` — the `(3, 1)` gradient with respect to row totals.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex115_out_shape", "ex115", "out_shape")
_check_tensor("ex115_dcounts_sum", "ex115", "dcounts_sum")


### Exercise 116 — Normalize weights by reciprocal totals

**Purpose:** Learn how probability normalization sends gradients to both candidate weights and their shared row denominator.

**Inputs:** `counts` is `(3, 5)`, `counts_sum_inv` is `(3, 1)`, and `probs` is `(3, 5)`.

**Forward operation:** `probs = counts * counts_sum_inv`.

**Derive/do:** Derive the direct `dcounts` and accumulated `dcounts_sum_inv`.

**Ingredients:** Product backward plus summation over five candidates for the broadcast inverse total.

**Required outputs:**

- `ex116_out_shape`: a Python tuple containing the forward output shape.
- `ex116_dcounts`: the direct numerator gradient.
- `ex116_dcounts_sum_inv`: the `(3, 1)` gradient accumulated across candidates.

**Next concept:** Log probabilities.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
counts = torch.linspace(0.1, 1.5, steps=15, dtype=DTYPE).reshape(3, 5).requires_grad_()
counts_sum_inv = torch.tensor([[0.25], [0.5], [0.125]], dtype=DTYPE, requires_grad=True)
probs = counts * counts_sum_inv
dout = torch.linspace(1.0, -1.4, steps=15, dtype=DTYPE).reshape(3, 5)
_capture("ex116", probs, {"dcounts": counts, "dcounts_sum_inv": counts_sum_inv}, dout)
print("counts", tuple(counts.shape), "counts_sum_inv", tuple(counts_sum_inv.shape), "probs", tuple(probs.shape))


In [ ]:
# Exercise 116: derive manually; do not use autograd in this cell.
# Define `ex116_out_shape` — a Python tuple containing the forward output shape.
# Define `ex116_dcounts` — the direct numerator gradient.
# Define `ex116_dcounts_sum_inv` — the `(3, 1)` gradient accumulated across candidates.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex116_out_shape", "ex116", "out_shape")
_check_tensor("ex116_dcounts", "ex116", "dcounts")
_check_tensor("ex116_dcounts_sum_inv", "ex116", "dcounts_sum_inv")


### Exercise 117 — Log probabilities

**Purpose:** Learn to pass gradients from log-probabilities back into probabilities.

**Inputs:** Positive `probs`, `logprobs`, and `dout` have shape `(3, 5)`.

**Forward operation:** `logprobs = probs.log()`.

**Derive/do:** Derive `dprobs`.

**Ingredients:** The reciprocal local slope of log and the supplied upstream tensor.

**Required outputs:**

- `ex117_out_shape`: a Python tuple containing the forward output shape.
- `ex117_dprobs`: the gradient with respect to probabilities.

**Next concept:** Select one target from one example.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
probs = torch.tensor([[0.1, 0.2, 0.3, 0.15, 0.25], [0.4, 0.1, 0.2, 0.2, 0.1], [0.05, 0.15, 0.5, 0.2, 0.1]], dtype=DTYPE, requires_grad=True)
logprobs = probs.log()
dout = torch.linspace(1.0, -1.4, steps=15, dtype=DTYPE).reshape(3, 5)
_capture("ex117", logprobs, {"dprobs": probs}, dout)
print("probs", tuple(probs.shape), "logprobs", tuple(logprobs.shape))


In [ ]:
# Exercise 117: derive manually; do not use autograd in this cell.
# Define `ex117_out_shape` — a Python tuple containing the forward output shape.
# Define `ex117_dprobs` — the gradient with respect to probabilities.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex117_out_shape", "ex117", "out_shape")
_check_tensor("ex117_dprobs", "ex117", "dprobs")


### Exercise 118 — Select one target from one example

**Purpose:** Learn that selecting one target from one example gives a direct gradient only to that target position.

**Inputs:** `logprobs` has shape `(5,)` and the expected target is candidate 3.

**Forward operation:** `selected = logprobs[target]`.

**Derive/do:** Derive the full length-5 `dlogprobs`.

**Ingredients:** Only the indexed candidate has a direct path to the selected scalar.

**Required outputs:**

- `ex118_out_shape`: a Python tuple containing the forward output shape.
- `ex118_dlogprobs`: the sparse length-5 gradient.

**Next concept:** Select one target per batch row.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
logprobs = torch.tensor([-2.0, -1.5, -3.0, -0.2, -2.5], dtype=DTYPE, requires_grad=True)
target = 3
selected = logprobs[target]
dout = torch.tensor(-1.25, dtype=DTYPE)
_capture("ex118", selected, {"dlogprobs": logprobs}, dout)
print("logprobs", tuple(logprobs.shape), "selected", tuple(selected.shape))


In [ ]:
# Exercise 118: derive manually; do not use autograd in this cell.
# Define `ex118_out_shape` — a Python tuple containing the forward output shape.
# Define `ex118_dlogprobs` — the sparse length-5 gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex118_out_shape", "ex118", "out_shape")
_check_tensor("ex118_dlogprobs", "ex118", "dlogprobs")


### Exercise 119 — Select one target per batch row

**Purpose:** Learn to place one direct target gradient in every training-example row.

**Inputs:** `logprobs` is `(4, 5)` and `targets` is `(4,)`.

**Forward operation:** `selected = logprobs[range(4), targets]`.

**Derive/do:** Derive the full `(4, 5)` `dlogprobs` from a length-4 upstream tensor.

**Ingredients:** Place one upstream value at the target coordinate in each training-example row.

**Required outputs:**

- `ex119_out_shape`: a Python tuple containing the forward output shape.
- `ex119_dlogprobs`: the sparse `(4, 5)` target gradient.

**Next concept:** Mean NLL seed.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
logprobs = torch.linspace(-3.0, -0.1, steps=20, dtype=DTYPE).reshape(4, 5).requires_grad_()
targets = torch.tensor([3, 0, 4, 2], dtype=torch.long)
selected = logprobs[range(4), targets]
dout = torch.tensor([1.0, -2.0, 0.5, 3.0], dtype=DTYPE)
_capture("ex119", selected, {"dlogprobs": logprobs}, dout)
print("logprobs", tuple(logprobs.shape), "targets", tuple(targets.shape), "selected", tuple(selected.shape))


In [ ]:
# Exercise 119: derive manually; do not use autograd in this cell.
# Define `ex119_out_shape` — a Python tuple containing the forward output shape.
# Define `ex119_dlogprobs` — the sparse `(4, 5)` target gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex119_out_shape", "ex119", "out_shape")
_check_tensor("ex119_dlogprobs", "ex119", "dlogprobs")


### Exercise 120 — Mean NLL seed

**Purpose:** Learn why mean negative log-likelihood starts with `-1 / batch_size` at each selected target and zero elsewhere.

**Inputs:** Four training examples, five candidates each, and one expected target per example.

**Forward operation:** The supplied fixture computes `sm_loss = -sm_logprobs[range(sm_n), sm_targets].mean()`.

**Derive/do:** Derive `sm_dlogprobs` manually.

**Ingredients:** Start with zeros shaped like `sm_logprobs`; combine the minus sign and four-example mean only at paired target coordinates.

**Required outputs:**

- `sm_dlogprobs`: the `(4, 5)` gradient with respect to `sm_logprobs`.

**Next concept:** Backward through log.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
# Supplied mini-batch: four examples and five candidate classes per example.
sm_logits = torch.tensor([
    [1.2, -0.7, 2.1, 0.3, -1.4],
    [-0.2, 1.7, 0.4, 2.5, -0.9],
    [2.2, 0.1, -1.1, 0.8, 1.5],
    [0.6, 2.3, -0.4, 1.1, -1.7],
], dtype=DTYPE, requires_grad=True)
sm_targets = torch.tensor([2, 3, 0, 1], dtype=torch.long)
sm_n = sm_logits.shape[0]

# Expanded stable softmax and mean NLL; this scores all four supplied examples.
sm_logit_maxes = sm_logits.max(dim=1, keepdim=True).values
sm_norm_logits = sm_logits - sm_logit_maxes
sm_counts = sm_norm_logits.exp()
sm_counts_sum = sm_counts.sum(dim=1, keepdim=True)
sm_counts_sum_inv = sm_counts_sum**-1
sm_probs = sm_counts * sm_counts_sum_inv
sm_logprobs = sm_probs.log()
sm_loss = -sm_logprobs[range(sm_n), sm_targets].mean()

_capture(
    "softmax_capstone",
    sm_loss,
    {
        "dlogprobs": sm_logprobs,
        "dprobs": sm_probs,
        "dcounts_sum_inv": sm_counts_sum_inv,
        "dcounts_sum": sm_counts_sum,
        "dcounts": sm_counts,
        "dnorm_logits": sm_norm_logits,
        "dlogit_maxes": sm_logit_maxes,
        "dlogits": sm_logits,
    },
)
print("Softmax capstone ready:", tuple(sm_logits.shape), "-> loss", tuple(sm_loss.shape))


In [ ]:
# Exercise 120: derive manually; do not use autograd in this cell.
# Define `sm_dlogprobs` — the `(4, 5)` gradient with respect to `sm_logprobs`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_tensor("sm_dlogprobs", "softmax_capstone", "dlogprobs")


### Exercise 121 — Backward through log

**Purpose:** Learn to pass the sparse target gradient backward through the logarithm.

**Inputs:** `sm_probs` and `sm_dlogprobs` are `(4, 5)`.

**Forward operation:** `sm_logprobs = sm_probs.log()`.

**Derive/do:** Using your previous `sm_dlogprobs`, derive `sm_dprobs`.

**Ingredients:** Natural-log local derivative; this operation is elementwise.

**Required outputs:**

- `sm_dprobs`: the `(4, 5)` gradient with respect to `sm_probs`.

**Next concept:** Backward into the shared inverse row totals.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
# Continue from the supplied softmax capstone fixture above.
assert "softmax_capstone" in _REFS


In [ ]:
# Exercise 121: derive manually; do not use autograd in this cell.
# Define `sm_dprobs` — the `(4, 5)` gradient with respect to `sm_probs`.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_tensor("sm_dprobs", "softmax_capstone", "dprobs")


### Exercise 122 — Backward into the shared inverse row totals

**Purpose:** Learn why one shared inverse row total receives contributions from all five candidates in its example.

**Inputs:** `sm_counts` and `sm_dprobs` are `(4, 5)`; `sm_counts_sum_inv` is `(4, 1)`.

**Forward operation:** `sm_probs = sm_counts * sm_counts_sum_inv`.

**Derive/do:** Derive `sm_dcounts_sum_inv`.

**Ingredients:** Use product backward, then sum candidate contributions along dimension 1 while preserving it.

**Required outputs:**

- `sm_dcounts_sum_inv`: the `(4, 1)` gradient with respect to inverse row totals.

**Next concept:** Backward through reciprocal row totals.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
# Continue from the supplied softmax capstone fixture and prior manual gradients.
assert "softmax_capstone" in _REFS


In [ ]:
# Exercise 122: derive manually; do not use autograd in this cell.
# Define `sm_dcounts_sum_inv` — the `(4, 1)` gradient with respect to inverse row totals.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_tensor("sm_dcounts_sum_inv", "softmax_capstone", "dcounts_sum_inv")


### Exercise 123 — Backward through reciprocal row totals

**Purpose:** Learn to pass each inverse-total gradient backward to the original row total.

**Inputs:** `sm_counts_sum` and its inverse have shape `(4, 1)`.

**Forward operation:** `sm_counts_sum_inv = sm_counts_sum**-1`.

**Derive/do:** Using `sm_dcounts_sum_inv`, derive `sm_dcounts_sum`.

**Ingredients:** The fixed-power rule with exponent `-1`.

**Required outputs:**

- `sm_dcounts_sum`: the `(4, 1)` gradient with respect to row totals.

**Next concept:** Combine both paths into counts.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
# Continue from the supplied softmax capstone fixture and prior manual gradients.
assert "softmax_capstone" in _REFS


In [ ]:
# Exercise 123: derive manually; do not use autograd in this cell.
# Define `sm_dcounts_sum` — the `(4, 1)` gradient with respect to row totals.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_tensor("sm_dcounts_sum", "softmax_capstone", "dcounts_sum")


### Exercise 124 — Combine both paths into counts

**Purpose:** Learn that exponentiated logits receive one gradient as numerators and another through the shared denominator.

**Inputs:** `sm_counts` is `(4, 5)` and `sm_counts_sum` is `(4, 1)`.

**Forward operation:** `sm_probs` uses `sm_counts` directly, while `sm_counts_sum` also depends on every `sm_counts` entry.

**Derive/do:** Derive total `sm_dcounts`.

**Ingredients:** Compute the direct multiplication path from `sm_dprobs`; broadcast the row-sum path from `sm_dcounts_sum`; add them.

**Required outputs:**

- `sm_dcounts`: the total `(4, 5)` gradient from both count paths.

**Next concept:** Backward through exponentiation.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
# Continue from the supplied softmax capstone fixture and prior manual gradients.
assert "softmax_capstone" in _REFS


In [ ]:
# Exercise 124: derive manually; do not use autograd in this cell.
# Define `sm_dcounts` — the total `(4, 5)` gradient from both count paths.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_tensor("sm_dcounts", "softmax_capstone", "dcounts")


### Exercise 125 — Backward through exponentiation

**Purpose:** Learn to pass the combined candidate-weight gradients backward through exponentiation.

**Inputs:** `sm_counts` and `sm_dcounts` are `(4, 5)`.

**Forward operation:** `sm_counts = sm_norm_logits.exp()`.

**Derive/do:** Derive `sm_dnorm_logits`.

**Ingredients:** Use the exponentiated forward values as the local derivative.

**Required outputs:**

- `sm_dnorm_logits`: the `(4, 5)` gradient with respect to normalized logits.

**Next concept:** Backward through maximum shifting.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
# Continue from the supplied softmax capstone fixture and prior manual gradients.
assert "softmax_capstone" in _REFS


In [ ]:
# Exercise 125: derive manually; do not use autograd in this cell.
# Define `sm_dnorm_logits` — the `(4, 5)` gradient with respect to normalized logits.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_tensor("sm_dnorm_logits", "softmax_capstone", "dnorm_logits")


### Exercise 126 — Backward through maximum shifting

**Purpose:** Learn to combine the direct logit path with the path through each row's maximum.

**Inputs:** `sm_logits` and `sm_dnorm_logits` are `(4, 5)`; `sm_logit_maxes` is `(4, 1)`.

**Forward operation:** `sm_logit_maxes = sm_logits.max(dim=1, keepdim=True).values` and `sm_norm_logits = sm_logits - sm_logit_maxes`.

**Derive/do:** Derive `sm_dlogit_maxes` and total `sm_dlogits`.

**Ingredients:** First unbroadcast the subtraction's maximum path. Then route each row's maximum gradient to that row's unique argmax and add the direct logits path.

**Required outputs:**

- `sm_dlogit_maxes`: the `(4, 1)` gradient of row maxima.
- `sm_dlogits`: the final `(4, 5)` gradient with respect to logits.

**Next concept:** BatchNorm column means.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
# Continue from the supplied softmax capstone fixture and prior manual gradients.
assert "softmax_capstone" in _REFS


In [ ]:
# Exercise 126: derive manually; do not use autograd in this cell.
# Define `sm_dlogit_maxes` — the `(4, 1)` gradient of row maxima.
# Define `sm_dlogits` — the final `(4, 5)` gradient with respect to logits.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_tensor("sm_dlogit_maxes", "softmax_capstone", "dlogit_maxes")
_check_tensor("sm_dlogits", "softmax_capstone", "dlogits")


## 9. BatchNorm as small tensor operations

BatchNorm treats rows as training examples and columns as hidden neurons. In this section, `B` is the batch size and `H` is the number of hidden neurons. Each neuron gets one mean and one variance computed across the `B` examples.

The forward relationships are

$$
\mu_j
=
\frac{1}{B}\sum_i a_{i,j}
$$

$$
d_{i,j}
=
a_{i,j}-\mu_j
$$

$$
v_j
=
\frac{1}{B-1}\sum_i d_{i,j}^2
$$

$$
r_j
=
(v_j+\varepsilon)^{-1/2}
$$

$$
\widehat{a}_{i,j}
=
d_{i,j}r_j
$$

$$
\widetilde{a}_{i,j}
=
\gamma_j\widehat{a}_{i,j}+\beta_j
$$

Here, `a` is `hprebn`, `mu` is `bnmeani`, `d` is `bndiff`, `v` is `bnvar`, `r` is `bnvar_inv`, normalized values are `bnraw`, scale is `bngain`, shift is `bnbias`, and the final result is `hpreact`. Despite its historical name, `bnvar_inv` is the inverse **standard deviation**, not the inverse variance. The variance is a sample variance, so it divides by `B - 1`, not `B`.


### Exercise 127 — BatchNorm column means

**Purpose:** Learn that BatchNorm computes one mean per hidden-neuron column across all examples.

**Inputs:** `hprebn` has shape `(4, 3)` and `bnmeani` has shape `(1, 3)`.

**Forward operation:** `bnmeani = hprebn.mean(dim=0, keepdim=True)`.

**Derive/do:** Derive `dhprebn` from a `(1, 3)` upstream gradient.

**Ingredients:** Each neuron mean averages four example rows.

**Required outputs:**

- `ex127_out_shape`: a Python tuple containing the forward output shape.
- `ex127_dhprebn`: the `(4, 3)` gradient after reversing the column means.

**Next concept:** BatchNorm centering with independent means.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
hprebn = torch.linspace(-2.0, 3.5, steps=12, dtype=DTYPE).reshape(4, 3).requires_grad_()
bnmeani = hprebn.mean(dim=0, keepdim=True)
dout = torch.tensor([[1.0, -2.0, 0.5]], dtype=DTYPE)
_capture("ex127", bnmeani, {"dhprebn": hprebn}, dout)
print("hprebn", tuple(hprebn.shape), "bnmeani", tuple(bnmeani.shape))


In [ ]:
# Exercise 127: derive manually; do not use autograd in this cell.
# Define `ex127_out_shape` — a Python tuple containing the forward output shape.
# Define `ex127_dhprebn` — the `(4, 3)` gradient after reversing the column means.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex127_out_shape", "ex127", "out_shape")
_check_tensor("ex127_dhprebn", "ex127", "dhprebn")


### Exercise 128 — BatchNorm centering with independent means

**Purpose:** Learn how subtracting one hidden-neuron mean from every example sends gradients to both values and the mean.

**Inputs:** `hprebn` is `(4, 3)`, independent `bnmeani` is `(1, 3)`, and `bndiff` is `(4, 3)`.

**Forward operation:** `bndiff = hprebn - bnmeani`.

**Derive/do:** Derive the direct `dhprebn` and `dbnmeani`.

**Ingredients:** Subtraction sends a direct path to `hprebn`; sum the negative mean path over examples.

**Required outputs:**

- `ex128_out_shape`: a Python tuple containing the forward output shape.
- `ex128_dhprebn`: the direct centered-value gradient.
- `ex128_dbnmeani`: the `(1, 3)` unbroadcast mean gradient.

**Next concept:** Square centered deviations.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
hprebn = torch.linspace(-2.0, 3.5, steps=12, dtype=DTYPE).reshape(4, 3).requires_grad_()
bnmeani = torch.tensor([[0.5, -1.0, 2.0]], dtype=DTYPE, requires_grad=True)
bndiff = hprebn - bnmeani
dout = torch.linspace(1.0, -1.2, steps=12, dtype=DTYPE).reshape(4, 3)
_capture("ex128", bndiff, {"dhprebn": hprebn, "dbnmeani": bnmeani}, dout)
print("hprebn", tuple(hprebn.shape), "bnmeani", tuple(bnmeani.shape), "bndiff", tuple(bndiff.shape))


In [ ]:
# Exercise 128: derive manually; do not use autograd in this cell.
# Define `ex128_out_shape` — a Python tuple containing the forward output shape.
# Define `ex128_dhprebn` — the direct centered-value gradient.
# Define `ex128_dbnmeani` — the `(1, 3)` unbroadcast mean gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex128_out_shape", "ex128", "out_shape")
_check_tensor("ex128_dhprebn", "ex128", "dhprebn")
_check_tensor("ex128_dbnmeani", "ex128", "dbnmeani")


### Exercise 129 — Square centered deviations

**Purpose:** Learn to pass variance-related gradients backward through squared deviations.

**Inputs:** `bndiff` and `bndiff2` have shape `(4, 3)`.

**Forward operation:** `bndiff2 = bndiff**2`.

**Derive/do:** Derive `dbndiff`.

**Ingredients:** The square power rule with an arbitrary upstream tensor.

**Required outputs:**

- `ex129_out_shape`: a Python tuple containing the forward output shape.
- `ex129_dbndiff`: the gradient with respect to centered deviations.

**Next concept:** Unbiased variance reduction.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
bndiff = torch.linspace(-2.0, 3.5, steps=12, dtype=DTYPE).reshape(4, 3).requires_grad_()
bndiff2 = bndiff**2
dout = torch.linspace(1.0, -1.2, steps=12, dtype=DTYPE).reshape(4, 3)
_capture("ex129", bndiff2, {"dbndiff": bndiff}, dout)
print("bndiff", tuple(bndiff.shape), "bndiff2", tuple(bndiff2.shape))


In [24]:
# Exercise 129: derive manually; do not use autograd in this cell.
# Define `ex129_out_shape` — a Python tuple containing the forward output shape.
# Define `ex129_dbndiff` — the gradient with respect to centered deviations.
# Write your tensor operations below these comments, then run the test cell.


In [25]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex129_out_shape", "ex129", "out_shape")
_check_tensor("ex129_dbndiff", "ex129", "dbndiff")


AssertionError: Define `ex129_out_shape` in the answer cell first.

### Exercise 130 — Unbiased variance reduction

**Purpose:** Learn that sample variance spreads one gradient per hidden neuron across examples and divides by `B - 1`.

**Inputs:** `bndiff2` is `(4, 3)`, `B = 4`, and `bnvar` is `(1, 3)`.

**Forward operation:** `bnvar = bndiff2.sum(dim=0, keepdim=True) / (B - 1)`.

**Derive/do:** Derive `dbndiff2`.

**Ingredients:** Broadcast each neuron variance gradient over four rows and include division by 3.

**Required outputs:**

- `ex130_out_shape`: a Python tuple containing the forward output shape.
- `ex130_dbndiff2`: the `(4, 3)` gradient after reversing sample variance.

**Next concept:** Stabilized inverse standard deviation.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
bndiff2 = torch.linspace(0.1, 2.4, steps=12, dtype=DTYPE).reshape(4, 3).requires_grad_()
B = bndiff2.shape[0]
bnvar = bndiff2.sum(dim=0, keepdim=True) / (B - 1)
dout = torch.tensor([[1.0, -2.0, 0.5]], dtype=DTYPE)
_capture("ex130", bnvar, {"dbndiff2": bndiff2}, dout)
print("bndiff2", tuple(bndiff2.shape), "bnvar", tuple(bnvar.shape), "B", B)


In [ ]:
# Exercise 130: derive manually; do not use autograd in this cell.
# Define `ex130_out_shape` — a Python tuple containing the forward output shape.
# Define `ex130_dbndiff2` — the `(4, 3)` gradient after reversing sample variance.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex130_out_shape", "ex130", "out_shape")
_check_tensor("ex130_dbndiff2", "ex130", "dbndiff2")


### Exercise 131 — Stabilized inverse standard deviation

**Purpose:** Learn to pass gradients backward through epsilon addition and inverse square root.

**Inputs:** `bnvar` and `bnvar_inv` have shape `(1, 3)`.

**Forward operation:** `bnvar_inv = (bnvar + eps)**-0.5`.

**Derive/do:** Derive `dbnvar`.

**Ingredients:** Epsilon is constant; apply the fixed-power rule to the stabilized variance.

**Required outputs:**

- `ex131_out_shape`: a Python tuple containing the forward output shape.
- `ex131_dbnvar`: the `(1, 3)` variance gradient.

**Next concept:** Normalize centered values.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
bnvar = torch.tensor([[0.25, 1.0, 4.0]], dtype=DTYPE, requires_grad=True)
eps = 1e-5
bnvar_inv = (bnvar + eps) ** -0.5
dout = torch.tensor([[1.0, -2.0, 0.5]], dtype=DTYPE)
_capture("ex131", bnvar_inv, {"dbnvar": bnvar}, dout)
print("bnvar", tuple(bnvar.shape), "bnvar_inv", tuple(bnvar_inv.shape))


In [ ]:
# Exercise 131: derive manually; do not use autograd in this cell.
# Define `ex131_out_shape` — a Python tuple containing the forward output shape.
# Define `ex131_dbnvar` — the `(1, 3)` variance gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex131_out_shape", "ex131", "out_shape")
_check_tensor("ex131_dbnvar", "ex131", "dbnvar")


### Exercise 132 — Normalize centered values

**Purpose:** Learn how one inverse standard deviation reused across examples receives all of their contributions.

**Inputs:** `bndiff` is `(4, 3)`, `bnvar_inv` is `(1, 3)`, and `bnraw` is `(4, 3)`.

**Forward operation:** `bnraw = bndiff * bnvar_inv`.

**Derive/do:** Derive direct `dbndiff` and accumulated `dbnvar_inv`.

**Ingredients:** Product backward; sum inverse-standard-deviation contributions over four examples.

**Required outputs:**

- `ex132_out_shape`: a Python tuple containing the forward output shape.
- `ex132_dbndiff`: the direct centered-value gradient.
- `ex132_dbnvar_inv`: the `(1, 3)` gradient accumulated over examples.

**Next concept:** BatchNorm learned scale and shift.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
bndiff = torch.linspace(-2.0, 3.5, steps=12, dtype=DTYPE).reshape(4, 3).requires_grad_()
bnvar_inv = torch.tensor([[2.0, 1.0, 0.5]], dtype=DTYPE, requires_grad=True)
bnraw = bndiff * bnvar_inv
dout = torch.linspace(1.0, -1.2, steps=12, dtype=DTYPE).reshape(4, 3)
_capture("ex132", bnraw, {"dbndiff": bndiff, "dbnvar_inv": bnvar_inv}, dout)
print("bndiff", tuple(bndiff.shape), "bnvar_inv", tuple(bnvar_inv.shape), "bnraw", tuple(bnraw.shape))


In [ ]:
# Exercise 132: derive manually; do not use autograd in this cell.
# Define `ex132_out_shape` — a Python tuple containing the forward output shape.
# Define `ex132_dbndiff` — the direct centered-value gradient.
# Define `ex132_dbnvar_inv` — the `(1, 3)` gradient accumulated over examples.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex132_out_shape", "ex132", "out_shape")
_check_tensor("ex132_dbndiff", "ex132", "dbndiff")
_check_tensor("ex132_dbnvar_inv", "ex132", "dbnvar_inv")


### Exercise 133 — BatchNorm learned scale and shift

**Purpose:** Learn how BatchNorm's learned gain and bias collect gradients from every example.

**Inputs:** `bnraw` is `(4, 3)`; `bngain` and `bnbias` are `(1, 3)`.

**Forward operation:** `hpreact = bngain * bnraw + bnbias`.

**Derive/do:** Derive `dbnraw`, `dbngain`, and `dbnbias`.

**Ingredients:** Elementwise affine backward plus accumulation over four examples for gain and bias.

**Required outputs:**

- `ex133_out_shape`: a Python tuple containing the forward output shape.
- `ex133_dbnraw`: the gradient entering normalized values.
- `ex133_dbngain`: the `(1, 3)` learned-gain gradient.
- `ex133_dbnbias`: the `(1, 3)` learned-bias gradient.

**Next concept:** Full BatchNorm: affine backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
bnraw = torch.linspace(-1.5, 1.5, steps=12, dtype=DTYPE).reshape(4, 3).requires_grad_()
bngain = torch.tensor([[1.2, 0.8, -1.1]], dtype=DTYPE, requires_grad=True)
bnbias = torch.tensor([[0.1, -0.2, 0.3]], dtype=DTYPE, requires_grad=True)
hpreact = bngain * bnraw + bnbias
dout = torch.linspace(1.0, -1.2, steps=12, dtype=DTYPE).reshape(4, 3)
_capture("ex133", hpreact, {"dbnraw": bnraw, "dbngain": bngain, "dbnbias": bnbias}, dout)
print("bnraw", tuple(bnraw.shape), "gain", tuple(bngain.shape), "hpreact", tuple(hpreact.shape))


In [ ]:
# Exercise 133: derive manually; do not use autograd in this cell.
# Define `ex133_out_shape` — a Python tuple containing the forward output shape.
# Define `ex133_dbnraw` — the gradient entering normalized values.
# Define `ex133_dbngain` — the `(1, 3)` learned-gain gradient.
# Define `ex133_dbnbias` — the `(1, 3)` learned-bias gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("ex133_out_shape", "ex133", "out_shape")
_check_tensor("ex133_dbnraw", "ex133", "dbnraw")
_check_tensor("ex133_dbngain", "ex133", "dbngain")
_check_tensor("ex133_dbnbias", "ex133", "dbnbias")


### Exercise 134 — Full BatchNorm: affine backward

**Purpose:** Learn to begin the complete BatchNorm backward pass at its final learned scale-and-shift operation.

**Inputs:** `bncap_hpreact` and `bncap_dhpreact` are `(4, 3)`; gain and bias are `(1, 3)`.

**Forward operation:** The full fixture ends with `bncap_hpreact = gain * bnraw + bias`.

**Derive/do:** Derive `bncap_dbnraw`, `bncap_dbngain`, and `bncap_dbnbias`.

**Ingredients:** Reverse the learned affine step and unbroadcast parameter gradients over examples.

**Required outputs:**

- `bncap_dbnraw`: the `(4, 3)` gradient entering normalized values.
- `bncap_dbngain`: the `(1, 3)` gain gradient.
- `bncap_dbnbias`: the `(1, 3)` bias gradient.

**Next concept:** Full BatchNorm: normalization product.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
# Supplied full BatchNorm graph: four examples and three hidden neurons.
bncap_hprebn = torch.tensor([
    [-1.0, 0.5, 2.0],
    [0.0, -1.5, 1.0],
    [2.0, 1.5, -0.5],
    [1.0, 0.0, 3.0],
], dtype=DTYPE, requires_grad=True)
bncap_n = bncap_hprebn.shape[0]
bncap_bngain = torch.tensor([[1.2, 0.8, -1.1]], dtype=DTYPE, requires_grad=True)
bncap_bnbias = torch.tensor([[0.1, -0.2, 0.3]], dtype=DTYPE, requires_grad=True)
bncap_eps = 1e-5

bncap_bnmeani = bncap_hprebn.mean(dim=0, keepdim=True)
bncap_bndiff = bncap_hprebn - bncap_bnmeani
bncap_bndiff2 = bncap_bndiff**2
bncap_bnvar = bncap_bndiff2.sum(dim=0, keepdim=True) / (bncap_n - 1)
bncap_bnvar_inv = (bncap_bnvar + bncap_eps) ** -0.5
bncap_bnraw = bncap_bndiff * bncap_bnvar_inv
bncap_hpreact = bncap_bngain * bncap_bnraw + bncap_bnbias

# This supplied tensor is the gradient arriving from the next graph operation.
bncap_dhpreact = torch.linspace(1.0, -1.2, steps=12, dtype=DTYPE).reshape(4, 3)
bncap_objective = (bncap_hpreact * bncap_dhpreact).sum()
_capture(
    "batchnorm_capstone",
    bncap_objective,
    {
        "dbngain": bncap_bngain,
        "dbnbias": bncap_bnbias,
        "dbnraw": bncap_bnraw,
        "dbnvar_inv": bncap_bnvar_inv,
        "dbnvar": bncap_bnvar,
        "dbndiff2": bncap_bndiff2,
        "dbndiff": bncap_bndiff,
        "dbnmeani": bncap_bnmeani,
        "dhprebn": bncap_hprebn,
    },
    retain_graph=True,
)
print("BatchNorm capstone ready:", tuple(bncap_hprebn.shape), "->", tuple(bncap_hpreact.shape))
# Private staged references for paths that merge later.
_capture_product_path(
    "batchnorm_capstone",
    "dbndiff_direct",
    bncap_bndiff,
    bncap_bnvar_inv,
    _REFS["batchnorm_capstone"]["dbnraw"],
)
_capture_path(
    "batchnorm_capstone",
    "dbndiff_variance",
    bncap_bnvar,
    bncap_bndiff,
    _REFS["batchnorm_capstone"]["dbnvar"],
)


In [ ]:
# Exercise 134: derive manually; do not use autograd in this cell.
# Define `bncap_dbnraw` — the `(4, 3)` gradient entering normalized values.
# Define `bncap_dbngain` — the `(1, 3)` gain gradient.
# Define `bncap_dbnbias` — the `(1, 3)` bias gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_tensor("bncap_dbnraw", "batchnorm_capstone", "dbnraw")
_check_tensor("bncap_dbngain", "batchnorm_capstone", "dbngain")
_check_tensor("bncap_dbnbias", "batchnorm_capstone", "dbnbias")


### Exercise 135 — Full BatchNorm: normalization product

**Purpose:** Learn to split the normalized-value gradient between centered values and the shared inverse standard deviation.

**Inputs:** `bncap_bndiff` is `(4, 3)` and `bncap_bnvar_inv` is `(1, 3)`.

**Forward operation:** `bncap_bnraw = bncap_bndiff * bncap_bnvar_inv`.

**Derive/do:** Using `bncap_dbnraw`, derive direct `bncap_dbndiff_direct` and `bncap_dbnvar_inv`.

**Ingredients:** Product backward; sum the inverse-standard-deviation path over examples.

**Required outputs:**

- `bncap_dbndiff_direct`: the direct `(4, 3)` centered-value contribution.
- `bncap_dbnvar_inv`: the `(1, 3)` inverse-standard-deviation gradient.

**Next concept:** Full BatchNorm: inverse square root.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
# Continue from the BatchNorm capstone fixture above.
assert "batchnorm_capstone" in _REFS


In [ ]:
# Exercise 135: derive manually; do not use autograd in this cell.
# Define `bncap_dbndiff_direct` — the direct `(4, 3)` centered-value contribution.
# Define `bncap_dbnvar_inv` — the `(1, 3)` inverse-standard-deviation gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_tensor("bncap_dbndiff_direct", "batchnorm_capstone", "dbndiff_direct")
_check_tensor("bncap_dbnvar_inv", "batchnorm_capstone", "dbnvar_inv")


### Exercise 136 — Full BatchNorm: inverse square root

**Purpose:** Learn to pass the inverse-standard-deviation gradient back to variance.

**Inputs:** `bncap_bnvar` and `bncap_bnvar_inv` are `(1, 3)`.

**Forward operation:** `bncap_bnvar_inv = (bncap_bnvar + eps)**-0.5`.

**Derive/do:** Using `bncap_dbnvar_inv`, derive `bncap_dbnvar`.

**Ingredients:** The fixed-power rule with exponent `-0.5`; epsilon is constant.

**Required outputs:**

- `bncap_dbnvar`: the `(1, 3)` variance gradient.

**Next concept:** Full BatchNorm: variance reduction.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
# Continue from the BatchNorm capstone fixture and prior gradients.
assert "batchnorm_capstone" in _REFS


In [ ]:
# Exercise 136: derive manually; do not use autograd in this cell.
# Define `bncap_dbnvar` — the `(1, 3)` variance gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_tensor("bncap_dbnvar", "batchnorm_capstone", "dbnvar")


### Exercise 137 — Full BatchNorm: variance reduction

**Purpose:** Learn to spread each variance gradient back across all squared deviations for that hidden neuron.

**Inputs:** `bncap_bndiff2` is `(4, 3)` and `bncap_bnvar` is `(1, 3)`.

**Forward operation:** `bncap_bnvar = bncap_bndiff2.sum(dim=0, keepdim=True) / (bncap_n - 1)`.

**Derive/do:** Derive `bncap_dbndiff2`.

**Ingredients:** Broadcast across examples and include Bessel's divisor `bncap_n - 1`.

**Required outputs:**

- `bncap_dbndiff2`: the `(4, 3)` squared-deviation gradient.

**Next concept:** Full BatchNorm: square backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
# Continue from the BatchNorm capstone fixture and prior gradients.
assert "batchnorm_capstone" in _REFS


In [ ]:
# Exercise 137: derive manually; do not use autograd in this cell.
# Define `bncap_dbndiff2` — the `(4, 3)` squared-deviation gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_tensor("bncap_dbndiff2", "batchnorm_capstone", "dbndiff2")


### Exercise 138 — Full BatchNorm: square backward

**Purpose:** Learn to turn squared-deviation gradients into the variance branch's centered-value gradients.

**Inputs:** `bncap_bndiff` and `bncap_bndiff2` are `(4, 3)`.

**Forward operation:** `bncap_bndiff2 = bncap_bndiff**2`.

**Derive/do:** Using `bncap_dbndiff2`, derive `bncap_dbndiff_variance`.

**Ingredients:** The square local derivative; this is only the variance-path contribution.

**Required outputs:**

- `bncap_dbndiff_variance`: the variance-path contribution to centered values.

**Next concept:** Full BatchNorm: merge centered-value paths.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
# Continue from the BatchNorm capstone fixture and prior gradients.
assert "batchnorm_capstone" in _REFS


In [ ]:
# Exercise 138: derive manually; do not use autograd in this cell.
# Define `bncap_dbndiff_variance` — the variance-path contribution to centered values.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_tensor("bncap_dbndiff_variance", "batchnorm_capstone", "dbndiff_variance")


### Exercise 139 — Full BatchNorm: merge centered-value paths

**Purpose:** Learn to add the direct normalization path and variance path when they meet at centered values.

**Inputs:** Both centered-value contributions have shape `(4, 3)`.

**Forward operation:** `bncap_bndiff` feeds both `bnraw` directly and the variance branch.

**Derive/do:** Derive total `bncap_dbndiff` by combining your two path gradients.

**Ingredients:** Fan-out backward adds contributions where branches meet.

**Required outputs:**

- `bncap_dbndiff`: the total `(4, 3)` centered-value gradient.

**Next concept:** Full BatchNorm: centering and mean.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
# Continue from the BatchNorm capstone fixture and prior gradients.
assert "batchnorm_capstone" in _REFS


In [ ]:
# Exercise 139: derive manually; do not use autograd in this cell.
# Define `bncap_dbndiff` — the total `(4, 3)` centered-value gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_tensor("bncap_dbndiff", "batchnorm_capstone", "dbndiff")


### Exercise 140 — Full BatchNorm: centering and mean

**Purpose:** Learn to finish BatchNorm by returning through mean subtraction and the mean computed from the same inputs.

**Inputs:** `bncap_bnmeani` is `(1, 3)` and `bncap_hprebn` is `(4, 3)`.

**Forward operation:** `bncap_bndiff = bncap_hprebn - bncap_bnmeani` and `bncap_bnmeani = bncap_hprebn.mean(dim=0, keepdim=True)`.

**Derive/do:** Derive `bncap_dbnmeani` and total `bncap_dhprebn`.

**Ingredients:** Unbroadcast subtraction into the mean, reverse the four-example mean, then add its contribution to the direct path.

**Required outputs:**

- `bncap_dbnmeani`: the `(1, 3)` mean gradient.
- `bncap_dhprebn`: the final `(4, 3)` pre-BatchNorm gradient.

**Next concept:** MLP forward shape audit.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
# Continue from the BatchNorm capstone fixture and prior gradients.
assert "batchnorm_capstone" in _REFS


In [ ]:
# Exercise 140: derive manually; do not use autograd in this cell.
# Define `bncap_dbnmeani` — the `(1, 3)` mean gradient.
# Define `bncap_dhprebn` — the final `(4, 3)` pre-BatchNorm gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_tensor("bncap_dbnmeani", "batchnorm_capstone", "dbnmeani")
_check_tensor("bncap_dhprebn", "batchnorm_capstone", "dhprebn")


## 10. Capstone: manually backpropagate a next-character MLP

The final graph mirrors the lecture at a deliberately small scale. It uses four training examples, context length three, embedding size two, five hidden neurons, and six vocabulary candidates.

Forward path:

1. look up three embeddings per example;
2. flatten each context;
3. apply a hidden affine layer;
4. apply BatchNorm;
5. apply tanh;
6. apply an output affine layer;
7. compute stable softmax;
8. select each example's expected target;
9. average all four negative log-likelihoods.

The loss scores the complete supplied four-example mini-batch. Each row is a training example, each output column is a vocabulary candidate, and only the indexed expected target probability contributes directly to that row's loss. The other candidates influence it through softmax normalization.

Do not call autograd in answer cells. Work backward one named forward tensor at a time and keep a shape ledger beside you.


### Exercise 141 — MLP forward shape audit

**Purpose:** Learn to name the meaning and size of every tensor axis in the complete MLP before deriving gradients.

**Inputs:** The supplied fixture defines the complete tiny MLP and all dimension constants.

**Forward operation:** Read the forward pass without using stored reference shapes.

**Derive/do:** Write a Python tuple for every requested intermediate shape.

**Ingredients:** Track examples, context positions, embedding features, hidden neurons, and vocabulary candidates independently.

**Required outputs:**

- `mlp_emb_shape`: the shape of the embedding lookup.
- `mlp_embcat_shape`: the shape after flattening each context.
- `mlp_hprebn_shape`: the hidden affine output shape.
- `mlp_bnmeani_shape`: the per-hidden-neuron mean shape.
- `mlp_bnvar_shape`: the per-hidden-neuron variance shape.
- `mlp_bnraw_shape`: the normalized hidden-value shape.
- `mlp_hpreact_shape`: the BatchNorm affine output shape.
- `mlp_h_shape`: the hidden activation shape.
- `mlp_logits_shape`: the example-by-candidate score shape.
- `mlp_counts_sum_shape`: the row-denominator shape.
- `mlp_probs_shape`: the candidate-probability shape.
- `mlp_logprobs_shape`: the log-probability shape.
- `mlp_loss_shape`: the scalar average-loss shape.

**Next concept:** MLP: mean NLL seed.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
# Supplied tiny next-character MLP dimensions.
mlp_B = 4
mlp_BLOCK = 3
mlp_E = 2
mlp_H = 5
mlp_V = 6
mlp_eps = 1e-5

# Four complete training examples; repeated IDs make embedding accumulation observable.
mlp_Xb = torch.tensor([
    [0, 1, 2],
    [1, 2, 1],
    [3, 1, 0],
    [2, 4, 1],
], dtype=torch.long)
mlp_Yb = torch.tensor([2, 3, 1, 5], dtype=torch.long)

# Deterministic parameters; no training update occurs in this notebook.
mlp_g = torch.Generator().manual_seed(2147483647)
mlp_C = (torch.randn((mlp_V, mlp_E), generator=mlp_g, dtype=DTYPE) * 0.4).requires_grad_()
mlp_W1 = (torch.randn((mlp_BLOCK * mlp_E, mlp_H), generator=mlp_g, dtype=DTYPE) * 0.3).requires_grad_()
mlp_b1 = (torch.randn(mlp_H, generator=mlp_g, dtype=DTYPE) * 0.1).requires_grad_()
mlp_bngain = (1.0 + torch.randn((1, mlp_H), generator=mlp_g, dtype=DTYPE) * 0.1).requires_grad_()
mlp_bnbias = (torch.randn((1, mlp_H), generator=mlp_g, dtype=DTYPE) * 0.1).requires_grad_()
mlp_W2 = (torch.randn((mlp_H, mlp_V), generator=mlp_g, dtype=DTYPE) * 0.2).requires_grad_()
mlp_b2 = (torch.randn(mlp_V, generator=mlp_g, dtype=DTYPE) * 0.1).requires_grad_()

# Embedding and hidden affine layer.
mlp_emb = mlp_C[mlp_Xb]
mlp_embcat = mlp_emb.reshape(mlp_B, mlp_BLOCK * mlp_E)
mlp_hprebn = mlp_embcat @ mlp_W1 + mlp_b1

# Expanded BatchNorm across the four training-example rows.
mlp_bnmeani = mlp_hprebn.mean(dim=0, keepdim=True)
mlp_bndiff = mlp_hprebn - mlp_bnmeani
mlp_bndiff2 = mlp_bndiff**2
mlp_bnvar = mlp_bndiff2.sum(dim=0, keepdim=True) / (mlp_B - 1)
mlp_bnvar_inv = (mlp_bnvar + mlp_eps) ** -0.5
mlp_bnraw = mlp_bndiff * mlp_bnvar_inv
mlp_hpreact = mlp_bngain * mlp_bnraw + mlp_bnbias
mlp_h = mlp_hpreact.tanh()

# Output layer and expanded stable softmax cross-entropy.
mlp_logits = mlp_h @ mlp_W2 + mlp_b2
mlp_logit_maxes = mlp_logits.max(dim=1, keepdim=True).values
mlp_norm_logits = mlp_logits - mlp_logit_maxes
mlp_counts = mlp_norm_logits.exp()
mlp_counts_sum = mlp_counts.sum(dim=1, keepdim=True)
mlp_counts_sum_inv = mlp_counts_sum**-1
mlp_probs = mlp_counts * mlp_counts_sum_inv
mlp_logprobs = mlp_probs.log()
mlp_loss = -mlp_logprobs[range(mlp_B), mlp_Yb].mean()

_capture(
    "mlp_capstone",
    mlp_loss,
    {
        "dlogprobs": mlp_logprobs,
        "dprobs": mlp_probs,
        "dcounts_sum_inv": mlp_counts_sum_inv,
        "dcounts_sum": mlp_counts_sum,
        "dcounts": mlp_counts,
        "dnorm_logits": mlp_norm_logits,
        "dlogit_maxes": mlp_logit_maxes,
        "dlogits": mlp_logits,
        "dh": mlp_h,
        "dW2": mlp_W2,
        "db2": mlp_b2,
        "dhpreact": mlp_hpreact,
        "dbngain": mlp_bngain,
        "dbnbias": mlp_bnbias,
        "dbnraw": mlp_bnraw,
        "dbnvar_inv": mlp_bnvar_inv,
        "dbnvar": mlp_bnvar,
        "dbndiff2": mlp_bndiff2,
        "dbndiff": mlp_bndiff,
        "dbnmeani": mlp_bnmeani,
        "dhprebn": mlp_hprebn,
        "dembcat": mlp_embcat,
        "dW1": mlp_W1,
        "db1": mlp_b1,
        "demb": mlp_emb,
        "dC": mlp_C,
    },
    retain_graph=True,
)
for mlp_name, mlp_tensor in {
    "emb_shape": mlp_emb,
    "embcat_shape": mlp_embcat,
    "hprebn_shape": mlp_hprebn,
    "bnmeani_shape": mlp_bnmeani,
    "bnvar_shape": mlp_bnvar,
    "bnraw_shape": mlp_bnraw,
    "hpreact_shape": mlp_hpreact,
    "h_shape": mlp_h,
    "logits_shape": mlp_logits,
    "counts_sum_shape": mlp_counts_sum,
    "probs_shape": mlp_probs,
    "logprobs_shape": mlp_logprobs,
    "loss_shape": mlp_loss,
}.items():
    _store_shape("mlp_capstone", mlp_name, mlp_tensor)
print("MLP capstone ready:", tuple(mlp_Xb.shape), "-> loss", tuple(mlp_loss.shape))
# Private staged references for paths that merge later.
_capture_product_path(
    "mlp_capstone",
    "dcounts_direct",
    mlp_counts,
    mlp_counts_sum_inv,
    _REFS["mlp_capstone"]["dprobs"],
)
_capture_product_path(
    "mlp_capstone",
    "dbndiff_direct",
    mlp_bndiff,
    mlp_bnvar_inv,
    _REFS["mlp_capstone"]["dbnraw"],
)
_capture_path(
    "mlp_capstone",
    "dbndiff_variance",
    mlp_bnvar,
    mlp_bndiff,
    _REFS["mlp_capstone"]["dbnvar"],
)
_REFS["mlp_capstone"]["logit_row_grad_sums"] = (
    _REFS["mlp_capstone"]["dlogits"].sum(dim=1)
).detach().clone()
_REFS["mlp_capstone"]["hprebn_column_grad_sums"] = (
    _REFS["mlp_capstone"]["dhprebn"].sum(dim=0)
).detach().clone()


In [ ]:
# Exercise 141: derive manually; do not use autograd in this cell.
# Define `mlp_emb_shape` — the shape of the embedding lookup.
# Define `mlp_embcat_shape` — the shape after flattening each context.
# Define `mlp_hprebn_shape` — the hidden affine output shape.
# Define `mlp_bnmeani_shape` — the per-hidden-neuron mean shape.
# Define `mlp_bnvar_shape` — the per-hidden-neuron variance shape.
# Define `mlp_bnraw_shape` — the normalized hidden-value shape.
# Define `mlp_hpreact_shape` — the BatchNorm affine output shape.
# Define `mlp_h_shape` — the hidden activation shape.
# Define `mlp_logits_shape` — the example-by-candidate score shape.
# Define `mlp_counts_sum_shape` — the row-denominator shape.
# Define `mlp_probs_shape` — the candidate-probability shape.
# Define `mlp_logprobs_shape` — the log-probability shape.
# Define `mlp_loss_shape` — the scalar average-loss shape.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_shape("mlp_emb_shape", "mlp_capstone", "emb_shape")
_check_shape("mlp_embcat_shape", "mlp_capstone", "embcat_shape")
_check_shape("mlp_hprebn_shape", "mlp_capstone", "hprebn_shape")
_check_shape("mlp_bnmeani_shape", "mlp_capstone", "bnmeani_shape")
_check_shape("mlp_bnvar_shape", "mlp_capstone", "bnvar_shape")
_check_shape("mlp_bnraw_shape", "mlp_capstone", "bnraw_shape")
_check_shape("mlp_hpreact_shape", "mlp_capstone", "hpreact_shape")
_check_shape("mlp_h_shape", "mlp_capstone", "h_shape")
_check_shape("mlp_logits_shape", "mlp_capstone", "logits_shape")
_check_shape("mlp_counts_sum_shape", "mlp_capstone", "counts_sum_shape")
_check_shape("mlp_probs_shape", "mlp_capstone", "probs_shape")
_check_shape("mlp_logprobs_shape", "mlp_capstone", "logprobs_shape")
_check_shape("mlp_loss_shape", "mlp_capstone", "loss_shape")


### Exercise 142 — MLP: mean NLL seed

**Purpose:** Learn to start the MLP backward pass with one direct target gradient per example.

**Inputs:** `mlp_logprobs` is `(4, 6)`, with four examples and six candidates; `mlp_Yb` has one target per example.

**Forward operation:** `mlp_loss = -mlp_logprobs[range(mlp_B), mlp_Yb].mean()`.

**Derive/do:** Derive `mlp_dlogprobs`.

**Ingredients:** Zeros everywhere except paired targets; account for negation and the four-example average.

**Required outputs:**

- `mlp_dlogprobs`: the sparse `(4, 6)` gradient.

**Next concept:** MLP: log backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
# Continue from the supplied MLP capstone fixture.
assert "mlp_capstone" in _REFS


In [ ]:
# Exercise 142: derive manually; do not use autograd in this cell.
# Define `mlp_dlogprobs` — the sparse `(4, 6)` gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_tensor("mlp_dlogprobs", "mlp_capstone", "dlogprobs")


### Exercise 143 — MLP: log backward

**Purpose:** Learn to pass the MLP's target gradients from log-probabilities back to probabilities.

**Inputs:** `mlp_probs` and `mlp_dlogprobs` are `(4, 6)`.

**Forward operation:** `mlp_logprobs = mlp_probs.log()`.

**Derive/do:** Derive `mlp_dprobs`.

**Ingredients:** Natural-log local derivative, elementwise.

**Required outputs:**

- `mlp_dprobs`: the `(4, 6)` probability gradient.

**Next concept:** MLP: normalization multiplication.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
# Continue from the MLP capstone fixture and prior gradients.
assert "mlp_capstone" in _REFS


In [ ]:
# Exercise 143: derive manually; do not use autograd in this cell.
# Define `mlp_dprobs` — the `(4, 6)` probability gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_tensor("mlp_dprobs", "mlp_capstone", "dprobs")


### Exercise 144 — MLP: normalization multiplication

**Purpose:** Learn to split each probability gradient between its candidate weight and its shared row denominator.

**Inputs:** `mlp_counts` is `(4, 6)` and `mlp_counts_sum_inv` is `(4, 1)`.

**Forward operation:** `mlp_probs = mlp_counts * mlp_counts_sum_inv`.

**Derive/do:** Derive direct `mlp_dcounts_direct` and `mlp_dcounts_sum_inv`.

**Ingredients:** Product backward; accumulate the broadcast denominator factor over six candidates.

**Required outputs:**

- `mlp_dcounts_direct`: the direct `(4, 6)` numerator contribution.
- `mlp_dcounts_sum_inv`: the `(4, 1)` reciprocal-denominator gradient.

**Next concept:** MLP: denominator branch and count merge.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
# Continue from the MLP capstone fixture and prior gradients.
assert "mlp_capstone" in _REFS


In [ ]:
# Exercise 144: derive manually; do not use autograd in this cell.
# Define `mlp_dcounts_direct` — the direct `(4, 6)` numerator contribution.
# Define `mlp_dcounts_sum_inv` — the `(4, 1)` reciprocal-denominator gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_tensor("mlp_dcounts_direct", "mlp_capstone", "dcounts_direct")
_check_tensor("mlp_dcounts_sum_inv", "mlp_capstone", "dcounts_sum_inv")


### Exercise 145 — MLP: denominator branch and count merge

**Purpose:** Learn to reverse the reciprocal and row sum, then add the denominator path to the direct candidate-weight path.

**Inputs:** `mlp_counts_sum` is `(4, 1)` and `mlp_counts` is `(4, 6)`.

**Forward operation:** The denominator branch is reciprocal after a row-wise sum.

**Derive/do:** Derive `mlp_dcounts_sum` and total `mlp_dcounts`.

**Ingredients:** Reverse power `-1`, broadcast through the candidate sum, then add to the direct count contribution.

**Required outputs:**

- `mlp_dcounts_sum`: the `(4, 1)` row-total gradient.
- `mlp_dcounts`: the total `(4, 6)` count gradient.

**Next concept:** MLP: exp and maximum shift.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
# Continue from the MLP capstone fixture and prior gradients.
assert "mlp_capstone" in _REFS


In [ ]:
# Exercise 145: derive manually; do not use autograd in this cell.
# Define `mlp_dcounts_sum` — the `(4, 1)` row-total gradient.
# Define `mlp_dcounts` — the total `(4, 6)` count gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_tensor("mlp_dcounts_sum", "mlp_capstone", "dcounts_sum")
_check_tensor("mlp_dcounts", "mlp_capstone", "dcounts")


### Exercise 146 — MLP: exp and maximum shift

**Purpose:** Learn to return through exponentiation and maximum shifting to obtain gradients for every logit.

**Inputs:** Normalized logits and logits are `(4, 6)`; row maxima are `(4, 1)`.

**Forward operation:** Counts are exponentials of max-shifted logits.

**Derive/do:** Derive `mlp_dnorm_logits`, `mlp_dlogit_maxes`, and total `mlp_dlogits`.

**Ingredients:** Reverse exp; reverse broadcast subtraction; route max gradients to unique row argmax positions and add the direct path.

**Required outputs:**

- `mlp_dnorm_logits`: the normalized-logit gradient.
- `mlp_dlogit_maxes`: the row-maximum gradient.
- `mlp_dlogits`: the final logit gradient.

**Next concept:** MLP: output affine layer.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
# Continue from the MLP capstone fixture and prior gradients.
assert "mlp_capstone" in _REFS


In [ ]:
# Exercise 146: derive manually; do not use autograd in this cell.
# Define `mlp_dnorm_logits` — the normalized-logit gradient.
# Define `mlp_dlogit_maxes` — the row-maximum gradient.
# Define `mlp_dlogits` — the final logit gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_tensor("mlp_dnorm_logits", "mlp_capstone", "dnorm_logits")
_check_tensor("mlp_dlogit_maxes", "mlp_capstone", "dlogit_maxes")
_check_tensor("mlp_dlogits", "mlp_capstone", "dlogits")


### Exercise 147 — MLP: output affine layer

**Purpose:** Learn to send logit gradients back to hidden activations, output weights, and output bias.

**Inputs:** `mlp_h` is `(4, 5)`, `mlp_W2` is `(5, 6)`, and `mlp_b2` is `(6,)`.

**Forward operation:** `mlp_logits = mlp_h @ mlp_W2 + mlp_b2`.

**Derive/do:** Using `mlp_dlogits`, derive `mlp_dh`, `mlp_dW2`, and `mlp_db2`.

**Ingredients:** Affine-layer backward; sum the bias path over four examples.

**Required outputs:**

- `mlp_dh`: the `(4, 5)` hidden-activation gradient.
- `mlp_dW2`: the `(5, 6)` output-weight gradient.
- `mlp_db2`: the length-6 output-bias gradient.

**Next concept:** MLP: tanh backward.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
# Continue from the MLP capstone fixture and prior gradients.
assert "mlp_capstone" in _REFS


In [ ]:
# Exercise 147: derive manually; do not use autograd in this cell.
# Define `mlp_dh` — the `(4, 5)` hidden-activation gradient.
# Define `mlp_dW2` — the `(5, 6)` output-weight gradient.
# Define `mlp_db2` — the length-6 output-bias gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_tensor("mlp_dh", "mlp_capstone", "dh")
_check_tensor("mlp_dW2", "mlp_capstone", "dW2")
_check_tensor("mlp_db2", "mlp_capstone", "db2")


### Exercise 148 — MLP: tanh backward

**Purpose:** Learn to send hidden-activation gradients backward through `tanh`.

**Inputs:** `mlp_hpreact` and `mlp_h` are `(4, 5)`.

**Forward operation:** `mlp_h = mlp_hpreact.tanh()`.

**Derive/do:** Using `mlp_dh`, derive `mlp_dhpreact`.

**Ingredients:** Tanh's local derivative expressed with the stored forward activation `mlp_h`.

**Required outputs:**

- `mlp_dhpreact`: the `(4, 5)` gradient entering BatchNorm.

**Next concept:** MLP: BatchNorm learned affine.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
# Continue from the MLP capstone fixture and prior gradients.
assert "mlp_capstone" in _REFS


In [ ]:
# Exercise 148: derive manually; do not use autograd in this cell.
# Define `mlp_dhpreact` — the `(4, 5)` gradient entering BatchNorm.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_tensor("mlp_dhpreact", "mlp_capstone", "dhpreact")


### Exercise 149 — MLP: BatchNorm learned affine

**Purpose:** Learn to compute gradients for BatchNorm's learned gain, learned bias, and normalized hidden values.

**Inputs:** `mlp_bnraw` is `(4, 5)`; gain and bias are `(1, 5)`.

**Forward operation:** `mlp_hpreact = mlp_bngain * mlp_bnraw + mlp_bnbias`.

**Derive/do:** Derive `mlp_dbnraw`, `mlp_dbngain`, and `mlp_dbnbias`.

**Ingredients:** Elementwise affine backward and unbroadcasting over four examples.

**Required outputs:**

- `mlp_dbnraw`: the normalized-hidden gradient.
- `mlp_dbngain`: the learned BatchNorm gain gradient.
- `mlp_dbnbias`: the learned BatchNorm bias gradient.

**Next concept:** MLP: BatchNorm normalization and inverse std.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
# Continue from the MLP capstone fixture and prior gradients.
assert "mlp_capstone" in _REFS


In [ ]:
# Exercise 149: derive manually; do not use autograd in this cell.
# Define `mlp_dbnraw` — the normalized-hidden gradient.
# Define `mlp_dbngain` — the learned BatchNorm gain gradient.
# Define `mlp_dbnbias` — the learned BatchNorm bias gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_tensor("mlp_dbnraw", "mlp_capstone", "dbnraw")
_check_tensor("mlp_dbngain", "mlp_capstone", "dbngain")
_check_tensor("mlp_dbnbias", "mlp_capstone", "dbnbias")


### Exercise 150 — MLP: BatchNorm normalization and inverse std

**Purpose:** Learn to split BatchNorm gradients between the direct centered-value path and the variance path.

**Inputs:** `mlp_bndiff` is `(4, 5)`, inverse standard deviation is `(1, 5)`.

**Forward operation:** `mlp_bnraw = mlp_bndiff * mlp_bnvar_inv`, then inverse std is a power of stabilized variance.

**Derive/do:** Derive `mlp_dbndiff_direct`, `mlp_dbnvar_inv`, and `mlp_dbnvar`.

**Ingredients:** Product backward with batch accumulation, followed by power `-0.5`.

**Required outputs:**

- `mlp_dbndiff_direct`: the direct centered-value contribution.
- `mlp_dbnvar_inv`: the inverse-standard-deviation gradient.
- `mlp_dbnvar`: the variance gradient.

**Next concept:** MLP: BatchNorm variance and centering.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
# Continue from the MLP capstone fixture and prior gradients.
assert "mlp_capstone" in _REFS


In [ ]:
# Exercise 150: derive manually; do not use autograd in this cell.
# Define `mlp_dbndiff_direct` — the direct centered-value contribution.
# Define `mlp_dbnvar_inv` — the inverse-standard-deviation gradient.
# Define `mlp_dbnvar` — the variance gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_tensor("mlp_dbndiff_direct", "mlp_capstone", "dbndiff_direct")
_check_tensor("mlp_dbnvar_inv", "mlp_capstone", "dbnvar_inv")
_check_tensor("mlp_dbnvar", "mlp_capstone", "dbnvar")


### Exercise 151 — MLP: BatchNorm variance and centering

**Purpose:** Learn to finish the MLP's BatchNorm backward pass by merging variance and direct paths and reversing centering.

**Inputs:** `mlp_bndiff2` and `mlp_bndiff` are `(4, 5)`; means are `(1, 5)`.

**Forward operation:** Variance is an unbiased reduction of squared deviations; deviations subtract column means.

**Derive/do:** Derive `mlp_dbndiff2`, `mlp_dbndiff_variance`, total `mlp_dbndiff`, `mlp_dbnmeani`, and `mlp_dhprebn`.

**Ingredients:** Reverse division by `B - 1`, square, branch merge, broadcast subtraction, and four-example mean.

**Required outputs:**

- `mlp_dbndiff2`: the squared-deviation gradient.
- `mlp_dbndiff_variance`: the variance-path centered-value contribution.
- `mlp_dbndiff`: the total centered-value gradient.
- `mlp_dbnmeani`: the column-mean gradient.
- `mlp_dhprebn`: the pre-BatchNorm hidden gradient.

**Next concept:** MLP: hidden affine and reshape.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
# Continue from the MLP capstone fixture and prior gradients.
assert "mlp_capstone" in _REFS


In [ ]:
# Exercise 151: derive manually; do not use autograd in this cell.
# Define `mlp_dbndiff2` — the squared-deviation gradient.
# Define `mlp_dbndiff_variance` — the variance-path centered-value contribution.
# Define `mlp_dbndiff` — the total centered-value gradient.
# Define `mlp_dbnmeani` — the column-mean gradient.
# Define `mlp_dhprebn` — the pre-BatchNorm hidden gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_tensor("mlp_dbndiff2", "mlp_capstone", "dbndiff2")
_check_tensor("mlp_dbndiff_variance", "mlp_capstone", "dbndiff_variance")
_check_tensor("mlp_dbndiff", "mlp_capstone", "dbndiff")
_check_tensor("mlp_dbnmeani", "mlp_capstone", "dbnmeani")
_check_tensor("mlp_dhprebn", "mlp_capstone", "dhprebn")


### Exercise 152 — MLP: hidden affine and reshape

**Purpose:** Learn to send pre-BatchNorm gradients through the first affine layer and restore the embedding-grid shape.

**Inputs:** `mlp_embcat` is `(4, 6)`, `mlp_W1` is `(6, 5)`, and `mlp_b1` is `(5,)`.

**Forward operation:** `mlp_hprebn = mlp_embcat @ mlp_W1 + mlp_b1`; `mlp_embcat` reshapes `mlp_emb`.

**Derive/do:** Derive `mlp_dembcat`, `mlp_dW1`, `mlp_db1`, and reshaped `mlp_demb`.

**Ingredients:** Affine backward, batch-axis bias accumulation, then restore shape `(4, 3, 2)`.

**Required outputs:**

- `mlp_dembcat`: the flattened embedding gradient.
- `mlp_dW1`: the first-layer weight gradient.
- `mlp_db1`: the first-layer bias gradient.
- `mlp_demb`: the `(4, 3, 2)` embedding-output gradient.

**Next concept:** MLP: embedding-table gradient.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
# Continue from the MLP capstone fixture and prior gradients.
assert "mlp_capstone" in _REFS


In [ ]:
# Exercise 152: derive manually; do not use autograd in this cell.
# Define `mlp_dembcat` — the flattened embedding gradient.
# Define `mlp_dW1` — the first-layer weight gradient.
# Define `mlp_db1` — the first-layer bias gradient.
# Define `mlp_demb` — the `(4, 3, 2)` embedding-output gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_tensor("mlp_dembcat", "mlp_capstone", "dembcat")
_check_tensor("mlp_dW1", "mlp_capstone", "dW1")
_check_tensor("mlp_db1", "mlp_capstone", "db1")
_check_tensor("mlp_demb", "mlp_capstone", "demb")


### Exercise 153 — MLP: embedding-table gradient

**Purpose:** Learn to accumulate every context-position gradient into the correct embedding row, including repeated IDs.

**Inputs:** `mlp_C` is `(6, 2)`, `mlp_Xb` is `(4, 3)`, and `mlp_demb` is `(4, 3, 2)`.

**Forward operation:** `mlp_emb = mlp_C[mlp_Xb]`.

**Derive/do:** Derive `mlp_dC`, including repeated token IDs.

**Ingredients:** Initialize zeros shaped like `mlp_C`; flatten positions if useful; use an accumulating indexed write rather than replacement.

**Required outputs:**

- `mlp_dC`: the final `(6, 2)` embedding-table gradient.

**Next concept:** MLP: gradient invariants.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
# Continue from the MLP capstone fixture and prior gradients.
assert "mlp_capstone" in _REFS


In [ ]:
# Exercise 153: derive manually; do not use autograd in this cell.
# Define `mlp_dC` — the final `(6, 2)` embedding-table gradient.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_tensor("mlp_dC", "mlp_capstone", "dC")


### Exercise 154 — MLP: gradient invariants

**Purpose:** Learn two quick sum checks that can reveal mistakes in the completed softmax and BatchNorm gradients.

**Inputs:** `mlp_dlogits` is `(4, 6)` and `mlp_dhprebn` is `(4, 5)`.

**Forward operation:** Softmax is unchanged by a constant row-logit shift, and BatchNorm centering removes a constant column shift.

**Derive/do:** Compute one sum of `mlp_dlogits` per example row and one sum of `mlp_dhprebn` per hidden-neuron column.

**Ingredients:** Reduce `mlp_dlogits` over candidates and `mlp_dhprebn` over examples; both results should be numerically near zero for these shift-invariant paths.

**Required outputs:**

- `mlp_logit_row_grad_sums`: the length-4 sum of logit gradients within each example.
- `mlp_hprebn_column_grad_sums`: the length-5 sum of pre-BatchNorm gradients over examples.

**Next concept:** completion and comparison with the lecture notebook.


In [ ]:
# Supplied fixture: run this before writing the manual answer.
# Continue from the completed MLP manual backward pass.
assert "mlp_capstone" in _REFS


In [ ]:
# Exercise 154: derive manually; do not use autograd in this cell.
# Define `mlp_logit_row_grad_sums` — the length-4 sum of logit gradients within each example.
# Define `mlp_hprebn_column_grad_sums` — the length-5 sum of pre-BatchNorm gradients over examples.
# Write your tensor operations below these comments, then run the test cell.


In [ ]:
# Supplied test: checks your manual result without showing the reference answer.
_check_tensor("mlp_logit_row_grad_sums", "mlp_capstone", "logit_row_grad_sums")
_check_tensor("mlp_hprebn_column_grad_sums", "mlp_capstone", "hprebn_column_grad_sums")


## Completion standard

You are ready to return to the lecture notebook when you can restart the kernel, run all completed cells, and pass all 154 tests while explaining aloud:

- why every gradient has its forward variable's shape;
- why reductions broadcast in backward;
- why broadcasts reduce in backward;
- why shared variables and repeated embedding IDs accumulate contributions;
- why only target log-probabilities receive a direct NLL gradient while all logits can receive a softmax gradient;
- how BatchNorm creates direct and variance paths;
- how gradients travel from mean NLL through the entire MLP to every parameter.

## Official references

- [PyTorch autograd mechanics](https://docs.pytorch.org/docs/stable/notes/autograd.html)
- [Broadcasting semantics](https://docs.pytorch.org/docs/stable/notes/broadcasting.html)
- [`torch.matmul`](https://docs.pytorch.org/docs/stable/generated/torch.matmul.html)
- [Tensor views](https://docs.pytorch.org/docs/stable/tensor_view.html)
- [`torch.nn.functional.cross_entropy`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html)
- [`torch.nn.BatchNorm1d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.BatchNorm1d.html)
